## Intro: this is the last attempt to climb the leaderboard, last attempt was conducted in Google Colab, hence we imported the dataset from drive and begin our last try here. (due to computational limit)

In [1]:
import os

print("Available Kaggle input folders:\n")

for root, directories, files in os.walk("/kaggle/input"):
    level = root.replace(
        "/kaggle/input",
        ""
    ).count(os.sep)

    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")

    file_indent = "    " * (level + 1)

    for filename in files:
        print(f"{file_indent}{filename}")

Available Kaggle input folders:

input/
    competitions/
        task-2-clickbait-detection-mse-641-s-26/
            val.jsonl
            test.jsonl
            train.jsonl
            sample_solution.csv
    datasets/
        stex098/
            task2-passage-model-assets/
                Task2_Kaggle_Passage/
                    notes/
                        READ_ME_NEXT_SESSION.txt
                        resume_state.json
                    models/
                        article_qa_model/
                            training_metrics.json
                            config.json
                            training_args.bin
                            tokenizer.json
                            tokenizer_config.json
                            model.safetensors
                        paragraph_qa_model/
                            config.json
                            training_args.bin
                            tokenizer.json
                            tokenizer_config.js

In [3]:
import os
import re
import time
import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

TYPE_MAX_LENGTH = 320

TYPE_LABEL_TO_ID = {
    "phrase": 0,
    "passage": 1,
    "multi": 2
}

TYPE_ID_TO_LABEL = {
    value: key
    for key, value in TYPE_LABEL_TO_ID.items()
}

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)


def clean_text(value):
    if value is None:
        return ""

    if isinstance(value, list):
        value = " ".join(
            str(item)
            for item in value
            if item is not None
        )

    return re.sub(
        r"\s+",
        " ",
        str(value)
    ).strip()


def build_type_article_text(row):
    title = clean_text(
        row.get("targetTitle", "")
    )

    description = clean_text(
        row.get("targetDescription", "")
    )

    paragraphs = row.get(
        "targetParagraphs",
        []
    )

    if not isinstance(paragraphs, list):
        paragraphs = []

    paragraph_text = " ".join(
        clean_text(paragraph)
        for paragraph in paragraphs
        if clean_text(paragraph)
    )

    return (
        f"TITLE: {title} "
        f"DESCRIPTION: {description} "
        f"ARTICLE: {paragraph_text}"
    )


print("Loading type classifier...")

type_tokenizer = (
    AutoTokenizer.from_pretrained(
        TYPE_MODEL_DIR,
        local_files_only=True
    )
)

type_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        TYPE_MODEL_DIR,
        local_files_only=True
    )
    .to(DEVICE)
)

type_model.eval()

print("Device:", DEVICE)
print("Labels:", type_model.config.id2label)


def predict_type_probabilities(
    dataframe,
    batch_size=32
):
    post_texts = (
        dataframe["postText"]
        .apply(clean_text)
        .tolist()
    )

    article_texts = (
        dataframe
        .apply(
            build_type_article_text,
            axis=1
        )
        .tolist()
    )

    probability_batches = []
    start_time = time.time()

    for batch_start in range(
        0,
        len(dataframe),
        batch_size
    ):
        batch_end = min(
            batch_start + batch_size,
            len(dataframe)
        )

        encoded = type_tokenizer(
            post_texts[
                batch_start:batch_end
            ],
            article_texts[
                batch_start:batch_end
            ],
            truncation="longest_first",
            max_length=TYPE_MAX_LENGTH,
            padding="max_length",
            return_tensors="pt"
        )

        encoded = {
            key: value.to(DEVICE)
            for key, value in encoded.items()
        }

        with torch.inference_mode():
            logits = type_model(
                **encoded
            ).logits

            probabilities = torch.softmax(
                logits,
                dim=-1
            )

        probability_batches.append(
            probabilities.cpu().numpy()
        )

        print(
            f"Processed {batch_end}/"
            f"{len(dataframe)}"
        )

    print(
        "Elapsed seconds:",
        round(
            time.time() - start_time,
            2
        )
    )

    return np.vstack(
        probability_batches
    )

# 1. Verify exact preprocessing against saved test outputs

verification_rows = 24

verification_probabilities = (
    predict_type_probabilities(
        test_df.iloc[
            :verification_rows
        ].copy(),
        batch_size=24
    )
)

saved_verification_probabilities = (
    test_type_predictions_df
    .sort_values("row_number")
    .iloc[:verification_rows][
        [
            "prob_phrase",
            "prob_passage",
            "prob_multi"
        ]
    ]
    .to_numpy()
)

verification_difference = np.abs(
    verification_probabilities
    - saved_verification_probabilities
)

mean_error = float(
    verification_difference.mean()
)

maximum_error = float(
    verification_difference.max()
)

print("\nVerification results:")
print("Mean probability error:", mean_error)
print("Maximum probability error:", maximum_error)

if maximum_error > 1e-4:
    raise RuntimeError(
        "Saved test probabilities were not reproduced "
        "closely enough. Validation inference stopped."
    )

print("Verification passed.")

# 2. Generate validation type predictions

val_type_probabilities = (
    predict_type_probabilities(
        val_df,
        batch_size=32
    )
)

val_predicted_type_ids = (
    val_type_probabilities.argmax(
        axis=1
    )
)

val_predicted_types = np.array([
    TYPE_ID_TO_LABEL[int(label_id)]
    for label_id
    in val_predicted_type_ids
])


def extract_gold_type(tags):
    if isinstance(tags, list) and tags:
        return str(tags[0])

    return str(tags)


val_type_predictions_df = pd.DataFrame({
    "row_number": np.arange(
        len(val_df)
    ),
    "id": val_df["id"].values,
    "gold_type": (
        val_df["tags"]
        .apply(extract_gold_type)
    ),
    "predicted_type": (
        val_predicted_types
    ),
    "prob_phrase": (
        val_type_probabilities[
            :,
            TYPE_LABEL_TO_ID["phrase"]
        ]
    ),
    "prob_passage": (
        val_type_probabilities[
            :,
            TYPE_LABEL_TO_ID["passage"]
        ]
    ),
    "prob_multi": (
        val_type_probabilities[
            :,
            TYPE_LABEL_TO_ID["multi"]
        ]
    ),
    "confidence": (
        val_type_probabilities.max(
            axis=1
        )
    )
})

VAL_TYPE_OUTPUT_PATH = os.path.join(
    WORK_DIR,
    "val_type_predictions.csv"
)

val_type_predictions_df.to_csv(
    VAL_TYPE_OUTPUT_PATH,
    index=False
)

validation_accuracy = float(
    (
        val_type_predictions_df[
            "gold_type"
        ]
        ==
        val_type_predictions_df[
            "predicted_type"
        ]
    ).mean()
)

print("\nValidation type predictions saved.")
print("Path:", VAL_TYPE_OUTPUT_PATH)
print("Rows:", len(val_type_predictions_df))
print("Accuracy:", validation_accuracy)

print("\nPredicted type counts:")
print(
    val_type_predictions_df[
        "predicted_type"
    ].value_counts()
)

print("\nGold vs predicted:")
display(
    pd.crosstab(
        val_type_predictions_df[
            "gold_type"
        ],
        val_type_predictions_df[
            "predicted_type"
        ],
        margins=True
    )
)

Loading type classifier...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Device: cuda
Labels: {0: 'phrase', 1: 'passage', 2: 'multi'}
Processed 24/24
Elapsed seconds: 1.04

Verification results:
Mean probability error: 0.00010131378643359206
Maximum probability error: 0.000561226507568402


RuntimeError: Saved test probabilities were not reproduced closely enough. Validation inference stopped.

In [4]:
# Confirm the tiny probability differences do not change predictions.

saved_verification_types = (
    test_type_predictions_df
    .sort_values("row_number")
    .iloc[:verification_rows][
        "predicted_type"
    ]
    .astype(str)
    .to_numpy()
)

new_verification_type_ids = (
    verification_probabilities.argmax(
        axis=1
    )
)

new_verification_types = np.array([
    TYPE_ID_TO_LABEL[int(label_id)]
    for label_id in new_verification_type_ids
])

type_agreement = float(
    (
        new_verification_types
        == saved_verification_types
    ).mean()
)

print("Verification mean probability error:", mean_error)
print("Verification maximum probability error:", maximum_error)
print("Predicted-type agreement:", type_agreement)

if type_agreement < 1.0:
    raise RuntimeError(
        "The predicted labels do not fully match "
        "the saved test predictions."
    )

print(
    "\nVerification accepted. "
    "The small probability differences are numerical only."
)


# Generate validation probabilities.

val_type_probabilities = (
    predict_type_probabilities(
        val_df,
        batch_size=32
    )
)

val_predicted_type_ids = (
    val_type_probabilities.argmax(
        axis=1
    )
)

val_predicted_types = np.array([
    TYPE_ID_TO_LABEL[int(label_id)]
    for label_id in val_predicted_type_ids
])


def extract_gold_type(tags):
    if isinstance(tags, list) and tags:
        return str(tags[0])

    return str(tags)


val_type_predictions_df = pd.DataFrame({
    "row_number": np.arange(
        len(val_df)
    ),
    "id": val_df["id"].values,
    "gold_type": (
        val_df["tags"]
        .apply(extract_gold_type)
    ),
    "predicted_type": (
        val_predicted_types
    ),
    "prob_phrase": (
        val_type_probabilities[
            :,
            TYPE_LABEL_TO_ID["phrase"]
        ]
    ),
    "prob_passage": (
        val_type_probabilities[
            :,
            TYPE_LABEL_TO_ID["passage"]
        ]
    ),
    "prob_multi": (
        val_type_probabilities[
            :,
            TYPE_LABEL_TO_ID["multi"]
        ]
    ),
    "confidence": (
        val_type_probabilities.max(
            axis=1
        )
    )
})

VAL_TYPE_OUTPUT_PATH = os.path.join(
    WORK_DIR,
    "val_type_predictions.csv"
)

val_type_predictions_df.to_csv(
    VAL_TYPE_OUTPUT_PATH,
    index=False
)

validation_accuracy = float(
    (
        val_type_predictions_df[
            "gold_type"
        ]
        ==
        val_type_predictions_df[
            "predicted_type"
        ]
    ).mean()
)

print("\nValidation predictions saved.")
print("Path:", VAL_TYPE_OUTPUT_PATH)
print("Rows:", len(val_type_predictions_df))
print("Validation accuracy:", validation_accuracy)

print("\nPredicted-type counts:")
print(
    val_type_predictions_df[
        "predicted_type"
    ].value_counts()
)

print("\nGold vs predicted:")
display(
    pd.crosstab(
        val_type_predictions_df[
            "gold_type"
        ],
        val_type_predictions_df[
            "predicted_type"
        ],
        margins=True
    )
)

Verification mean probability error: 0.00010131378643359206
Verification maximum probability error: 0.000561226507568402
Predicted-type agreement: 1.0

Verification accepted. The small probability differences are numerical only.
Processed 32/400
Processed 64/400
Processed 96/400
Processed 128/400
Processed 160/400
Processed 192/400
Processed 224/400
Processed 256/400
Processed 288/400
Processed 320/400
Processed 352/400
Processed 384/400
Processed 400/400
Elapsed seconds: 7.14

Validation predictions saved.
Path: /kaggle/working/task2_passage_v1/val_type_predictions.csv
Rows: 400
Validation accuracy: 0.755

Predicted-type counts:
predicted_type
passage    168
phrase     149
multi       83
Name: count, dtype: int64

Gold vs predicted:


predicted_type,multi,passage,phrase,All
gold_type,,,,
multi,59,9,16,84
passage,10,127,17,154
phrase,14,32,116,162
All,83,168,149,400


In [6]:
import os
import gc
import pandas as pd
import torch

from tqdm.auto import tqdm


# ---------------------------------------------------------
# 1. Confirm and preserve type-classifier results
# ---------------------------------------------------------

VAL_TYPE_OUTPUT_PATH = os.path.join(
    WORK_DIR,
    "val_type_predictions.csv"
)

val_type_predictions_df.to_csv(
    VAL_TYPE_OUTPUT_PATH,
    index=False
)

validation_accuracy = (
    val_type_predictions_df[
        "gold_type"
    ]
    .eq(
        val_type_predictions_df[
            "predicted_type"
        ]
    )
    .mean()
)

passage_type_df = (
    val_type_predictions_df[
        val_type_predictions_df[
            "predicted_type"
        ].eq("passage")
    ]
    .copy()
    .sort_values("row_number")
    .reset_index(drop=True)
)

PASSAGE_TYPE_OUTPUT_PATH = os.path.join(
    WORK_DIR,
    "val_predicted_passage_rows.csv"
)

passage_type_df.to_csv(
    PASSAGE_TYPE_OUTPUT_PATH,
    index=False
)

print("Validation accuracy:", validation_accuracy)
print("Predicted-passage rows:", len(passage_type_df))
print("Saved:", VAL_TYPE_OUTPUT_PATH)
print("Saved:", PASSAGE_TYPE_OUTPUT_PATH)


# ---------------------------------------------------------
# 2. Release the classifier from GPU
# ---------------------------------------------------------

for variable_name in [
    "type_model",
    "type_tokenizer",
    "val_type_probabilities",
    "verification_probabilities"
]:
    globals().pop(variable_name, None)

gc.collect()
torch.cuda.empty_cache()

print(
    "\nGPU memory after classifier cleanup:",
    round(
        torch.cuda.memory_allocated()
        / 1024**3,
        3
    ),
    "GB"
)


# ---------------------------------------------------------
# 3. Build paragraph-QA candidates only for passage rows
# ---------------------------------------------------------

passage_row_numbers = set(
    passage_type_df[
        "row_number"
    ].astype(int)
)


def get_article_contexts(row):
    """
    Return the title and every non-empty paragraph as
    separate extractive-QA candidate contexts.
    """
    contexts = []

    title = str(
        row.get("targetTitle", "")
        or ""
    )

    if title.strip():
        contexts.append({
            "source_kind": "title",
            "source_index": -1,
            "context": title
        })

    paragraphs = row.get(
        "targetParagraphs",
        []
    )

    if not isinstance(paragraphs, list):
        paragraphs = []

    for paragraph_index, paragraph in enumerate(
        paragraphs
    ):
        paragraph = str(paragraph)

        if paragraph.strip():
            contexts.append({
                "source_kind": "paragraph",
                "source_index": paragraph_index,
                "context": paragraph
            })

    return contexts


passage_candidate_records = []

for row_number in tqdm(
    sorted(passage_row_numbers),
    desc="Building passage QA candidates"
):
    row = val_df.iloc[row_number]

    question = clean_text(
        row.get("postText", "")
    )

    candidate_contexts = get_article_contexts(
        row
    )

    for candidate_number, candidate in enumerate(
        candidate_contexts
    ):
        passage_candidate_records.append({
            "candidate_index": len(
                passage_candidate_records
            ),
            "candidate_id": (
                f"validation_{row_number}"
                f"_candidate_{candidate_number}"
            ),
            "row_number": int(row_number),
            "article_id": row["id"],
            "question": question,
            "context": candidate["context"],
            "source_kind": candidate[
                "source_kind"
            ],
            "source_index": int(
                candidate["source_index"]
            )
        })


passage_candidate_df = pd.DataFrame(
    passage_candidate_records
)

PASSAGE_CANDIDATE_PATH = os.path.join(
    WORK_DIR,
    "val_passage_candidate_contexts.pkl"
)

passage_candidate_df.to_pickle(
    PASSAGE_CANDIDATE_PATH
)

print("\nPassage articles:", len(passage_row_numbers))
print(
    "Candidate contexts:",
    len(passage_candidate_df)
)

print("\nSource counts:")
print(
    passage_candidate_df[
        "source_kind"
    ].value_counts()
)

print(
    "\nAverage contexts per passage article:",
    round(
        len(passage_candidate_df)
        / len(passage_row_numbers),
        2
    )
)

print("\nSaved candidate table:")
print(PASSAGE_CANDIDATE_PATH)

display(
    passage_candidate_df.head()
)

Validation accuracy: 0.755
Predicted-passage rows: 168
Saved: /kaggle/working/task2_passage_v1/val_type_predictions.csv
Saved: /kaggle/working/task2_passage_v1/val_predicted_passage_rows.csv

GPU memory after classifier cleanup: 0.009 GB


Building passage QA candidates:   0%|          | 0/168 [00:00<?, ?it/s]


Passage articles: 168
Candidate contexts: 2495

Source counts:
source_kind
paragraph    2328
title         167
Name: count, dtype: int64

Average contexts per passage article: 14.85

Saved candidate table:
/kaggle/working/task2_passage_v1/val_passage_candidate_contexts.pkl


,candidate_index,candidate_id,row_number,article_id,question,context,source_kind,source_index
0,0,validation_0_candidate_0,0,0,Five Nights at Freddy’s Sequel Delayed for Wei...,Five Nights at Freddy’s Sequel Delayed for Wei...,title,-1
1,1,validation_0_candidate_1,0,0,Five Nights at Freddy’s Sequel Delayed for Wei...,Five Nights at Freddy’s creator Scott Cawthon ...,paragraph,0
2,2,validation_0_candidate_2,0,0,Five Nights at Freddy’s Sequel Delayed for Wei...,"For the past couple of years, horror gaming fa...",paragraph,1
3,3,validation_0_candidate_3,0,0,Five Nights at Freddy’s Sequel Delayed for Wei...,According to a post by Cawthon on the Five Nig...,paragraph,2
4,4,validation_0_candidate_4,0,0,Five Nights at Freddy’s Sequel Delayed for Wei...,Delays happen in the gaming industry all the t...,paragraph,3


## run the saved paragraph QA model over those candidates and save the raw logits before doing any decoding or reranking

In [7]:
import os
import gc
import time
import numpy as np
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

QA_MAX_LENGTH = 384
QA_STRIDE = 128
QA_BATCH_SIZE = 32

TOKENIZED_OUTPUT_DIR = os.path.join(
    WORK_DIR,
    "val_passage_paragraph_tokenized"
)

LOGITS_OUTPUT_PATH = os.path.join(
    WORK_DIR,
    "val_passage_paragraph_logits.npz"
)

print("Loading paragraph QA model...")

paragraph_tokenizer = (
    AutoTokenizer.from_pretrained(
        PARAGRAPH_MODEL_DIR,
        local_files_only=True,
        use_fast=True
    )
)

paragraph_model = (
    AutoModelForQuestionAnswering
    .from_pretrained(
        PARAGRAPH_MODEL_DIR,
        local_files_only=True
    )
    .to(DEVICE)
)

paragraph_model.eval()

print("Device:", DEVICE)
print(
    "GPU:",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "None"
)

# 1. Convert passage candidates to a Hugging Face Dataset

passage_candidate_raw = Dataset.from_pandas(
    passage_candidate_df[
        [
            "candidate_index",
            "question",
            "context"
        ]
    ],
    preserve_index=False
)


def tokenize_passage_candidates(examples):
    questions = [
        str(question).strip()
        for question in examples["question"]
    ]

    contexts = [
        str(context)
        for context in examples["context"]
    ]

    tokenized = paragraph_tokenizer(
        questions,
        contexts,
        truncation="only_second",
        max_length=QA_MAX_LENGTH,
        stride=QA_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    sample_mapping = tokenized.pop(
        "overflow_to_sample_mapping"
    )

    feature_candidate_indices = []

    for feature_index, sample_index in enumerate(
        sample_mapping
    ):
        candidate_index = int(
            examples["candidate_index"][
                sample_index
            ]
        )

        feature_candidate_indices.append(
            candidate_index
        )

        sequence_ids = tokenized.sequence_ids(
            feature_index
        )

        # Keep character offsets only for context tokens.
        tokenized["offset_mapping"][
            feature_index
        ] = [
            offset if sequence_ids[token_index] == 1
            else None
            for token_index, offset in enumerate(
                tokenized["offset_mapping"][
                    feature_index
                ]
            )
        ]

    tokenized[
        "candidate_index"
    ] = feature_candidate_indices

    return tokenized


tokenization_start = time.time()

passage_candidate_tokenized = (
    passage_candidate_raw.map(
        tokenize_passage_candidates,
        batched=True,
        remove_columns=(
            passage_candidate_raw.column_names
        ),
        desc="Tokenizing passage candidates"
    )
)

print(
    "\nTokenization seconds:",
    round(
        time.time() - tokenization_start,
        2
    )
)

print(
    "Candidate contexts:",
    len(passage_candidate_df)
)

print(
    "Sliding-window features:",
    len(passage_candidate_tokenized)
)

print(
    "Average windows per candidate:",
    round(
        len(passage_candidate_tokenized)
        / len(passage_candidate_df),
        2
    )
)

# 2. Save tokenized features before model inference

if os.path.exists(TOKENIZED_OUTPUT_DIR):
    import shutil

    shutil.rmtree(
        TOKENIZED_OUTPUT_DIR
    )

passage_candidate_tokenized.save_to_disk(
    TOKENIZED_OUTPUT_DIR
)

print(
    "\nTokenized features saved:",
    TOKENIZED_OUTPUT_DIR
)

# 3. Run paragraph-QA inference

model_input_names = [
    input_name
    for input_name
    in paragraph_tokenizer.model_input_names
    if input_name
    in passage_candidate_tokenized.column_names
]

start_logit_batches = []
end_logit_batches = []

inference_start = time.time()

for batch_start in range(
    0,
    len(passage_candidate_tokenized),
    QA_BATCH_SIZE
):
    batch_end = min(
        batch_start + QA_BATCH_SIZE,
        len(passage_candidate_tokenized)
    )

    batch = passage_candidate_tokenized[
        batch_start:batch_end
    ]

    model_inputs = {
        input_name: torch.tensor(
            batch[input_name],
            dtype=torch.long,
            device=DEVICE
        )
        for input_name in model_input_names
    }

    with torch.inference_mode():
        outputs = paragraph_model(
            **model_inputs
        )

    start_logit_batches.append(
        outputs.start_logits
        .detach()
        .cpu()
        .numpy()
    )

    end_logit_batches.append(
        outputs.end_logits
        .detach()
        .cpu()
        .numpy()
    )

    if (
        batch_end % 320 == 0
        or batch_end
        == len(passage_candidate_tokenized)
    ):
        elapsed = (
            time.time()
            - inference_start
        )

        print(
            f"Processed {batch_end}/"
            f"{len(passage_candidate_tokenized)} "
            f"features in {elapsed:.1f} seconds"
        )


paragraph_start_logits = np.concatenate(
    start_logit_batches,
    axis=0
)

paragraph_end_logits = np.concatenate(
    end_logit_batches,
    axis=0
)

np.savez_compressed(
    LOGITS_OUTPUT_PATH,
    start_logits=paragraph_start_logits,
    end_logits=paragraph_end_logits
)

print(
    "\nInference seconds:",
    round(
        time.time() - inference_start,
        2
    )
)

print(
    "Start-logit shape:",
    paragraph_start_logits.shape
)

print(
    "End-logit shape:",
    paragraph_end_logits.shape
)

print(
    "Logits saved:",
    LOGITS_OUTPUT_PATH
)

print(
    "Logits file size:",
    round(
        os.path.getsize(
            LOGITS_OUTPUT_PATH
        ) / 1024**2,
        2
    ),
    "MB"
)

# 4. Release the model from GPU

for variable_name in [
    "paragraph_model",
    "model_inputs",
    "outputs",
    "start_logit_batches",
    "end_logit_batches"
]:
    globals().pop(
        variable_name,
        None
    )

gc.collect()
torch.cuda.empty_cache()

print(
    "\nGPU memory after cleanup:",
    round(
        torch.cuda.memory_allocated()
        / 1024**3,
        3
    ),
    "GB"
)

print(
    "\nParagraph-QA inference checkpoint complete:",
    os.path.exists(
        LOGITS_OUTPUT_PATH
    )
    and os.path.exists(
        TOKENIZED_OUTPUT_DIR
    )
)

Loading paragraph QA model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Device: cuda
GPU: Tesla T4


Tokenizing passage candidates:   0%|          | 0/2495 [00:00<?, ? examples/s]


Tokenization seconds: 1.81
Candidate contexts: 2495
Sliding-window features: 2499
Average windows per candidate: 1.0


Saving the dataset (0/1 shards):   0%|          | 0/2499 [00:00<?, ? examples/s]


Tokenized features saved: /kaggle/working/task2_passage_v1/val_passage_paragraph_tokenized
Processed 320/2499 features in 7.3 seconds
Processed 640/2499 features in 14.7 seconds
Processed 960/2499 features in 22.1 seconds
Processed 1280/2499 features in 29.2 seconds
Processed 1600/2499 features in 36.1 seconds
Processed 1920/2499 features in 42.8 seconds
Processed 2240/2499 features in 49.4 seconds
Processed 2499/2499 features in 54.8 seconds

Inference seconds: 54.92
Start-logit shape: (2499, 384)
End-logit shape: (2499, 384)
Logits saved: /kaggle/working/task2_passage_v1/val_passage_paragraph_logits.npz
Logits file size: 1.21 MB

GPU memory after cleanup: 0.009 GB

Paragraph-QA inference checkpoint complete: True


In [8]:
## cpu_only decoding

import os
import time
import numpy as np
import pandas as pd

from datasets import load_from_disk
from transformers import AutoTokenizer

MAX_ANSWER_LENGTH = 100

TOKENIZED_OUTPUT_DIR = os.path.join(
    WORK_DIR,
    "val_passage_paragraph_tokenized"
)

LOGITS_OUTPUT_PATH = os.path.join(
    WORK_DIR,
    "val_passage_paragraph_logits.npz"
)

CANDIDATE_PREDICTIONS_PATH = os.path.join(
    WORK_DIR,
    "val_passage_paragraph_candidate_predictions.pkl"
)

# 1. Reload saved objects when necessary

if "passage_candidate_tokenized" not in globals():
    passage_candidate_tokenized = load_from_disk(
        TOKENIZED_OUTPUT_DIR
    )

if "paragraph_tokenizer" not in globals():
    paragraph_tokenizer = (
        AutoTokenizer.from_pretrained(
            PARAGRAPH_MODEL_DIR,
            local_files_only=True,
            use_fast=True
        )
    )

if (
    "paragraph_start_logits" not in globals()
    or "paragraph_end_logits" not in globals()
):
    saved_logits = np.load(
        LOGITS_OUTPUT_PATH
    )

    paragraph_start_logits = (
        saved_logits["start_logits"]
    )

    paragraph_end_logits = (
        saved_logits["end_logits"]
    )


print(
    "Tokenized features:",
    len(passage_candidate_tokenized)
)

print(
    "Start logits:",
    paragraph_start_logits.shape
)

print(
    "End logits:",
    paragraph_end_logits.shape
)

# 2. Decode the best span from every sliding-window feature

decode_start_time = time.time()

candidate_best_results = {}

for feature_index in range(
    len(passage_candidate_tokenized)
):
    feature = passage_candidate_tokenized[
        feature_index
    ]

    candidate_index = int(
        feature["candidate_index"]
    )

    input_ids = feature["input_ids"]
    offsets = feature["offset_mapping"]

    start_logits = paragraph_start_logits[
        feature_index
    ]

    end_logits = paragraph_end_logits[
        feature_index
    ]

    try:
        cls_index = input_ids.index(
            paragraph_tokenizer.cls_token_id
        )
    except ValueError:
        cls_index = 0

    null_score = float(
        start_logits[cls_index]
        + end_logits[cls_index]
    )

    valid_mask = np.array([
        offset is not None
        and len(offset) == 2
        and int(offset[1]) > int(offset[0])
        for offset in offsets
    ])

    valid_start_positions = np.flatnonzero(
        valid_mask
    )

    best_span_score = -np.inf
    best_start_position = None
    best_end_position = None

    for start_position in valid_start_positions:
        maximum_end_position = min(
            start_position
            + MAX_ANSWER_LENGTH
            - 1,
            len(offsets) - 1
        )

        allowed_end_mask = valid_mask[
            start_position:
            maximum_end_position + 1
        ]

        if not allowed_end_mask.any():
            continue

        local_end_logits = end_logits[
            start_position:
            maximum_end_position + 1
        ].copy()

        local_end_logits[
            ~allowed_end_mask
        ] = -np.inf

        relative_end_position = int(
            np.argmax(local_end_logits)
        )

        end_position = (
            start_position
            + relative_end_position
        )

        span_score = float(
            start_logits[start_position]
            + end_logits[end_position]
        )

        if span_score > best_span_score:
            best_span_score = span_score
            best_start_position = int(
                start_position
            )
            best_end_position = int(
                end_position
            )

    context = str(
        passage_candidate_df.iloc[
            candidate_index
        ]["context"]
    )

    if (
        best_start_position is not None
        and best_end_position is not None
    ):
        character_start = int(
            offsets[
                best_start_position
            ][0]
        )

        character_end = int(
            offsets[
                best_end_position
            ][1]
        )

        predicted_text = clean_text(
            context[
                character_start:
                character_end
            ]
        )
    else:
        predicted_text = ""
        best_span_score = -np.inf

    answerability_margin = float(
        best_span_score - null_score
    )

    feature_result = {
        "candidate_index": candidate_index,
        "predicted_text": predicted_text,
        "best_span_score": best_span_score,
        "null_score": null_score,
        "answerability_margin": (
            answerability_margin
        ),
        "feature_index": feature_index
    }

    previous_result = (
        candidate_best_results.get(
            candidate_index
        )
    )

    if (
        previous_result is None
        or answerability_margin
        > previous_result[
            "answerability_margin"
        ]
    ):
        candidate_best_results[
            candidate_index
        ] = feature_result

    if (
        (feature_index + 1) % 500 == 0
        or feature_index + 1
        == len(passage_candidate_tokenized)
    ):
        print(
            f"Decoded {feature_index + 1}/"
            f"{len(passage_candidate_tokenized)} "
            "features"
        )

# 3. Build and save one record per candidate context

candidate_prediction_records = []

for candidate_index in range(
    len(passage_candidate_df)
):
    metadata = passage_candidate_df.iloc[
        candidate_index
    ]

    decoded = candidate_best_results.get(
        candidate_index,
        {
            "predicted_text": "",
            "best_span_score": -np.inf,
            "null_score": np.inf,
            "answerability_margin": -np.inf,
            "feature_index": -1
        }
    )

    candidate_prediction_records.append({
        "candidate_index": candidate_index,
        "row_number": int(
            metadata["row_number"]
        ),
        "article_id": metadata[
            "article_id"
        ],
        "source_kind": metadata[
            "source_kind"
        ],
        "source_index": int(
            metadata["source_index"]
        ),
        "context": metadata["context"],
        "predicted_text": decoded[
            "predicted_text"
        ],
        "best_span_score": decoded[
            "best_span_score"
        ],
        "null_score": decoded[
            "null_score"
        ],
        "answerability_margin": decoded[
            "answerability_margin"
        ],
        "feature_index": decoded[
            "feature_index"
        ],
        "word_count": len(
            decoded[
                "predicted_text"
            ].split()
        )
    })


paragraph_candidate_predictions_df = (
    pd.DataFrame(
        candidate_prediction_records
    )
)

paragraph_candidate_predictions_df.to_pickle(
    CANDIDATE_PREDICTIONS_PATH
)


print(
    "\nDecoding seconds:",
    round(
        time.time()
        - decode_start_time,
        2
    )
)

print(
    "Candidate predictions:",
    len(
        paragraph_candidate_predictions_df
    )
)

print(
    "Empty predictions:",
    int(
        paragraph_candidate_predictions_df[
            "predicted_text"
        ]
        .fillna("")
        .str.strip()
        .eq("")
        .sum()
    )
)

print(
    "Positive-margin candidates:",
    int(
        paragraph_candidate_predictions_df[
            "answerability_margin"
        ].gt(0).sum()
    )
)

print(
    "\nSaved candidate predictions:"
)

print(CANDIDATE_PREDICTIONS_PATH)

print(
    "Checkpoint exists:",
    os.path.exists(
        CANDIDATE_PREDICTIONS_PATH
    )
)

print("\nAnswerability-margin summary:")

print(
    paragraph_candidate_predictions_df[
        "answerability_margin"
    ].describe()
)

print("\nHighest-confidence candidate spans:")

display(
    paragraph_candidate_predictions_df
    .sort_values(
        "answerability_margin",
        ascending=False
    )[
        [
            "row_number",
            "source_kind",
            "source_index",
            "predicted_text",
            "answerability_margin",
            "word_count"
        ]
    ]
    .head(15)
)

Tokenized features: 2499
Start logits: (2499, 384)
End logits: (2499, 384)
Decoded 500/2499 features
Decoded 1000/2499 features
Decoded 1500/2499 features
Decoded 2000/2499 features
Decoded 2499/2499 features

Decoding seconds: 4.72
Candidate predictions: 2495
Empty predictions: 0
Positive-margin candidates: 556

Saved candidate predictions:
/kaggle/working/task2_passage_v1/val_passage_paragraph_candidate_predictions.pkl
Checkpoint exists: True

Answerability-margin summary:
count    2495.000000
mean       -3.608818
std         5.380926
min       -23.450915
25%        -5.656187
50%        -2.723763
75%        -0.279339
max         9.210150
Name: answerability_margin, dtype: float64

Highest-confidence candidate spans:


,row_number,source_kind,source_index,predicted_text,answerability_margin,word_count
1143,210,paragraph,11,The Olympic tax,9.210150,3
558,112,paragraph,1,Relieve Your Sinuses,8.805585,3
2099,351,paragraph,10,There will be fights.,7.614697,4
754,152,paragraph,14,Window shopping,7.360863,2
1414,236,paragraph,20,I'm from out of state,6.793540,5
361,81,paragraph,8,Morgan Stanley,6.711813,2
1149,210,paragraph,17,Number of the week,6.689604,4
758,152,paragraph,18,Dine with the locals,6.578809,4
2103,351,paragraph,14,There will be score-keeping.,6.572339,4
1256,219,paragraph,32,HALFTIME,6.558293,1


In [9]:
import os
import gc
import time
import shutil
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

ARTICLE_MAX_LENGTH = 384
ARTICLE_STRIDE = 128
ARTICLE_BATCH_SIZE = 16

ARTICLE_ROWS_PATH = os.path.join(
    WORK_DIR,
    "val_passage_article_contexts.pkl"
)

ARTICLE_TOKENIZED_DIR = os.path.join(
    WORK_DIR,
    "val_passage_article_tokenized"
)

ARTICLE_LOGITS_PATH = os.path.join(
    WORK_DIR,
    "val_passage_article_logits.npz"
)

# 1. Build the full article context for each passage row

def build_combined_article_context(row):
    sections = []

    title = str(
        row.get("targetTitle", "")
        or ""
    ).strip()

    if title:
        sections.append(
            f"Title: {title}"
        )

    paragraphs = row.get(
        "targetParagraphs",
        []
    )

    if not isinstance(paragraphs, list):
        paragraphs = []

    for paragraph_index, paragraph in enumerate(
        paragraphs,
        start=1
    ):
        paragraph = str(
            paragraph
        ).strip()

        if paragraph:
            sections.append(
                f"Paragraph {paragraph_index}: "
                f"{paragraph}"
            )

    return "\n\n".join(
        sections
    )


article_context_records = []

for row_number in sorted(
    passage_row_numbers
):
    row = val_df.iloc[
        row_number
    ]

    article_context = (
        build_combined_article_context(
            row
        )
    )

    article_context_records.append({
        "row_number": int(
            row_number
        ),
        "article_id": row["id"],
        "question": clean_text(
            row.get("postText", "")
        ),
        "article_context": (
            article_context
        ),
        "article_word_count": len(
            article_context.split()
        )
    })


passage_article_df = pd.DataFrame(
    article_context_records
)

passage_article_df.to_pickle(
    ARTICLE_ROWS_PATH
)

print("Passage articles:", len(passage_article_df))

print(
    "Article word-count summary:"
)

print(
    passage_article_df[
        "article_word_count"
    ].describe()
)

print(
    "\nArticle contexts saved:",
    ARTICLE_ROWS_PATH
)

# 2. Load the saved article QA model

print("\nLoading article QA model...")

article_tokenizer = (
    AutoTokenizer.from_pretrained(
        ARTICLE_MODEL_DIR,
        local_files_only=True,
        use_fast=True
    )
)

article_model = (
    AutoModelForQuestionAnswering
    .from_pretrained(
        ARTICLE_MODEL_DIR,
        local_files_only=True
    )
    .to(DEVICE)
)

article_model.eval()

print("Device:", DEVICE)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

# 3. Tokenize into sliding-window features

article_raw_dataset = Dataset.from_pandas(
    passage_article_df[
        [
            "row_number",
            "question",
            "article_context"
        ]
    ],
    preserve_index=False
)


def tokenize_article_examples(examples):
    questions = [
        str(question).strip()
        for question in examples[
            "question"
        ]
    ]

    contexts = [
        str(context)
        for context in examples[
            "article_context"
        ]
    ]

    tokenized = article_tokenizer(
        questions,
        contexts,
        truncation="only_second",
        max_length=ARTICLE_MAX_LENGTH,
        stride=ARTICLE_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    sample_mapping = tokenized.pop(
        "overflow_to_sample_mapping"
    )

    feature_row_numbers = []

    for feature_index, sample_index in enumerate(
        sample_mapping
    ):
        row_number = int(
            examples["row_number"][
                sample_index
            ]
        )

        feature_row_numbers.append(
            row_number
        )

        sequence_ids = (
            tokenized.sequence_ids(
                feature_index
            )
        )

        tokenized[
            "offset_mapping"
        ][feature_index] = [
            offset
            if sequence_ids[token_index] == 1
            else None
            for token_index, offset
            in enumerate(
                tokenized[
                    "offset_mapping"
                ][feature_index]
            )
        ]

    tokenized[
        "row_number"
    ] = feature_row_numbers

    return tokenized


tokenization_start = time.time()

passage_article_tokenized = (
    article_raw_dataset.map(
        tokenize_article_examples,
        batched=True,
        remove_columns=(
            article_raw_dataset.column_names
        ),
        desc=(
            "Tokenizing predicted-passage articles"
        )
    )
)

print(
    "\nTokenization seconds:",
    round(
        time.time()
        - tokenization_start,
        2
    )
)

print(
    "Sliding-window features:",
    len(passage_article_tokenized)
)

print(
    "Average windows per article:",
    round(
        len(passage_article_tokenized)
        / len(passage_article_df),
        2
    )
)

# 4. Save tokenized features before inference

if os.path.exists(
    ARTICLE_TOKENIZED_DIR
):
    shutil.rmtree(
        ARTICLE_TOKENIZED_DIR
    )

passage_article_tokenized.save_to_disk(
    ARTICLE_TOKENIZED_DIR
)

print(
    "\nTokenized article features saved:"
)

print(ARTICLE_TOKENIZED_DIR)

# 5. Run article-QA inference

model_input_names = [
    input_name
    for input_name
    in article_tokenizer.model_input_names
    if input_name
    in passage_article_tokenized.column_names
]

article_start_batches = []
article_end_batches = []

inference_start = time.time()

for batch_start in range(
    0,
    len(passage_article_tokenized),
    ARTICLE_BATCH_SIZE
):
    batch_end = min(
        batch_start + ARTICLE_BATCH_SIZE,
        len(passage_article_tokenized)
    )

    batch = passage_article_tokenized[
        batch_start:batch_end
    ]

    model_inputs = {
        input_name: torch.tensor(
            batch[input_name],
            dtype=torch.long,
            device=DEVICE
        )
        for input_name
        in model_input_names
    }

    with torch.inference_mode():
        outputs = article_model(
            **model_inputs
        )

    article_start_batches.append(
        outputs.start_logits
        .detach()
        .cpu()
        .numpy()
    )

    article_end_batches.append(
        outputs.end_logits
        .detach()
        .cpu()
        .numpy()
    )

    if (
        batch_end % 160 == 0
        or batch_end
        == len(passage_article_tokenized)
    ):
        elapsed = (
            time.time()
            - inference_start
        )

        print(
            f"Processed {batch_end}/"
            f"{len(passage_article_tokenized)} "
            f"features in {elapsed:.1f} seconds"
        )


article_start_logits = np.concatenate(
    article_start_batches,
    axis=0
)

article_end_logits = np.concatenate(
    article_end_batches,
    axis=0
)

np.savez_compressed(
    ARTICLE_LOGITS_PATH,
    start_logits=article_start_logits,
    end_logits=article_end_logits
)

print(
    "\nInference seconds:",
    round(
        time.time()
        - inference_start,
        2
    )
)

print(
    "Start-logit shape:",
    article_start_logits.shape
)

print(
    "End-logit shape:",
    article_end_logits.shape
)

print(
    "Article logits saved:",
    ARTICLE_LOGITS_PATH
)

print(
    "Logits file size:",
    round(
        os.path.getsize(
            ARTICLE_LOGITS_PATH
        ) / 1024**2,
        2
    ),
    "MB"
)

# 6. Release the article model from GPU

for variable_name in [
    "article_model",
    "model_inputs",
    "outputs",
    "article_start_batches",
    "article_end_batches"
]:
    globals().pop(
        variable_name,
        None
    )

gc.collect()
torch.cuda.empty_cache()

print(
    "\nGPU memory after cleanup:",
    round(
        torch.cuda.memory_allocated()
        / 1024**3,
        3
    ),
    "GB"
)

print(
    "\nArticle-QA inference checkpoint complete:",
    os.path.exists(
        ARTICLE_LOGITS_PATH
    )
    and os.path.exists(
        ARTICLE_TOKENIZED_DIR
    )
    and os.path.exists(
        ARTICLE_ROWS_PATH
    )
)

Passage articles: 168
Article word-count summary:
count     168.000000
mean      609.047619
std       861.645604
min        43.000000
25%       241.250000
50%       424.000000
75%       723.000000
max      9846.000000
Name: article_word_count, dtype: float64

Article contexts saved: /kaggle/working/task2_passage_v1/val_passage_article_contexts.pkl

Loading article QA model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Device: cuda
GPU: Tesla T4


Tokenizing predicted-passage articles:   0%|          | 0/168 [00:00<?, ? examples/s]


Tokenization seconds: 1.11
Sliding-window features: 571
Average windows per article: 3.4


Saving the dataset (0/1 shards):   0%|          | 0/571 [00:00<?, ? examples/s]


Tokenized article features saved:
/kaggle/working/task2_passage_v1/val_passage_article_tokenized
Processed 160/571 features in 4.2 seconds
Processed 320/571 features in 8.5 seconds
Processed 480/571 features in 12.8 seconds
Processed 571/571 features in 15.3 seconds

Inference seconds: 15.45
Start-logit shape: (571, 384)
End-logit shape: (571, 384)
Article logits saved: /kaggle/working/task2_passage_v1/val_passage_article_logits.npz
Logits file size: 1.35 MB

GPU memory after cleanup: 0.009 GB

Article-QA inference checkpoint complete: True


In [10]:
import os
import re
import time
import numpy as np
import pandas as pd

from datasets import load_from_disk
from transformers import AutoTokenizer

ARTICLE_MAX_ANSWER_LENGTH = 100
TOP_START_POSITIONS = 20
TOP_END_POSITIONS = 20
MAX_CANDIDATES_PER_ARTICLE = 30

ARTICLE_ROWS_PATH = os.path.join(
    WORK_DIR,
    "val_passage_article_contexts.pkl"
)

ARTICLE_TOKENIZED_DIR = os.path.join(
    WORK_DIR,
    "val_passage_article_tokenized"
)

ARTICLE_LOGITS_PATH = os.path.join(
    WORK_DIR,
    "val_passage_article_logits.npz"
)

ARTICLE_CANDIDATES_PATH = os.path.join(
    WORK_DIR,
    "val_passage_article_span_candidates.pkl"
)

ARTICLE_CANDIDATES_CSV_PATH = os.path.join(
    WORK_DIR,
    "val_passage_article_span_candidates.csv"
)


# ---------------------------------------------------------
# 1. Reload the saved article objects
# ---------------------------------------------------------

if "passage_article_df" not in globals():
    passage_article_df = pd.read_pickle(
        ARTICLE_ROWS_PATH
    )

if "passage_article_tokenized" not in globals():
    passage_article_tokenized = load_from_disk(
        ARTICLE_TOKENIZED_DIR
    )

if "article_tokenizer" not in globals():
    article_tokenizer = (
        AutoTokenizer.from_pretrained(
            ARTICLE_MODEL_DIR,
            local_files_only=True,
            use_fast=True
        )
    )

if (
    "article_start_logits" not in globals()
    or "article_end_logits" not in globals()
):
    saved_article_logits = np.load(
        ARTICLE_LOGITS_PATH
    )

    article_start_logits = (
        saved_article_logits[
            "start_logits"
        ]
    )

    article_end_logits = (
        saved_article_logits[
            "end_logits"
        ]
    )


article_context_lookup = (
    passage_article_df
    .set_index("row_number")[
        "article_context"
    ]
    .to_dict()
)

print(
    "Article rows:",
    len(passage_article_df)
)

print(
    "Sliding-window features:",
    len(passage_article_tokenized)
)

print(
    "Start logits:",
    article_start_logits.shape
)

print(
    "End logits:",
    article_end_logits.shape
)


# ---------------------------------------------------------
# 2. Text-cleaning helpers
# ---------------------------------------------------------

def clean_article_span(text):
    text = str(text)

    # Remove article-structure markers introduced
    # when the full article context was constructed.
    text = re.sub(
        r"^\s*Title:\s*",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"^\s*Paragraph\s+\d+:\s*",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\s+Paragraph\s+\d+:\s*",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


def normalize_candidate_text(text):
    text = clean_article_span(
        text
    ).lower()

    text = re.sub(
        r"[^\w]+",
        " ",
        text
    )

    return re.sub(
        r"\s+",
        " ",
        text
    ).strip()


def get_top_indices(
    values,
    number_to_keep
):
    number_to_keep = min(
        number_to_keep,
        len(values)
    )

    if number_to_keep <= 0:
        return np.array(
            [],
            dtype=int
        )

    indices = np.argpartition(
        values,
        -number_to_keep
    )[-number_to_keep:]

    return indices[
        np.argsort(
            values[indices]
        )[::-1]
    ]


# ---------------------------------------------------------
# 3. Decode multiple candidate spans from every feature
# ---------------------------------------------------------

decode_start_time = time.time()

raw_candidate_records = []

for feature_index in range(
    len(passage_article_tokenized)
):
    feature = passage_article_tokenized[
        feature_index
    ]

    row_number = int(
        feature["row_number"]
    )

    article_context = str(
        article_context_lookup[
            row_number
        ]
    )

    offsets = feature[
        "offset_mapping"
    ]

    input_ids = feature[
        "input_ids"
    ]

    start_logits = article_start_logits[
        feature_index
    ]

    end_logits = article_end_logits[
        feature_index
    ]

    try:
        cls_index = input_ids.index(
            article_tokenizer.cls_token_id
        )
    except ValueError:
        cls_index = 0

    null_score = float(
        start_logits[cls_index]
        + end_logits[cls_index]
    )

    valid_positions = np.array([
        offset is not None
        and len(offset) == 2
        and int(offset[1]) > int(offset[0])
        for offset in offsets
    ])

    masked_start_logits = (
        start_logits.copy()
    )

    masked_end_logits = (
        end_logits.copy()
    )

    masked_start_logits[
        ~valid_positions
    ] = -np.inf

    masked_end_logits[
        ~valid_positions
    ] = -np.inf

    top_start_indices = get_top_indices(
        masked_start_logits,
        TOP_START_POSITIONS
    )

    top_end_indices = get_top_indices(
        masked_end_logits,
        TOP_END_POSITIONS
    )

    for start_position in top_start_indices:
        if not np.isfinite(
            masked_start_logits[
                start_position
            ]
        ):
            continue

        for end_position in top_end_indices:
            if not np.isfinite(
                masked_end_logits[
                    end_position
                ]
            ):
                continue

            if end_position < start_position:
                continue

            span_token_length = (
                end_position
                - start_position
                + 1
            )

            if (
                span_token_length
                > ARTICLE_MAX_ANSWER_LENGTH
            ):
                continue

            character_start = int(
                offsets[
                    start_position
                ][0]
            )

            character_end = int(
                offsets[
                    end_position
                ][1]
            )

            if character_end <= character_start:
                continue

            raw_text = article_context[
                character_start:
                character_end
            ]

            cleaned_text = clean_article_span(
                raw_text
            )

            normalized_text = (
                normalize_candidate_text(
                    cleaned_text
                )
            )

            if not normalized_text:
                continue

            span_score = float(
                start_logits[
                    start_position
                ]
                + end_logits[
                    end_position
                ]
            )

            answerability_margin = float(
                span_score - null_score
            )

            raw_candidate_records.append({
                "row_number": row_number,
                "feature_index": (
                    feature_index
                ),
                "start_position": int(
                    start_position
                ),
                "end_position": int(
                    end_position
                ),
                "character_start": (
                    character_start
                ),
                "character_end": (
                    character_end
                ),
                "raw_text": raw_text,
                "predicted_text": (
                    cleaned_text
                ),
                "normalized_text": (
                    normalized_text
                ),
                "span_score": span_score,
                "null_score": null_score,
                "answerability_margin": (
                    answerability_margin
                ),
                "word_count": len(
                    cleaned_text.split()
                )
            })

    if (
        (feature_index + 1) % 100 == 0
        or feature_index + 1
        == len(passage_article_tokenized)
    ):
        print(
            f"Decoded {feature_index + 1}/"
            f"{len(passage_article_tokenized)} "
            "features"
        )


raw_article_candidates_df = pd.DataFrame(
    raw_candidate_records
)


# ---------------------------------------------------------
# 4. Deduplicate overlapping-window candidates
# ---------------------------------------------------------

article_span_candidates_df = (
    raw_article_candidates_df
    .sort_values(
        [
            "row_number",
            "answerability_margin",
            "span_score"
        ],
        ascending=[
            True,
            False,
            False
        ]
    )
    .drop_duplicates(
        subset=[
            "row_number",
            "normalized_text"
        ],
        keep="first"
    )
    .sort_values(
        [
            "row_number",
            "answerability_margin",
            "span_score"
        ],
        ascending=[
            True,
            False,
            False
        ]
    )
    .groupby(
        "row_number",
        group_keys=False
    )
    .head(
        MAX_CANDIDATES_PER_ARTICLE
    )
    .reset_index(drop=True)
)

article_span_candidates_df[
    "candidate_rank"
] = (
    article_span_candidates_df
    .groupby(
        "row_number"
    )
    .cumcount()
    + 1
)


# ---------------------------------------------------------
# 5. Verify coverage and save the checkpoint
# ---------------------------------------------------------

expected_row_numbers = set(
    passage_article_df[
        "row_number"
    ].astype(int)
)

decoded_row_numbers = set(
    article_span_candidates_df[
        "row_number"
    ].astype(int)
)

missing_row_numbers = sorted(
    expected_row_numbers
    - decoded_row_numbers
)

article_span_candidates_df.to_pickle(
    ARTICLE_CANDIDATES_PATH
)

article_span_candidates_df[
    [
        "row_number",
        "candidate_rank",
        "predicted_text",
        "answerability_margin",
        "span_score",
        "null_score",
        "word_count",
        "feature_index",
        "character_start",
        "character_end"
    ]
].to_csv(
    ARTICLE_CANDIDATES_CSV_PATH,
    index=False
)


print(
    "\nDecoding seconds:",
    round(
        time.time()
        - decode_start_time,
        2
    )
)

print(
    "Raw span combinations:",
    len(raw_article_candidates_df)
)

print(
    "Unique retained candidates:",
    len(article_span_candidates_df)
)

print(
    "Articles with candidates:",
    article_span_candidates_df[
        "row_number"
    ].nunique()
)

print(
    "Missing articles:",
    len(missing_row_numbers)
)

print(
    "Positive-margin retained candidates:",
    int(
        article_span_candidates_df[
            "answerability_margin"
        ].gt(0).sum()
    )
)

print(
    "\nCandidates per article:"
)

print(
    article_span_candidates_df
    .groupby("row_number")
    .size()
    .describe()
)

print(
    "\nSaved pickle:",
    ARTICLE_CANDIDATES_PATH
)

print(
    "Saved CSV:",
    ARTICLE_CANDIDATES_CSV_PATH
)

print(
    "Checkpoint complete:",
    os.path.exists(
        ARTICLE_CANDIDATES_PATH
    )
    and os.path.exists(
        ARTICLE_CANDIDATES_CSV_PATH
    )
    and len(missing_row_numbers) == 0
)

print(
    "\nHighest-ranked article candidates:"
)

display(
    article_span_candidates_df[
        [
            "row_number",
            "candidate_rank",
            "predicted_text",
            "answerability_margin",
            "word_count"
        ]
    ].head(20)
)

Article rows: 168
Sliding-window features: 571
Start logits: (571, 384)
End logits: (571, 384)
Decoded 100/571 features
Decoded 200/571 features
Decoded 300/571 features
Decoded 400/571 features
Decoded 500/571 features
Decoded 571/571 features

Decoding seconds: 7.76
Raw span combinations: 73008
Unique retained candidates: 5040
Articles with candidates: 168
Missing articles: 0
Positive-margin retained candidates: 1523

Candidates per article:
count    168.0
mean      30.0
std        0.0
min       30.0
25%       30.0
50%       30.0
75%       30.0
max       30.0
dtype: float64

Saved pickle: /kaggle/working/task2_passage_v1/val_passage_article_span_candidates.pkl
Saved CSV: /kaggle/working/task2_passage_v1/val_passage_article_span_candidates.csv
Checkpoint complete: True

Highest-ranked article candidates:


,row_number,candidate_rank,predicted_text,answerability_margin,word_count
0,0,1,too dark,5.433502,2
1,0,2,it’s too dark,4.976212,3
2,0,3,because it’s too dark,4.521859,4
3,0,4,the game is being delayed because it’s too dark,4.054933,9
4,0,5,too dark. Cawthon said that some of the plot e...,3.268556,39
5,0,6,According to a post by Cawthon on the Five Nig...,3.241751,25
6,0,7,Five Nights at Freddy’s: Sister Location,3.115918,6
7,0,8,it’s too dark. Cawthon said that some of the p...,2.811265,40
8,0,9,because it’s too dark. Cawthon said that some ...,2.356912,41
9,0,10,too dark. Cawthon said that some of the plot e...,2.301766,20


In [11]:
import pandas as pd


print("Validation columns:")
print(list(val_df.columns))

print("\nBaseline audit columns:")
print(list(baseline_audit_df.columns))

print("\nParagraph candidate prediction columns:")
print(list(paragraph_candidate_predictions_df.columns))

print("\nArticle candidate columns:")
print(list(article_span_candidates_df.columns))

# Inspect validation target structures

print("\nValidation spoiler value types:")
print(
    val_df["spoiler"]
    .apply(type)
    .value_counts()
)

print("\nValidation tag value types:")
print(
    val_df["tags"]
    .apply(type)
    .value_counts()
)


print("\nThree validation examples:")
for row_number in passage_type_df[
    "row_number"
].head(3):
    row_number = int(row_number)
    row = val_df.iloc[row_number]

    print("\n" + "=" * 80)
    print("row_number:", row_number)
    print("id:", row["id"])
    print("postText:", row["postText"])
    print("tags:", row["tags"])
    print("spoiler:", row["spoiler"])

# Inspect saved test audit examples

print("\n\nFirst three baseline audit rows:")
display(
    baseline_audit_df.head(3)
)

# Alignment checks

expected_passage_rows = set(
    passage_type_df["row_number"].astype(int)
)

paragraph_rows = set(
    paragraph_candidate_predictions_df[
        "row_number"
    ].astype(int)
)

article_rows = set(
    article_span_candidates_df[
        "row_number"
    ].astype(int)
)

print("\nAlignment checks:")
print(
    "Expected predicted-passage rows:",
    len(expected_passage_rows)
)
print(
    "Paragraph rows covered:",
    len(paragraph_rows)
)
print(
    "Article rows covered:",
    len(article_rows)
)
print(
    "Paragraph missing rows:",
    len(expected_passage_rows - paragraph_rows)
)
print(
    "Article missing rows:",
    len(expected_passage_rows - article_rows)
)

Validation columns:
['postId', 'postText', 'postPlatform', 'targetParagraphs', 'targetTitle', 'targetDescription', 'targetKeywords', 'targetMedia', 'targetUrl', 'provenance', 'spoiler', 'spoilerPositions', 'tags', 'id']

Baseline audit columns:
['row_number', 'id', 'predicted_type', 'operator', 'paragraph_prediction', 'article_prediction', 'multi_candidate', 'current_rule_fallback', 'previous_045436_prediction', 'spoiler', 'word_count', 'exceeded_multi_cap', 'changed_from_045436']

Paragraph candidate prediction columns:
['candidate_index', 'row_number', 'article_id', 'source_kind', 'source_index', 'context', 'predicted_text', 'best_span_score', 'null_score', 'answerability_margin', 'feature_index', 'word_count']

Article candidate columns:
['row_number', 'feature_index', 'start_position', 'end_position', 'character_start', 'character_end', 'raw_text', 'predicted_text', 'normalized_text', 'span_score', 'null_score', 'answerability_margin', 'word_count', 'candidate_rank']

Validation sp

,row_number,id,predicted_type,operator,paragraph_prediction,article_prediction,multi_candidate,current_rule_fallback,previous_045436_prediction,spoiler,word_count,exceeded_multi_cap,changed_from_045436
0,0,0,phrase,contain_shorter_075,"balloons and a sign in hand that reads, ""Heard...","balloons and a sign in hand that reads, ""Heard...",NaN,"balloons and a sign in hand that reads, ""Heard...","balloons and a sign in hand that reads, ""Heard...","balloons and a sign in hand that reads, ""Heard...",59,False,False
1,1,1,passage,current_rule,1. Prioritise b. Invite Juan to sit in on your...,Why you SHOULD be selfish at work: Helping oth...,NaN,Why you SHOULD be selfish at work: Helping oth...,Why you SHOULD be selfish at work: Helping oth...,Why you SHOULD be selfish at work: Helping oth...,67,False,False
2,2,2,phrase,contain_shorter_075,Have a Bunch of Money,Have a Bunch of Money,NaN,Have a Bunch of Money,Have a Bunch of Money,Have a Bunch of Money,5,False,False



Alignment checks:
Expected predicted-passage rows: 168
Paragraph rows covered: 168
Article rows covered: 168
Paragraph missing rows: 0
Article missing rows: 0


## Next, build one scored candidate pool from both QA models. This will tell us how much recoverable passage performance exists before choosing or training a reranker.

In [13]:
import os
import re
import time
import numpy as np
import pandas as pd

from nltk.corpus import wordnet
from nltk.tokenize import TreebankWordTokenizer
from nltk.translate.meteor_score import meteor_score


SCORED_POOL_PATH = os.path.join(
    WORK_DIR,
    "val_passage_candidate_pool_scored.pkl"
)

SCORED_POOL_CSV_PATH = os.path.join(
    WORK_DIR,
    "val_passage_candidate_pool_scored.csv"
)

ROW_SUMMARY_PATH = os.path.join(
    WORK_DIR,
    "val_passage_candidate_oracle_summary.csv"
)

MAX_PARAGRAPH_CANDIDATES_PER_ROW = 30

tokenizer_for_meteor = TreebankWordTokenizer()

# 1. Confirm that full NLTK METEOR is available

try:
    wordnet.synsets("example")
    print("NLTK WordNet available: True")
except LookupError as error:
    raise RuntimeError(
        "NLTK WordNet is not available in this Kaggle "
        "environment, so accurate METEOR scoring cannot "
        "continue yet."
    ) from error


def normalize_pool_text(value):
    value = clean_text(value).lower()

    value = re.sub(
        r"[^\w]+",
        " ",
        value
    )

    return re.sub(
        r"\s+",
        " ",
        value
    ).strip()


def combine_gold_spoilers(value):
    if isinstance(value, list):
        return clean_text(
            " ".join(
                clean_text(item)
                for item in value
                if clean_text(item)
            )
        )

    return clean_text(value)


def calculate_meteor(
    reference_text,
    prediction_text
):
    reference_tokens = (
        tokenizer_for_meteor.tokenize(
            clean_text(reference_text)
        )
    )

    prediction_tokens = (
        tokenizer_for_meteor.tokenize(
            clean_text(prediction_text)
        )
    )

    if not reference_tokens:
        return 0.0

    if not prediction_tokens:
        return 0.0

    return float(
        meteor_score(
            [reference_tokens],
            prediction_tokens
        )
    )

# 2. Gold-answer lookup for the 168 routed passage rows

gold_lookup = {}

for row_number in sorted(
    passage_row_numbers
):
    validation_row = val_df.iloc[
        int(row_number)
    ]

    gold_lookup[int(row_number)] = {
        "gold_text": combine_gold_spoilers(
            validation_row["spoiler"]
        ),
        "gold_type": clean_text(
            validation_row["tags"][0]
        ),
        "question": clean_text(
            validation_row["postText"]
        ),
        "title": clean_text(
            validation_row[
                "targetTitle"
            ]
        )
    }

# 3. Prepare paragraph-model candidates

paragraph_pool_df = (
    paragraph_candidate_predictions_df
    .copy()
)

paragraph_pool_df[
    "normalized_text"
] = (
    paragraph_pool_df[
        "predicted_text"
    ]
    .apply(normalize_pool_text)
)

paragraph_pool_df = (
    paragraph_pool_df[
        paragraph_pool_df[
            "normalized_text"
        ].ne("")
    ]
    .sort_values(
        [
            "row_number",
            "answerability_margin",
            "best_span_score"
        ],
        ascending=[
            True,
            False,
            False
        ]
    )
    .drop_duplicates(
        subset=[
            "row_number",
            "normalized_text"
        ],
        keep="first"
    )
)

paragraph_pool_df[
    "source_rank"
] = (
    paragraph_pool_df
    .groupby("row_number")
    .cumcount()
    + 1
)

paragraph_pool_df = (
    paragraph_pool_df[
        paragraph_pool_df[
            "source_rank"
        ].le(
            MAX_PARAGRAPH_CANDIDATES_PER_ROW
        )
    ]
    .copy()
)

paragraph_pool_df[
    "source_family"
] = "paragraph"

paragraph_pool_df[
    "model_score"
] = paragraph_pool_df[
    "best_span_score"
]

# 4. Prepare article-model candidates

article_pool_df = (
    article_span_candidates_df
    .copy()
)

article_pool_df[
    "source_rank"
] = article_pool_df[
    "candidate_rank"
]

article_pool_df[
    "source_family"
] = "article"

article_pool_df[
    "source_kind"
] = "full_article"

article_pool_df[
    "source_index"
] = -1

article_pool_df[
    "model_score"
] = article_pool_df[
    "span_score"
]

# 5. Standardize and combine both sources

pool_columns = [
    "row_number",
    "source_family",
    "source_kind",
    "source_index",
    "source_rank",
    "predicted_text",
    "normalized_text",
    "model_score",
    "null_score",
    "answerability_margin",
    "word_count"
]

combined_candidate_pool_df = pd.concat(
    [
        paragraph_pool_df[
            pool_columns
        ],
        article_pool_df[
            pool_columns
        ]
    ],
    ignore_index=True
)

combined_candidate_pool_df[
    "row_number"
] = (
    combined_candidate_pool_df[
        "row_number"
    ].astype(int)
)

combined_candidate_pool_df[
    "gold_text"
] = (
    combined_candidate_pool_df[
        "row_number"
    ].map(
        lambda row_number:
        gold_lookup[row_number][
            "gold_text"
        ]
    )
)

combined_candidate_pool_df[
    "gold_type"
] = (
    combined_candidate_pool_df[
        "row_number"
    ].map(
        lambda row_number:
        gold_lookup[row_number][
            "gold_type"
        ]
    )
)

combined_candidate_pool_df[
    "question"
] = (
    combined_candidate_pool_df[
        "row_number"
    ].map(
        lambda row_number:
        gold_lookup[row_number][
            "question"
        ]
    )
)

combined_candidate_pool_df[
    "title"
] = (
    combined_candidate_pool_df[
        "row_number"
    ].map(
        lambda row_number:
        gold_lookup[row_number][
            "title"
        ]
    )
)

# 6. Calculate candidate-level METEOR

score_start_time = time.time()

meteor_values = []

for candidate_number, row in enumerate(
    combined_candidate_pool_df.itertuples(
        index=False
    ),
    start=1
):
    meteor_values.append(
        calculate_meteor(
            row.gold_text,
            row.predicted_text
        )
    )

    if (
        candidate_number % 1000 == 0
        or candidate_number
        == len(combined_candidate_pool_df)
    ):
        print(
            f"Scored {candidate_number}/"
            f"{len(combined_candidate_pool_df)} "
            "candidates"
        )

combined_candidate_pool_df[
    "meteor"
] = meteor_values

# 7. Build per-row top-one and oracle comparisons

def select_row_score(
    dataframe,
    source_family,
    selection_method
):
    source_df = dataframe[
        dataframe[
            "source_family"
        ].eq(source_family)
    ].copy()

    if selection_method == "top1":
        selected = (
            source_df[
                source_df[
                    "source_rank"
                ].eq(1)
            ]
            .set_index("row_number")[
                "meteor"
            ]
        )

    elif selection_method == "oracle":
        selected = (
            source_df
            .groupby("row_number")[
                "meteor"
            ]
            .max()
        )

    else:
        raise ValueError(
            selection_method
        )

    return selected


article_top1 = select_row_score(
    combined_candidate_pool_df,
    "article",
    "top1"
)

paragraph_top1 = select_row_score(
    combined_candidate_pool_df,
    "paragraph",
    "top1"
)

article_oracle = select_row_score(
    combined_candidate_pool_df,
    "article",
    "oracle"
)

paragraph_oracle = select_row_score(
    combined_candidate_pool_df,
    "paragraph",
    "oracle"
)

combined_oracle = (
    combined_candidate_pool_df
    .groupby("row_number")[
        "meteor"
    ]
    .max()
)

row_summary_df = pd.DataFrame({
    "row_number": sorted(
        passage_row_numbers
    )
})

row_summary_df[
    "gold_type"
] = row_summary_df[
    "row_number"
].map(
    lambda row_number:
    gold_lookup[int(row_number)][
        "gold_type"
    ]
)

row_summary_df[
    "article_top1_meteor"
] = row_summary_df[
    "row_number"
].map(article_top1)

row_summary_df[
    "paragraph_top1_meteor"
] = row_summary_df[
    "row_number"
].map(paragraph_top1)

row_summary_df[
    "article_oracle_meteor"
] = row_summary_df[
    "row_number"
].map(article_oracle)

row_summary_df[
    "paragraph_oracle_meteor"
] = row_summary_df[
    "row_number"
].map(paragraph_oracle)

row_summary_df[
    "combined_oracle_meteor"
] = row_summary_df[
    "row_number"
].map(combined_oracle)

row_summary_df[
    "best_top1_meteor"
] = row_summary_df[
    [
        "article_top1_meteor",
        "paragraph_top1_meteor"
    ]
].max(axis=1)

# 8. Save all scoring checkpoints

combined_candidate_pool_df.to_pickle(
    SCORED_POOL_PATH
)

combined_candidate_pool_df.to_csv(
    SCORED_POOL_CSV_PATH,
    index=False
)

row_summary_df.to_csv(
    ROW_SUMMARY_PATH,
    index=False
)

# 9. Report overall and true-passage results

def print_score_summary(
    dataframe,
    label
):
    print(f"\n{label}")
    print("-" * len(label))

    for column in [
        "article_top1_meteor",
        "paragraph_top1_meteor",
        "best_top1_meteor",
        "article_oracle_meteor",
        "paragraph_oracle_meteor",
        "combined_oracle_meteor"
    ]:
        print(
            f"{column}: "
            f"{dataframe[column].mean():.6f}"
        )


print(
    "\nScoring seconds:",
    round(
        time.time()
        - score_start_time,
        2
    )
)

print(
    "Combined candidates:",
    len(combined_candidate_pool_df)
)

print(
    "Article candidates:",
    int(
        combined_candidate_pool_df[
            "source_family"
        ].eq("article").sum()
    )
)

print(
    "Paragraph candidates:",
    int(
        combined_candidate_pool_df[
            "source_family"
        ].eq("paragraph").sum()
    )
)

print_score_summary(
    row_summary_df,
    "All 168 predicted-passage rows"
)

true_passage_summary_df = (
    row_summary_df[
        row_summary_df[
            "gold_type"
        ].eq("passage")
    ]
)

print_score_summary(
    true_passage_summary_df,
    (
        "True-passage rows inside the "
        "predicted-passage group"
    )
)

print(
    "\nTrue-passage row count:",
    len(true_passage_summary_df)
)

print(
    "Rows where article oracle is better:",
    int(
        row_summary_df[
            "article_oracle_meteor"
        ].gt(
            row_summary_df[
                "paragraph_oracle_meteor"
            ]
        ).sum()
    )
)

print(
    "Rows where paragraph oracle is better:",
    int(
        row_summary_df[
            "paragraph_oracle_meteor"
        ].gt(
            row_summary_df[
                "article_oracle_meteor"
            ]
        ).sum()
    )
)

print(
    "Oracle ties:",
    int(
        row_summary_df[
            "paragraph_oracle_meteor"
        ].eq(
            row_summary_df[
                "article_oracle_meteor"
            ]
        ).sum()
    )
)

print("\nSaved scored candidate pool:")
print(SCORED_POOL_PATH)

print("\nSaved row summary:")
print(ROW_SUMMARY_PATH)

print(
    "\nCheckpoint complete:",
    os.path.exists(
        SCORED_POOL_PATH
    )
    and os.path.exists(
        ROW_SUMMARY_PATH
    )
)

NLTK WordNet available: True
Scored 1000/7267 candidates
Scored 2000/7267 candidates
Scored 3000/7267 candidates
Scored 4000/7267 candidates
Scored 5000/7267 candidates
Scored 6000/7267 candidates
Scored 7000/7267 candidates
Scored 7267/7267 candidates

Scoring seconds: 7.91
Combined candidates: 7267
Article candidates: 5040
Paragraph candidates: 2227

All 168 predicted-passage rows
------------------------------
article_top1_meteor: 0.359408
paragraph_top1_meteor: 0.249593
best_top1_meteor: 0.411231
article_oracle_meteor: 0.789584
paragraph_oracle_meteor: 0.629833
combined_oracle_meteor: 0.849458

True-passage rows inside the predicted-passage group
----------------------------------------------------
article_top1_meteor: 0.307908
paragraph_top1_meteor: 0.227413
best_top1_meteor: 0.360698
article_oracle_meteor: 0.784722
paragraph_oracle_meteor: 0.618151
combined_oracle_meteor: 0.863255

True-passage row count: 127
Rows where article oracle is better: 106
Rows where paragraph oracle is

In [14]:
import os
import re
import time
import numpy as np
import pandas as pd

from sklearn.ensemble import (
    ExtraTreesRegressor,
    HistGradientBoostingRegressor
)
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


RERANKER_OOF_PATH = os.path.join(
    WORK_DIR,
    "val_passage_reranker_oof_predictions.pkl"
)

RERANKER_SUMMARY_PATH = os.path.join(
    WORK_DIR,
    "val_passage_reranker_cv_summary.csv"
)

RERANKER_SELECTIONS_PATH = os.path.join(
    WORK_DIR,
    "val_passage_reranker_cv_selections.csv"
)


# 1. Start from the scored candidate pool

reranker_df = (
    combined_candidate_pool_df
    .copy()
    .reset_index(drop=True)
)

reranker_df["candidate_row_id"] = np.arange(
    len(reranker_df)
)


# 2. Add type-classifier probabilities

type_feature_df = (
    val_type_predictions_df[
        [
            "row_number",
            "prob_phrase",
            "prob_passage",
            "prob_multi",
            "confidence"
        ]
    ]
    .copy()
)

reranker_df = reranker_df.merge(
    type_feature_df,
    on="row_number",
    how="left",
    validate="many_to_one"
)


# 3. Add article-position information

article_position_df = (
    article_span_candidates_df[
        [
            "row_number",
            "normalized_text",
            "character_start",
            "character_end"
        ]
    ]
    .drop_duplicates(
        subset=[
            "row_number",
            "normalized_text"
        ],
        keep="first"
    )
)

reranker_df = reranker_df.merge(
    article_position_df,
    on=[
        "row_number",
        "normalized_text"
    ],
    how="left",
    validate="many_to_one"
)

article_character_lengths = (
    passage_article_df
    .set_index("row_number")[
        "article_context"
    ]
    .astype(str)
    .str.len()
    .to_dict()
)

reranker_df[
    "article_character_length"
] = (
    reranker_df["row_number"]
    .map(article_character_lengths)
    .fillna(1)
    .clip(lower=1)
)

reranker_df[
    "article_start_fraction"
] = np.where(
    reranker_df["source_family"].eq(
        "article"
    ),
    reranker_df["character_start"].fillna(0)
    / reranker_df["article_character_length"],
    -1.0
)

reranker_df[
    "article_span_fraction"
] = np.where(
    reranker_df["source_family"].eq(
        "article"
    ),
    (
        reranker_df["character_end"].fillna(0)
        - reranker_df["character_start"].fillna(0)
    ).clip(lower=0)
    / reranker_df["article_character_length"],
    -1.0
)

# 4. Add paragraph-position information

paragraph_max_indices = (
    passage_candidate_df[
        passage_candidate_df[
            "source_kind"
        ].eq("paragraph")
    ]
    .groupby("row_number")[
        "source_index"
    ]
    .max()
    .clip(lower=1)
    .to_dict()
)

reranker_df[
    "paragraph_max_index"
] = (
    reranker_df["row_number"]
    .map(paragraph_max_indices)
    .fillna(1)
)

reranker_df[
    "paragraph_position_fraction"
] = np.where(
    reranker_df[
        "source_family"
    ].eq("paragraph"),
    reranker_df[
        "source_index"
    ].clip(lower=0)
    / reranker_df[
        "paragraph_max_index"
    ],
    -1.0
)

# 5. Text feature helpers

def get_word_tokens(text):
    return re.findall(
        r"\b\w+\b",
        clean_text(text).lower()
    )


def get_word_set(text):
    return set(
        get_word_tokens(text)
    )


def safe_jaccard(
    first_set,
    second_set
):
    union = first_set | second_set

    if not union:
        return 0.0

    return len(
        first_set & second_set
    ) / len(union)


def safe_coverage(
    first_set,
    second_set
):
    if not first_set:
        return 0.0

    return len(
        first_set & second_set
    ) / len(first_set)


candidate_token_sets = (
    reranker_df[
        "predicted_text"
    ]
    .apply(get_word_set)
)

question_token_sets = (
    reranker_df[
        "question"
    ]
    .apply(get_word_set)
)

title_token_sets = (
    reranker_df[
        "title"
    ]
    .apply(get_word_set)
)

reranker_df[
    "question_jaccard"
] = [
    safe_jaccard(
        candidate_tokens,
        question_tokens
    )
    for candidate_tokens, question_tokens
    in zip(
        candidate_token_sets,
        question_token_sets
    )
]

reranker_df[
    "question_candidate_coverage"
] = [
    safe_coverage(
        candidate_tokens,
        question_tokens
    )
    for candidate_tokens, question_tokens
    in zip(
        candidate_token_sets,
        question_token_sets
    )
]

reranker_df[
    "title_jaccard"
] = [
    safe_jaccard(
        candidate_tokens,
        title_tokens
    )
    for candidate_tokens, title_tokens
    in zip(
        candidate_token_sets,
        title_token_sets
    )
]

reranker_df[
    "title_candidate_coverage"
] = [
    safe_coverage(
        candidate_tokens,
        title_tokens
    )
    for candidate_tokens, title_tokens
    in zip(
        candidate_token_sets,
        title_token_sets
    )
]

# 6. Candidate text-shape features

candidate_text_series = (
    reranker_df[
        "predicted_text"
    ]
    .fillna("")
    .astype(str)
)

reranker_df[
    "character_count"
] = candidate_text_series.str.len()

reranker_df[
    "log_word_count"
] = np.log1p(
    reranker_df["word_count"]
)

reranker_df[
    "comma_count"
] = candidate_text_series.str.count(",")

reranker_df[
    "period_count"
] = candidate_text_series.str.count(
        r"\."
    )

reranker_df[
    "question_mark_count"
] = candidate_text_series.str.count(
        r"\?"
    )

reranker_df[
    "exclamation_count"
] = candidate_text_series.str.count("!")

reranker_df[
    "quote_count"
] = (
    candidate_text_series.str.count('"')
    + candidate_text_series.str.count("“")
    + candidate_text_series.str.count("”")
)

reranker_df[
    "starts_lowercase"
] = candidate_text_series.apply(
    lambda text:
    float(
        bool(text)
        and text[0].islower()
    )
)

reranker_df[
    "ends_with_terminal_punctuation"
] = candidate_text_series.apply(
    lambda text:
    float(
        text.rstrip().endswith(
            (".", "!", "?", '"', "”")
        )
    )
)

reranker_df[
    "contains_paragraph_marker"
] = candidate_text_series.str.contains(
    r"\bParagraph\s+\d+\b",
    case=False,
    regex=True
).astype(float)


# 7. Source and rank features

reranker_df[
    "is_article_model"
] = reranker_df[
    "source_family"
].eq("article").astype(float)

reranker_df[
    "is_paragraph_model"
] = reranker_df[
    "source_family"
].eq("paragraph").astype(float)

reranker_df[
    "is_title_context"
] = reranker_df[
    "source_kind"
].eq("title").astype(float)

reranker_df[
    "is_paragraph_context"
] = reranker_df[
    "source_kind"
].eq("paragraph").astype(float)

reranker_df[
    "is_full_article_context"
] = reranker_df[
    "source_kind"
].eq("full_article").astype(float)

reranker_df[
    "reciprocal_source_rank"
] = 1.0 / reranker_df[
    "source_rank"
].clip(lower=1)

reranker_df[
    "log_source_rank"
] = np.log1p(
    reranker_df["source_rank"]
)

# 8. Within-family score normalization

family_group = reranker_df.groupby(
    [
        "row_number",
        "source_family"
    ]
)

for score_column in [
    "model_score",
    "answerability_margin",
    "word_count"
]:
    family_max = family_group[
        score_column
    ].transform("max")

    family_mean = family_group[
        score_column
    ].transform("mean")

    family_std = family_group[
        score_column
    ].transform("std").replace(
        0,
        np.nan
    )

    reranker_df[
        f"{score_column}_gap_from_family_max"
    ] = (
        family_max
        - reranker_df[score_column]
    )

    reranker_df[
        f"{score_column}_family_z"
    ] = (
        reranker_df[score_column]
        - family_mean
    ) / family_std

reranker_df[
    "overall_margin_rank"
] = (
    reranker_df
    .groupby("row_number")[
        "answerability_margin"
    ]
    .rank(
        method="min",
        ascending=False
    )
)

reranker_df[
    "reciprocal_overall_margin_rank"
] = 1.0 / reranker_df[
    "overall_margin_rank"
].clip(lower=1)

# 9. Cross-family agreement features

reranker_df[
    "cross_family_max_jaccard"
] = 0.0

reranker_df[
    "cross_family_max_containment"
] = 0.0

reranker_df[
    "cross_family_exact_match"
] = 0.0

candidate_sets_by_index = {
    index: token_set
    for index, token_set
    in zip(
        reranker_df.index,
        candidate_token_sets
    )
}

for row_number, row_group in reranker_df.groupby(
    "row_number"
):
    article_indices = row_group[
        row_group[
            "source_family"
        ].eq("article")
    ].index.tolist()

    paragraph_indices = row_group[
        row_group[
            "source_family"
        ].eq("paragraph")
    ].index.tolist()

    for current_indices, other_indices in [
        (
            article_indices,
            paragraph_indices
        ),
        (
            paragraph_indices,
            article_indices
        )
    ]:
        for current_index in current_indices:
            current_set = (
                candidate_sets_by_index[
                    current_index
                ]
            )

            current_normalized = (
                reranker_df.at[
                    current_index,
                    "normalized_text"
                ]
            )

            maximum_jaccard = 0.0
            maximum_containment = 0.0
            exact_match = 0.0

            for other_index in other_indices:
                other_set = (
                    candidate_sets_by_index[
                        other_index
                    ]
                )

                intersection_size = len(
                    current_set & other_set
                )

                union_size = len(
                    current_set | other_set
                )

                if union_size:
                    maximum_jaccard = max(
                        maximum_jaccard,
                        intersection_size
                        / union_size
                    )

                smaller_size = min(
                    len(current_set),
                    len(other_set)
                )

                if smaller_size:
                    maximum_containment = max(
                        maximum_containment,
                        intersection_size
                        / smaller_size
                    )

                if (
                    current_normalized
                    ==
                    reranker_df.at[
                        other_index,
                        "normalized_text"
                    ]
                ):
                    exact_match = 1.0

            reranker_df.at[
                current_index,
                "cross_family_max_jaccard"
            ] = maximum_jaccard

            reranker_df.at[
                current_index,
                "cross_family_max_containment"
            ] = maximum_containment

            reranker_df.at[
                current_index,
                "cross_family_exact_match"
            ] = exact_match


# 10. Final deployable feature matrix

feature_columns = [
    "model_score",
    "null_score",
    "answerability_margin",
    "word_count",
    "character_count",
    "log_word_count",
    "source_rank",
    "reciprocal_source_rank",
    "log_source_rank",
    "source_index",
    "is_article_model",
    "is_paragraph_model",
    "is_title_context",
    "is_paragraph_context",
    "is_full_article_context",
    "article_start_fraction",
    "article_span_fraction",
    "paragraph_position_fraction",
    "question_jaccard",
    "question_candidate_coverage",
    "title_jaccard",
    "title_candidate_coverage",
    "comma_count",
    "period_count",
    "question_mark_count",
    "exclamation_count",
    "quote_count",
    "starts_lowercase",
    "ends_with_terminal_punctuation",
    "contains_paragraph_marker",
    "model_score_gap_from_family_max",
    "model_score_family_z",
    "answerability_margin_gap_from_family_max",
    "answerability_margin_family_z",
    "word_count_gap_from_family_max",
    "word_count_family_z",
    "overall_margin_rank",
    "reciprocal_overall_margin_rank",
    "cross_family_max_jaccard",
    "cross_family_max_containment",
    "cross_family_exact_match",
    "prob_phrase",
    "prob_passage",
    "prob_multi",
    "confidence"
]

feature_matrix = (
    reranker_df[
        feature_columns
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .fillna(0.0)
    .astype(float)
)

target_values = (
    reranker_df["meteor"]
    .astype(float)
    .to_numpy()
)

group_sizes = (
    reranker_df
    .groupby("row_number")
    ["candidate_row_id"]
    .transform("count")
    .to_numpy()
)

sample_weights = (
    1.0
    / group_sizes
)

# 11. Five-fold row-level cross-validation

unique_rows_df = (
    reranker_df[
        [
            "row_number",
            "gold_type"
        ]
    ]
    .drop_duplicates()
    .sort_values("row_number")
    .reset_index(drop=True)
)

splitter = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=641
)

models = {
    "ridge": Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            Ridge(
                alpha=10.0
            )
        )
    ]),

    "hist_gradient_boosting":
        HistGradientBoostingRegressor(
            learning_rate=0.05,
            max_iter=250,
            max_leaf_nodes=15,
            min_samples_leaf=20,
            l2_regularization=1.0,
            random_state=641
        ),

    "extra_trees":
        ExtraTreesRegressor(
            n_estimators=350,
            max_depth=14,
            min_samples_leaf=4,
            max_features=0.8,
            n_jobs=-1,
            random_state=641
        )
}

oof_predictions = {
    model_name: np.full(
        len(reranker_df),
        np.nan
    )
    for model_name in models
}

cv_start_time = time.time()

for fold_number, (
    training_row_indices,
    validation_row_indices
) in enumerate(
    splitter.split(
        unique_rows_df[
            "row_number"
        ],
        unique_rows_df[
            "gold_type"
        ]
    ),
    start=1
):
    training_rows = set(
        unique_rows_df.iloc[
            training_row_indices
        ]["row_number"]
    )

    validation_rows = set(
        unique_rows_df.iloc[
            validation_row_indices
        ]["row_number"]
    )

    training_mask = (
        reranker_df[
            "row_number"
        ].isin(training_rows)
        .to_numpy()
    )

    validation_mask = (
        reranker_df[
            "row_number"
        ].isin(validation_rows)
        .to_numpy()
    )

    print(
        f"\nFold {fold_number}: "
        f"{len(training_rows)} train rows, "
        f"{len(validation_rows)} validation rows"
    )

    for model_name, model in models.items():
        if model_name == "ridge":
            model.fit(
                feature_matrix.loc[
                    training_mask
                ],
                target_values[
                    training_mask
                ],
                model__sample_weight=(
                    sample_weights[
                        training_mask
                    ]
                )
            )

        else:
            model.fit(
                feature_matrix.loc[
                    training_mask
                ],
                target_values[
                    training_mask
                ],
                sample_weight=(
                    sample_weights[
                        training_mask
                    ]
                )
            )

        oof_predictions[
            model_name
        ][validation_mask] = model.predict(
            feature_matrix.loc[
                validation_mask
            ]
        )

# 12. Add simple prediction blends

oof_predictions[
    "tree_blend"
] = (
    oof_predictions[
        "hist_gradient_boosting"
    ]
    + oof_predictions[
        "extra_trees"
    ]
) / 2.0

oof_predictions[
    "all_model_blend"
] = (
    oof_predictions["ridge"]
    + oof_predictions[
        "hist_gradient_boosting"
    ]
    + oof_predictions[
        "extra_trees"
    ]
) / 3.0

for model_name, predictions in (
    oof_predictions.items()
):
    reranker_df[
        f"oof_{model_name}"
    ] = predictions

# 13. Select one candidate per held-out row
summary_records = []
selection_frames = []

for model_name in oof_predictions:
    prediction_column = (
        f"oof_{model_name}"
    )

    selected_indices = (
        reranker_df
        .groupby("row_number")[
            prediction_column
        ]
        .idxmax()
    )

    selected_df = (
        reranker_df.loc[
            selected_indices
        ]
        .copy()
        .sort_values("row_number")
    )

    selected_df[
        "reranker_name"
    ] = model_name

    all_rows_score = float(
        selected_df["meteor"].mean()
    )

    true_passage_score = float(
        selected_df[
            selected_df[
                "gold_type"
            ].eq("passage")
        ]["meteor"].mean()
    )

    article_selection_rate = float(
        selected_df[
            "source_family"
        ].eq("article").mean()
    )

    summary_records.append({
        "reranker": model_name,
        "all_168_meteor": (
            all_rows_score
        ),
        "true_passage_meteor": (
            true_passage_score
        ),
        "article_selection_rate": (
            article_selection_rate
        ),
        "paragraph_selection_rate": (
            1.0
            - article_selection_rate
        )
    })

    selection_frames.append(
        selected_df[
            [
                "row_number",
                "gold_type",
                "reranker_name",
                "source_family",
                "source_kind",
                "source_rank",
                "predicted_text",
                "meteor",
                prediction_column
            ]
        ].rename(
            columns={
                prediction_column:
                "predicted_candidate_score"
            }
        )
    )


reranker_summary_df = (
    pd.DataFrame(
        summary_records
    )
    .sort_values(
        "all_168_meteor",
        ascending=False
    )
    .reset_index(drop=True)
)

reranker_selections_df = pd.concat(
    selection_frames,
    ignore_index=True
)

# 14. Save checkpoints

reranker_df.to_pickle(
    RERANKER_OOF_PATH
)

reranker_summary_df.to_csv(
    RERANKER_SUMMARY_PATH,
    index=False
)

reranker_selections_df.to_csv(
    RERANKER_SELECTIONS_PATH,
    index=False
)

# 15. Report results
print(
    "\nCross-validation seconds:",
    round(
        time.time()
        - cv_start_time,
        2
    )
)

print(
    "All OOF predictions complete:",
    all(
        np.isfinite(
            predictions
        ).all()
        for predictions
        in oof_predictions.values()
    )
)

print(
    "\nReference candidate-selection scores:"
)

print(
    "Article top-1:",
    round(
        row_summary_df[
            "article_top1_meteor"
        ].mean(),
        6
    )
)

print(
    "Paragraph top-1:",
    round(
        row_summary_df[
            "paragraph_top1_meteor"
        ].mean(),
        6
    )
)

print(
    "Gold-based best of the two top-1 candidates:",
    round(
        row_summary_df[
            "best_top1_meteor"
        ].mean(),
        6
    )
)

print(
    "Combined candidate oracle:",
    round(
        row_summary_df[
            "combined_oracle_meteor"
        ].mean(),
        6
    )
)

print(
    "\nDeployable row-held-out reranker results:"
)

display(
    reranker_summary_df
)

print(
    "\nSaved OOF candidate table:",
    RERANKER_OOF_PATH
)

print(
    "Saved summary:",
    RERANKER_SUMMARY_PATH
)

print(
    "Saved selections:",
    RERANKER_SELECTIONS_PATH
)

print(
    "\nCheckpoint complete:",
    os.path.exists(
        RERANKER_OOF_PATH
    )
    and os.path.exists(
        RERANKER_SUMMARY_PATH
    )
    and os.path.exists(
        RERANKER_SELECTIONS_PATH
    )
)


Fold 1: 134 train rows, 34 validation rows

Fold 2: 134 train rows, 34 validation rows

Fold 3: 134 train rows, 34 validation rows

Fold 4: 135 train rows, 33 validation rows

Fold 5: 135 train rows, 33 validation rows

Cross-validation seconds: 18.21
All OOF predictions complete: True

Reference candidate-selection scores:
Article top-1: 0.359408
Paragraph top-1: 0.249593
Gold-based best of the two top-1 candidates: 0.411231
Combined candidate oracle: 0.849458

Deployable row-held-out reranker results:


,reranker,all_168_meteor,true_passage_meteor,article_selection_rate,paragraph_selection_rate
0,hist_gradient_boosting,0.403512,0.422685,0.952381,0.047619
1,all_model_blend,0.391094,0.423413,0.976190,0.023810
2,tree_blend,0.376480,0.410580,0.952381,0.047619
3,extra_trees,0.359901,0.379825,0.851190,0.148810
4,ridge,0.344614,0.383603,0.994048,0.005952



Saved OOF candidate table: /kaggle/working/task2_passage_v1/val_passage_reranker_oof_predictions.pkl
Saved summary: /kaggle/working/task2_passage_v1/val_passage_reranker_cv_summary.csv
Saved selections: /kaggle/working/task2_passage_v1/val_passage_reranker_cv_selections.csv

Checkpoint complete: True


In [16]:
import os
import time
import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import StratifiedKFold


REPEATED_CV_PATH = os.path.join(
    WORK_DIR,
    "val_passage_hist_repeated_cv_summary.csv"
)

REPEATED_OOF_PATH = os.path.join(
    WORK_DIR,
    "val_passage_hist_repeated_oof.npz"
)

CONSENSUS_SELECTION_PATH = os.path.join(
    WORK_DIR,
    "val_passage_hist_consensus_selections.csv"
)

SPLIT_SEEDS = [
    17,
    83,
    641,
    2026,
    9917
]

HISTORICAL_PASSAGE_ROUTE_SCORE = 0.387260
HISTORICAL_OVERALL_SCORE = 0.416904

number_of_candidates = len(
    reranker_df
)

repeat_oof_predictions = np.full(
    (
        len(SPLIT_SEEDS),
        number_of_candidates
    ),
    np.nan,
    dtype=float
)

repeat_selected_candidate_ids = np.full(
    (
        len(SPLIT_SEEDS),
        len(unique_rows_df)
    ),
    -1,
    dtype=int
)

repeat_summary_records = []

cv_start_time = time.time()

# Repeated row-held-out cross-validation

for repeat_index, split_seed in enumerate(
    SPLIT_SEEDS
):
    splitter = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=split_seed
    )

    current_oof = np.full(
        number_of_candidates,
        np.nan,
        dtype=float
    )

    print(
        f"\nRepeat {repeat_index + 1}/"
        f"{len(SPLIT_SEEDS)} — seed {split_seed}"
    )

    for fold_number, (
        training_row_indices,
        validation_row_indices
    ) in enumerate(
        splitter.split(
            unique_rows_df["row_number"],
            unique_rows_df["gold_type"]
        ),
        start=1
    ):
        training_rows = set(
            unique_rows_df.iloc[
                training_row_indices
            ]["row_number"]
        )

        validation_rows = set(
            unique_rows_df.iloc[
                validation_row_indices
            ]["row_number"]
        )

        training_mask = (
            reranker_df["row_number"]
            .isin(training_rows)
            .to_numpy()
        )

        validation_mask = (
            reranker_df["row_number"]
            .isin(validation_rows)
            .to_numpy()
        )

        model = HistGradientBoostingRegressor(
            learning_rate=0.05,
            max_iter=250,
            max_leaf_nodes=15,
            min_samples_leaf=20,
            l2_regularization=1.0,
            random_state=split_seed
        )

        model.fit(
            feature_matrix.loc[
                training_mask
            ],
            target_values[
                training_mask
            ],
            sample_weight=sample_weights[
                training_mask
            ]
        )

        current_oof[
            validation_mask
        ] = model.predict(
            feature_matrix.loc[
                validation_mask
            ]
        )

        print(
            f"  Fold {fold_number}: "
            f"{len(validation_rows)} held-out rows"
        )

    if not np.isfinite(
        current_oof
    ).all():
        raise RuntimeError(
            f"Incomplete OOF predictions for seed "
            f"{split_seed}."
        )

    repeat_oof_predictions[
        repeat_index
    ] = current_oof

    temporary_df = reranker_df[
        [
            "candidate_row_id",
            "row_number",
            "gold_type",
            "source_family",
            "meteor"
        ]
    ].copy()

    temporary_df[
        "oof_prediction"
    ] = current_oof

    selected_indices = (
        temporary_df
        .groupby("row_number")[
            "oof_prediction"
        ]
        .idxmax()
    )

    selected_df = (
        temporary_df.loc[
            selected_indices
        ]
        .sort_values("row_number")
        .reset_index(drop=True)
    )

    repeat_selected_candidate_ids[
        repeat_index
    ] = selected_df[
        "candidate_row_id"
    ].to_numpy()

    all_rows_score = float(
        selected_df["meteor"].mean()
    )

    true_passage_score = float(
        selected_df[
            selected_df["gold_type"].eq(
                "passage"
            )
        ]["meteor"].mean()
    )

    article_selection_rate = float(
        selected_df[
            "source_family"
        ].eq("article").mean()
    )

    repeat_summary_records.append({
        "seed": split_seed,
        "all_168_meteor": all_rows_score,
        "true_passage_meteor": (
            true_passage_score
        ),
        "article_selection_rate": (
            article_selection_rate
        ),
        "paragraph_selection_rate": (
            1.0 - article_selection_rate
        )
    })

# Average the five independently held-out predictions

consensus_oof_predictions = (
    repeat_oof_predictions.mean(
        axis=0
    )
)

consensus_df = reranker_df.copy()

consensus_df[
    "consensus_oof_prediction"
] = consensus_oof_predictions

consensus_selected_indices = (
    consensus_df
    .groupby("row_number")[
        "consensus_oof_prediction"
    ]
    .idxmax()
)

consensus_selected_df = (
    consensus_df.loc[
        consensus_selected_indices
    ]
    .copy()
    .sort_values("row_number")
    .reset_index(drop=True)
)

consensus_all_score = float(
    consensus_selected_df[
        "meteor"
    ].mean()
)

consensus_true_passage_score = float(
    consensus_selected_df[
        consensus_selected_df[
            "gold_type"
        ].eq("passage")
    ]["meteor"].mean()
)

consensus_article_rate = float(
    consensus_selected_df[
        "source_family"
    ].eq("article").mean()
)

# Selection stability across repeated splits

selection_agreement_values = []

for row_position in range(
    repeat_selected_candidate_ids.shape[1]
):
    selected_ids = (
        repeat_selected_candidate_ids[
            :,
            row_position
        ]
    )

    _, selection_counts = np.unique(
        selected_ids,
        return_counts=True
    )

    selection_agreement_values.append(
        selection_counts.max()
        / len(SPLIT_SEEDS)
    )

mean_selection_agreement = float(
    np.mean(
        selection_agreement_values
    )
)

full_selection_agreement_rate = float(
    np.mean(
        np.array(
            selection_agreement_values
        ) == 1.0
    )
)

# Estimate impact on the full validation set

passage_subset_gain = (
    consensus_all_score
    - HISTORICAL_PASSAGE_ROUTE_SCORE
)

estimated_full_validation_gain = (
    passage_subset_gain
    * len(passage_type_df)
    / len(val_df)
)

estimated_full_validation_score = (
    HISTORICAL_OVERALL_SCORE
    + estimated_full_validation_gain
)

# Save results

repeated_cv_summary_df = pd.DataFrame(
    repeat_summary_records
)

repeated_cv_summary_df.to_csv(
    REPEATED_CV_PATH,
    index=False
)

np.savez_compressed(
    REPEATED_OOF_PATH,
    split_seeds=np.array(
        SPLIT_SEEDS
    ),
    repeat_oof_predictions=(
        repeat_oof_predictions
    ),
    consensus_oof_predictions=(
        consensus_oof_predictions
    ),
    repeat_selected_candidate_ids=(
        repeat_selected_candidate_ids
    )
)

consensus_selected_df[
    [
        "row_number",
        "gold_type",
        "source_family",
        "source_kind",
        "source_rank",
        "predicted_text",
        "meteor",
        "consensus_oof_prediction"
    ]
].to_csv(
    CONSENSUS_SELECTION_PATH,
    index=False
)

# Report stability

print(
    "\nRepeated-CV results:"
)

display(
    repeated_cv_summary_df
)

print(
    "\nRepeated-CV score summary:"
)

print(
    repeated_cv_summary_df[
        [
            "all_168_meteor",
            "true_passage_meteor",
            "article_selection_rate"
        ]
    ].describe()
)

print(
    "\nConsensus held-out selection:"
)

print(
    "All 168 METEOR:",
    round(
        consensus_all_score,
        6
    )
)

print(
    "True-passage METEOR:",
    round(
        consensus_true_passage_score,
        6
    )
)

print(
    "Article selection rate:",
    round(
        consensus_article_rate,
        6
    )
)

print(
    "Mean candidate-selection agreement:",
    round(
        mean_selection_agreement,
        6
    )
)

print(
    "Rows with identical selection "
    "across all repeats:",
    round(
        full_selection_agreement_rate,
        6
    )
)

print(
    "\nHistorical passage-route score:",
    HISTORICAL_PASSAGE_ROUTE_SCORE
)

print(
    "Consensus passage-subset gain:",
    round(
        passage_subset_gain,
        6
    )
)

print(
    "Estimated full-validation gain:",
    round(
        estimated_full_validation_gain,
        6
    )
)

print(
    "Estimated full-validation score:",
    round(
        estimated_full_validation_score,
        6
    )
)

print(
    "\nCross-validation seconds:",
    round(
        time.time() - cv_start_time,
        2
    )
)

print(
    "\nCheckpoint complete:",
    os.path.exists(
        REPEATED_CV_PATH
    )
    and os.path.exists(
        REPEATED_OOF_PATH
    )
    and os.path.exists(
        CONSENSUS_SELECTION_PATH
    )
)


Repeat 1/5 — seed 17
  Fold 1: 34 held-out rows
  Fold 2: 34 held-out rows
  Fold 3: 34 held-out rows
  Fold 4: 33 held-out rows
  Fold 5: 33 held-out rows

Repeat 2/5 — seed 83
  Fold 1: 34 held-out rows
  Fold 2: 34 held-out rows
  Fold 3: 34 held-out rows
  Fold 4: 33 held-out rows
  Fold 5: 33 held-out rows

Repeat 3/5 — seed 641
  Fold 1: 34 held-out rows
  Fold 2: 34 held-out rows
  Fold 3: 34 held-out rows
  Fold 4: 33 held-out rows
  Fold 5: 33 held-out rows

Repeat 4/5 — seed 2026
  Fold 1: 34 held-out rows
  Fold 2: 34 held-out rows
  Fold 3: 34 held-out rows
  Fold 4: 33 held-out rows
  Fold 5: 33 held-out rows

Repeat 5/5 — seed 9917
  Fold 1: 34 held-out rows
  Fold 2: 34 held-out rows
  Fold 3: 34 held-out rows
  Fold 4: 33 held-out rows
  Fold 5: 33 held-out rows

Repeated-CV results:


,seed,all_168_meteor,true_passage_meteor,article_selection_rate,paragraph_selection_rate
0,17,0.396999,0.412532,0.946429,0.053571
1,83,0.378508,0.404575,0.958333,0.041667
2,641,0.403512,0.422685,0.952381,0.047619
3,2026,0.395587,0.426739,0.928571,0.071429
4,9917,0.366905,0.387746,0.940476,0.059524



Repeated-CV score summary:
       all_168_meteor  true_passage_meteor  article_selection_rate
count        5.000000             5.000000                5.000000
mean         0.388302             0.410855                0.945238
std          0.015115             0.015561                0.011450
min          0.366905             0.387746                0.928571
25%          0.378508             0.404575                0.940476
50%          0.395587             0.412532                0.946429
75%          0.396999             0.422685                0.952381
max          0.403512             0.426739                0.958333

Consensus held-out selection:
All 168 METEOR: 0.389656
True-passage METEOR: 0.401684
Article selection rate: 0.964286
Mean candidate-selection agreement: 0.57619
Rows with identical selection across all repeats: 0.113095

Historical passage-route score: 0.38726
Consensus passage-subset gain: 0.002396
Estimated full-validation gain: 0.001006
Estimated full-validation

In [19]:
import os
import re
import numpy as np
import pandas as pd


BOUNDARY_RESULTS_PATH = os.path.join(
    WORK_DIR,
    "val_passage_boundary_expansion_results.csv"
)

BOUNDARY_SUMMARY_PATH = os.path.join(
    WORK_DIR,
    "val_passage_boundary_expansion_summary.csv"
)

HISTORICAL_PASSAGE_ROUTE_SCORE = 0.387260
BOOTSTRAP_ITERATIONS = 2000
TOP_ARTICLE_RANKS = 3

# 1. Article-context lookup

article_context_lookup = (
    passage_article_df
    .set_index("row_number")[
        "article_context"
    ]
    .astype(str)
    .to_dict()
)

# 2. Locate sections and sentence boundaries

def get_article_sections(article_context):
    sections = []

    for match in re.finditer(
        r"(?:^|\n\n)(.*?)(?=\n\n|$)",
        article_context,
        flags=re.DOTALL
    ):
        raw_section = match.group(1)

        if not raw_section.strip():
            continue

        section_start = match.start(1)
        section_end = match.end(1)

        marker_match = re.match(
            r"^\s*(?:Title:|Paragraph\s+\d+:)\s*",
            raw_section,
            flags=re.IGNORECASE
        )

        marker_length = (
            marker_match.end()
            if marker_match
            else 0
        )

        content_start = (
            section_start
            + marker_length
        )

        content_text = article_context[
            content_start:section_end
        ].strip()

        if not content_text:
            continue

        leading_whitespace = len(
            article_context[
                content_start:section_end
            ]
        ) - len(
            article_context[
                content_start:section_end
            ].lstrip()
        )

        content_start += leading_whitespace

        sections.append({
            "section_start": section_start,
            "section_end": section_end,
            "content_start": content_start,
            "content_end": section_end,
            "content_text": article_context[
                content_start:section_end
            ].strip()
        })

    return sections


def get_sentence_spans(text):
    sentence_spans = []

    sentence_pattern = re.compile(
        r"""
        \s*
        (
            .*?
            (?:
                [.!?]+
                ["'”’)]*
                (?=\s+|$)
                |
                $
            )
        )
        """,
        flags=re.DOTALL | re.VERBOSE
    )

    for match in sentence_pattern.finditer(
        text
    ):
        sentence_text = clean_text(
            match.group(1)
        )

        if not sentence_text:
            continue

        raw_start = match.start(1)
        raw_end = match.end(1)

        sentence_spans.append({
            "start": raw_start,
            "end": raw_end,
            "text": sentence_text
        })

    if not sentence_spans and clean_text(text):
        sentence_spans.append({
            "start": 0,
            "end": len(text),
            "text": clean_text(text)
        })

    return sentence_spans


def join_sentence_range(
    sentence_spans,
    start_index,
    end_index
):
    start_index = max(
        0,
        start_index
    )

    end_index = min(
        len(sentence_spans) - 1,
        end_index
    )

    if start_index > end_index:
        return ""

    return clean_text(
        " ".join(
            sentence_spans[index]["text"]
            for index in range(
                start_index,
                end_index + 1
            )
        )
    )

# 3. Expand one article-model span

def create_boundary_variants(
    article_context,
    character_start,
    character_end,
    raw_prediction
):
    sections = get_article_sections(
        article_context
    )

    span_midpoint = (
        character_start
        + character_end
    ) / 2

    selected_section = None

    for section in sections:
        if (
            section["section_start"]
            <= span_midpoint
            <= section["section_end"]
        ):
            selected_section = section
            break

    if selected_section is None:
        selected_section = min(
            sections,
            key=lambda section: abs(
                section["section_start"]
                - span_midpoint
            )
        )

    section_text = selected_section[
        "content_text"
    ]

    local_start = max(
        0,
        int(
            character_start
            - selected_section[
                "content_start"
            ]
        )
    )

    local_end = max(
        local_start,
        int(
            character_end
            - selected_section[
                "content_start"
            ]
        )
    )

    sentence_spans = get_sentence_spans(
        section_text
    )

    overlapping_indices = []

    for sentence_index, sentence in enumerate(
        sentence_spans
    ):
        overlaps = (
            sentence["end"] > local_start
            and sentence["start"] < local_end
        )

        if overlaps:
            overlapping_indices.append(
                sentence_index
            )

    if overlapping_indices:
        first_sentence_index = min(
            overlapping_indices
        )

        last_sentence_index = max(
            overlapping_indices
        )

    else:
        first_sentence_index = min(
            range(len(sentence_spans)),
            key=lambda index: abs(
                sentence_spans[index][
                    "start"
                ]
                - local_start
            )
        )

        last_sentence_index = (
            first_sentence_index
        )

    variants = {
        "raw_span": clean_text(
            raw_prediction
        ),

        "containing_sentence":
            join_sentence_range(
                sentence_spans,
                first_sentence_index,
                last_sentence_index
            ),

        "previous_and_current":
            join_sentence_range(
                sentence_spans,
                first_sentence_index - 1,
                last_sentence_index
            ),

        "current_and_next":
            join_sentence_range(
                sentence_spans,
                first_sentence_index,
                last_sentence_index + 1
            ),

        "previous_current_next":
            join_sentence_range(
                sentence_spans,
                first_sentence_index - 1,
                last_sentence_index + 1
            ),

        "current_and_next_two":
            join_sentence_range(
                sentence_spans,
                first_sentence_index,
                last_sentence_index + 2
            ),

        "previous_two_and_current":
            join_sentence_range(
                sentence_spans,
                first_sentence_index - 2,
                last_sentence_index
            )
    }

    return {
        variant_name: variant_text
        for variant_name, variant_text
        in variants.items()
        if clean_text(variant_text)
    }

# 4. Generate boundary-expanded predictions

top_article_candidates_df = (
    article_span_candidates_df[
        article_span_candidates_df[
            "candidate_rank"
        ].le(TOP_ARTICLE_RANKS)
    ]
    .copy()
    .sort_values(
        [
            "row_number",
            "candidate_rank"
        ]
    )
)

boundary_records = []

for candidate in (
    top_article_candidates_df.itertuples(
        index=False
    )
):
    row_number = int(
        candidate.row_number
    )

    article_context = (
        article_context_lookup[
            row_number
        ]
    )

    variants = create_boundary_variants(
        article_context=article_context,
        character_start=int(
            candidate.character_start
        ),
        character_end=int(
            candidate.character_end
        ),
        raw_prediction=(
            candidate.predicted_text
        )
    )

    for variant_name, prediction in (
        variants.items()
    ):
        rule_name = (
            f"article_rank"
            f"{int(candidate.candidate_rank)}"
            f"_{variant_name}"
        )

        gold_text = gold_lookup[
            row_number
        ]["gold_text"]

        boundary_records.append({
            "row_number": row_number,
            "gold_type": gold_lookup[
                row_number
            ]["gold_type"],
            "candidate_rank": int(
                candidate.candidate_rank
            ),
            "variant": variant_name,
            "rule": rule_name,
            "prediction": prediction,
            "word_count": len(
                prediction.split()
            ),
            "meteor": calculate_meteor(
                gold_text,
                prediction
            )
        })


boundary_results_df = pd.DataFrame(
    boundary_records
)

# 5. Ensure every fixed rule covers all 168 rows

rule_coverage = (
    boundary_results_df
    .groupby("rule")[
        "row_number"
    ]
    .nunique()
)

complete_rules = rule_coverage[
    rule_coverage.eq(
        len(passage_type_df)
    )
].index

boundary_results_df = (
    boundary_results_df[
        boundary_results_df[
            "rule"
        ].isin(complete_rules)
    ]
    .copy()
)

# 6. Bootstrap each fixed rule

random_generator = (
    np.random.default_rng(641)
)

summary_records = []

for rule_name, rule_df in (
    boundary_results_df.groupby("rule")
):
    rule_df = (
        rule_df
        .sort_values("row_number")
        .reset_index(drop=True)
    )

    all_scores = rule_df[
        "meteor"
    ].to_numpy()

    true_passage_scores = rule_df[
        rule_df[
            "gold_type"
        ].eq("passage")
    ]["meteor"].to_numpy()

    bootstrap_indices = (
        random_generator.integers(
            0,
            len(all_scores),
            size=(
                BOOTSTRAP_ITERATIONS,
                len(all_scores)
            )
        )
    )

    bootstrap_means = (
        all_scores[
            bootstrap_indices
        ].mean(axis=1)
    )

    mean_score = float(
        all_scores.mean()
    )

    summary_records.append({
        "rule": rule_name,
        "all_168_meteor": mean_score,
        "true_passage_meteor": float(
            true_passage_scores.mean()
        ),
        "gain_vs_historical_route": (
            mean_score
            - HISTORICAL_PASSAGE_ROUTE_SCORE
        ),
        "mean_word_count": float(
            rule_df["word_count"].mean()
        ),
        "median_word_count": float(
            rule_df["word_count"].median()
        ),
        "bootstrap_p05": float(
            np.quantile(
                bootstrap_means,
                0.05
            )
        ),
        "bootstrap_p50": float(
            np.quantile(
                bootstrap_means,
                0.50
            )
        ),
        "bootstrap_p95": float(
            np.quantile(
                bootstrap_means,
                0.95
            )
        ),
        "bootstrap_probability_above_historical": float(
            np.mean(
                bootstrap_means
                >
                HISTORICAL_PASSAGE_ROUTE_SCORE
            )
        )
    })


boundary_summary_df = (
    pd.DataFrame(
        summary_records
    )
    .sort_values(
        "all_168_meteor",
        ascending=False
    )
    .reset_index(drop=True)
)

# 7. Save and report

boundary_results_df.to_csv(
    BOUNDARY_RESULTS_PATH,
    index=False
)

boundary_summary_df.to_csv(
    BOUNDARY_SUMMARY_PATH,
    index=False
)

print(
    "Complete fixed rules:",
    len(boundary_summary_df)
)

print(
    "\nHistorical passage-route score:",
    HISTORICAL_PASSAGE_ROUTE_SCORE
)

print(
    "\nTop boundary-expansion rules:"
)

display(
    boundary_summary_df.head(15)
)

raw_rank1_score = (
    boundary_summary_df.loc[
        boundary_summary_df[
            "rule"
        ].eq(
            "article_rank1_raw_span"
        ),
        "all_168_meteor"
    ]
)

if len(raw_rank1_score):
    print(
        "\nRank-1 raw-span sanity check:",
        round(
            float(
                raw_rank1_score.iloc[0]
            ),
            6
        )
    )

best_rule = boundary_summary_df.iloc[0]

print("\nBest rule:")
print(best_rule["rule"])

print(
    "Best all-168 METEOR:",
    round(
        best_rule[
            "all_168_meteor"
        ],
        6
    )
)

print(
    "Gain versus historical passage route:",
    round(
        best_rule[
            "gain_vs_historical_route"
        ],
        6
    )
)

print(
    "Estimated full-validation gain:",
    round(
        best_rule[
            "gain_vs_historical_route"
        ]
        * len(passage_type_df)
        / len(val_df),
        6
    )
)

print(
    "Bootstrap probability above historical:",
    round(
        best_rule[
            "bootstrap_probability_above_historical"
        ],
        6
    )
)

print(
    "\nSaved summary:",
    BOUNDARY_SUMMARY_PATH
)

Complete fixed rules: 21

Historical passage-route score: 0.38726

Top boundary-expansion rules:


,rule,all_168_meteor,true_passage_meteor,gain_vs_historical_route,mean_word_count,median_word_count,bootstrap_p05,bootstrap_p50,bootstrap_p95,bootstrap_probability_above_historical
0,article_rank1_containing_sentence,0.412168,0.421606,0.024908,23.565476,23.0,0.366475,0.411512,0.456187,0.8085
1,article_rank1_previous_and_current,0.409873,0.430652,0.022613,28.821429,27.0,0.366134,0.411164,0.454955,0.8035
2,article_rank1_current_and_next,0.408179,0.424597,0.020919,32.113095,30.0,0.365418,0.408216,0.451045,0.7795
3,article_rank1_previous_current_next,0.407742,0.434030,0.020482,37.369048,34.0,0.363384,0.407379,0.446555,0.7740
4,article_rank1_current_and_next_two,0.407273,0.425963,0.020013,34.964286,31.0,0.366521,0.406170,0.450703,0.7845
5,article_rank1_previous_two_and_current,0.404679,0.425115,0.017419,30.690476,28.5,0.363658,0.404173,0.448082,0.7495
6,article_rank2_previous_current_next,0.380987,0.415522,-0.006273,39.470238,36.0,0.340137,0.381310,0.423661,0.3930
7,article_rank2_current_and_next,0.380085,0.406338,-0.007175,35.125000,31.0,0.338854,0.380367,0.422731,0.3955
8,article_rank3_current_and_next_two,0.378380,0.408021,-0.008880,37.434524,33.0,0.334857,0.377832,0.421182,0.3595
9,article_rank2_current_and_next_two,0.378005,0.405655,-0.009255,37.684524,33.0,0.337738,0.378423,0.418263,0.3495



Rank-1 raw-span sanity check: 0.359408

Best rule:
article_rank1_containing_sentence
Best all-168 METEOR: 0.412168
Gain versus historical passage route: 0.024908
Estimated full-validation gain: 0.010461
Bootstrap probability above historical: 0.8085

Saved summary: /kaggle/working/task2_passage_v1/val_passage_boundary_expansion_summary.csv


In [20]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold


RULE_SELECTION_CV_PATH = os.path.join(
    WORK_DIR,
    "val_passage_boundary_rule_selection_cv.csv"
)

RULE_SELECTION_FREQUENCY_PATH = os.path.join(
    WORK_DIR,
    "val_passage_boundary_rule_selection_frequency.csv"
)

HISTORICAL_PASSAGE_ROUTE_SCORE = 0.387260

SPLIT_SEEDS = [
    17,
    83,
    641,
    2026,
    9917,
    44,
    112,
    333,
    777,
    1234
]


# 1. Create one row × rule score matrix

rule_score_matrix = (
    boundary_results_df
    .pivot(
        index="row_number",
        columns="rule",
        values="meteor"
    )
    .sort_index()
)

row_metadata_df = (
    boundary_results_df[
        [
            "row_number",
            "gold_type"
        ]
    ]
    .drop_duplicates()
    .set_index("row_number")
    .loc[rule_score_matrix.index]
)

if rule_score_matrix.isna().any().any():
    raise RuntimeError(
        "Some boundary rules do not cover every row."
    )

print("Rows:", len(rule_score_matrix))
print("Rules:", len(rule_score_matrix.columns))

# 2. Repeated row-held-out rule selection

seed_summary_records = []
fold_selection_records = []
all_oof_scores = []

for split_seed in SPLIT_SEEDS:
    splitter = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=split_seed
    )

    oof_scores = pd.Series(
        np.nan,
        index=rule_score_matrix.index,
        dtype=float
    )

    print(f"\nSeed {split_seed}")

    for fold_number, (
        training_positions,
        validation_positions
    ) in enumerate(
        splitter.split(
            rule_score_matrix,
            row_metadata_df["gold_type"]
        ),
        start=1
    ):
        training_rows = (
            rule_score_matrix.index[
                training_positions
            ]
        )

        validation_rows = (
            rule_score_matrix.index[
                validation_positions
            ]
        )

        training_rule_means = (
            rule_score_matrix.loc[
                training_rows
            ].mean(axis=0)
        )

        selected_rule = (
            training_rule_means.idxmax()
        )

        heldout_rule_scores = (
            rule_score_matrix.loc[
                validation_rows,
                selected_rule
            ]
        )

        oof_scores.loc[
            validation_rows
        ] = heldout_rule_scores

        fold_selection_records.append({
            "seed": split_seed,
            "fold": fold_number,
            "selected_rule": selected_rule,
            "training_score": float(
                training_rule_means[
                    selected_rule
                ]
            ),
            "heldout_score": float(
                heldout_rule_scores.mean()
            ),
            "heldout_rows": len(
                validation_rows
            )
        })

        print(
            f"  Fold {fold_number}: "
            f"{selected_rule} | "
            f"held-out={heldout_rule_scores.mean():.6f}"
        )

    if oof_scores.isna().any():
        raise RuntimeError(
            f"Incomplete OOF scores for seed {split_seed}."
        )

    true_passage_rows = (
        row_metadata_df[
            row_metadata_df[
                "gold_type"
            ].eq("passage")
        ].index
    )

    seed_score = float(
        oof_scores.mean()
    )

    true_passage_score = float(
        oof_scores.loc[
            true_passage_rows
        ].mean()
    )

    seed_summary_records.append({
        "seed": split_seed,
        "heldout_all_168_meteor": (
            seed_score
        ),
        "heldout_true_passage_meteor": (
            true_passage_score
        ),
        "gain_vs_historical_route": (
            seed_score
            - HISTORICAL_PASSAGE_ROUTE_SCORE
        ),
        "estimated_full_validation_gain": (
            (
                seed_score
                - HISTORICAL_PASSAGE_ROUTE_SCORE
            )
            * len(passage_type_df)
            / len(val_df)
        )
    })

    all_oof_scores.append(
        oof_scores.to_numpy()
    )


# 3. Aggregate repeated held-out results

rule_selection_cv_df = pd.DataFrame(
    seed_summary_records
)

fold_selection_df = pd.DataFrame(
    fold_selection_records
)

rule_selection_frequency_df = (
    fold_selection_df[
        "selected_rule"
    ]
    .value_counts()
    .rename_axis("rule")
    .reset_index(name="selected_folds")
)

rule_selection_frequency_df[
    "selection_rate"
] = (
    rule_selection_frequency_df[
        "selected_folds"
    ]
    / len(fold_selection_df)
)

mean_heldout_score = float(
    rule_selection_cv_df[
        "heldout_all_168_meteor"
    ].mean()
)

minimum_heldout_score = float(
    rule_selection_cv_df[
        "heldout_all_168_meteor"
    ].min()
)

mean_gain = (
    mean_heldout_score
    - HISTORICAL_PASSAGE_ROUTE_SCORE
)

estimated_full_gain = (
    mean_gain
    * len(passage_type_df)
    / len(val_df)
)

# 4. Save and report

rule_selection_cv_df.to_csv(
    RULE_SELECTION_CV_PATH,
    index=False
)

rule_selection_frequency_df.to_csv(
    RULE_SELECTION_FREQUENCY_PATH,
    index=False
)

print("\nRepeated held-out results:")

display(
    rule_selection_cv_df
)

print("\nRule-selection frequency:")

display(
    rule_selection_frequency_df
)

print("\nSummary:")
print(
    "Mean held-out passage-route score:",
    round(mean_heldout_score, 6)
)

print(
    "Minimum seed score:",
    round(minimum_heldout_score, 6)
)

print(
    "Mean gain versus historical route:",
    round(mean_gain, 6)
)

print(
    "Estimated full-validation gain:",
    round(estimated_full_gain, 6)
)

print(
    "Containing-sentence full-data score:",
    round(
        float(
            boundary_summary_df.loc[
                boundary_summary_df[
                    "rule"
                ].eq(
                    "article_rank1_containing_sentence"
                ),
                "all_168_meteor"
            ].iloc[0]
        ),
        6
    )
)

print(
    "\nCheckpoint complete:",
    os.path.exists(
        RULE_SELECTION_CV_PATH
    )
    and os.path.exists(
        RULE_SELECTION_FREQUENCY_PATH
    )
)

Rows: 168
Rules: 21

Seed 17
  Fold 1: article_rank1_containing_sentence | held-out=0.419815
  Fold 2: article_rank1_containing_sentence | held-out=0.341880
  Fold 3: article_rank1_containing_sentence | held-out=0.428551
  Fold 4: article_rank1_current_and_next | held-out=0.396899
  Fold 5: article_rank1_previous_current_next | held-out=0.417867

Seed 83
  Fold 1: article_rank1_containing_sentence | held-out=0.355556
  Fold 2: article_rank1_containing_sentence | held-out=0.399585
  Fold 3: article_rank1_previous_current_next | held-out=0.346951
  Fold 4: article_rank1_current_and_next_two | held-out=0.427670
  Fold 5: article_rank1_previous_current_next | held-out=0.418415

Seed 641
  Fold 1: article_rank1_previous_and_current | held-out=0.491750
  Fold 2: article_rank1_previous_current_next | held-out=0.399868
  Fold 3: article_rank1_containing_sentence | held-out=0.451706
  Fold 4: article_rank1_containing_sentence | held-out=0.311642
  Fold 5: article_rank1_containing_sentence | hel

,seed,heldout_all_168_meteor,heldout_true_passage_meteor,gain_vs_historical_route,estimated_full_validation_gain
0,17,0.400927,0.412903,0.013667,0.005740
1,83,0.389238,0.408461,0.001978,0.000831
2,641,0.400947,0.415222,0.013687,0.005749
3,2026,0.394804,0.413010,0.007544,0.003169
4,9917,0.405699,0.418255,0.018439,0.007744
5,44,0.396473,0.411660,0.009213,0.003869
6,112,0.402650,0.418477,0.015390,0.006464
7,333,0.393429,0.406665,0.006169,0.002591
8,777,0.391024,0.404552,0.003764,0.001581
9,1234,0.401697,0.414679,0.014437,0.006064



Rule-selection frequency:


,rule,selected_folds,selection_rate
0,article_rank1_containing_sentence,29,0.58
1,article_rank1_previous_current_next,11,0.22
2,article_rank1_previous_and_current,5,0.10
3,article_rank1_current_and_next_two,3,0.06
4,article_rank1_current_and_next,2,0.04



Summary:
Mean held-out passage-route score: 0.397689
Minimum seed score: 0.389238
Mean gain versus historical route: 0.010429
Estimated full-validation gain: 0.00438
Containing-sentence full-data score: 0.412168

Checkpoint complete: True


In [21]:
import os
import gc
import time
import shutil
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

TEST_ARTICLE_MAX_LENGTH = 384
TEST_ARTICLE_STRIDE = 128
TEST_ARTICLE_BATCH_SIZE = 16

TEST_ARTICLE_ROWS_PATH = os.path.join(
    WORK_DIR,
    "test_passage_article_contexts.pkl"
)

TEST_ARTICLE_TOKENIZED_DIR = os.path.join(
    WORK_DIR,
    "test_passage_article_tokenized"
)

TEST_ARTICLE_LOGITS_PATH = os.path.join(
    WORK_DIR,
    "test_passage_article_logits.npz"
)


# 1. Identify the test rows routed as passage

test_passage_type_df = (
    test_type_predictions_df[
        test_type_predictions_df[
            "predicted_type"
        ].astype(str).eq("passage")
    ]
    .copy()
    .sort_values("row_number")
    .reset_index(drop=True)
)

test_passage_row_numbers = (
    test_passage_type_df[
        "row_number"
    ]
    .astype(int)
    .tolist()
)

print(
    "Predicted-passage test rows:",
    len(test_passage_row_numbers)
)

if len(test_passage_row_numbers) != 183:
    raise RuntimeError(
        "Expected 183 predicted-passage test rows, "
        f"but found {len(test_passage_row_numbers)}."
    )


# 2. Recreate the exact article-model context format

def build_test_combined_article_context(row):
    sections = []

    title = str(
        row.get("targetTitle", "")
        or ""
    ).strip()

    if title:
        sections.append(
            f"Title: {title}"
        )

    paragraphs = row.get(
        "targetParagraphs",
        []
    )

    if not isinstance(paragraphs, list):
        paragraphs = []

    for paragraph_index, paragraph in enumerate(
        paragraphs,
        start=1
    ):
        paragraph = str(
            paragraph
        ).strip()

        if paragraph:
            sections.append(
                f"Paragraph {paragraph_index}: "
                f"{paragraph}"
            )

    return "\n\n".join(sections)


test_article_records = []

for row_number in test_passage_row_numbers:
    row = test_df.iloc[row_number]

    article_context = (
        build_test_combined_article_context(
            row
        )
    )

    test_article_records.append({
        "row_number": int(row_number),
        "article_id": row["id"],
        "question": clean_text(
            row.get("postText", "")
        ),
        "article_context": article_context,
        "article_word_count": len(
            article_context.split()
        )
    })


test_passage_article_df = pd.DataFrame(
    test_article_records
)

test_passage_article_df.to_pickle(
    TEST_ARTICLE_ROWS_PATH
)

print(
    "\nTest article word-count summary:"
)

print(
    test_passage_article_df[
        "article_word_count"
    ].describe()
)

print(
    "\nTest article contexts saved:"
)

print(TEST_ARTICLE_ROWS_PATH)

# 3. Load the saved article QA model

print("\nLoading article QA model...")

test_article_tokenizer = (
    AutoTokenizer.from_pretrained(
        ARTICLE_MODEL_DIR,
        local_files_only=True,
        use_fast=True
    )
)

test_article_model = (
    AutoModelForQuestionAnswering
    .from_pretrained(
        ARTICLE_MODEL_DIR,
        local_files_only=True
    )
    .to(DEVICE)
)

test_article_model.eval()

print("Device:", DEVICE)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

# 4. Tokenize test articles into sliding windows

test_article_raw_dataset = Dataset.from_pandas(
    test_passage_article_df[
        [
            "row_number",
            "question",
            "article_context"
        ]
    ],
    preserve_index=False
)


def tokenize_test_article_examples(examples):
    questions = [
        str(question).strip()
        for question in examples["question"]
    ]

    contexts = [
        str(context)
        for context in examples[
            "article_context"
        ]
    ]

    tokenized = test_article_tokenizer(
        questions,
        contexts,
        truncation="only_second",
        max_length=TEST_ARTICLE_MAX_LENGTH,
        stride=TEST_ARTICLE_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    sample_mapping = tokenized.pop(
        "overflow_to_sample_mapping"
    )

    feature_row_numbers = []

    for feature_index, sample_index in enumerate(
        sample_mapping
    ):
        row_number = int(
            examples["row_number"][
                sample_index
            ]
        )

        feature_row_numbers.append(
            row_number
        )

        sequence_ids = (
            tokenized.sequence_ids(
                feature_index
            )
        )

        tokenized[
            "offset_mapping"
        ][feature_index] = [
            offset
            if sequence_ids[token_index] == 1
            else None
            for token_index, offset
            in enumerate(
                tokenized[
                    "offset_mapping"
                ][feature_index]
            )
        ]

    tokenized[
        "row_number"
    ] = feature_row_numbers

    return tokenized


tokenization_start = time.time()

test_passage_article_tokenized = (
    test_article_raw_dataset.map(
        tokenize_test_article_examples,
        batched=True,
        remove_columns=(
            test_article_raw_dataset.column_names
        ),
        desc=(
            "Tokenizing test passage articles"
        )
    )
)

print(
    "\nTokenization seconds:",
    round(
        time.time()
        - tokenization_start,
        2
    )
)

print(
    "Sliding-window features:",
    len(test_passage_article_tokenized)
)

print(
    "Average windows per article:",
    round(
        len(test_passage_article_tokenized)
        / len(test_passage_article_df),
        2
    )
)

# 5. Save tokenized features before inference


if os.path.exists(
    TEST_ARTICLE_TOKENIZED_DIR
):
    shutil.rmtree(
        TEST_ARTICLE_TOKENIZED_DIR
    )

test_passage_article_tokenized.save_to_disk(
    TEST_ARTICLE_TOKENIZED_DIR
)

print(
    "\nTokenized test features saved:"
)

print(TEST_ARTICLE_TOKENIZED_DIR)


# 6. Run test article-QA inference

model_input_names = [
    input_name
    for input_name
    in test_article_tokenizer.model_input_names
    if input_name
    in test_passage_article_tokenized.column_names
]

test_start_batches = []
test_end_batches = []

inference_start = time.time()

for batch_start in range(
    0,
    len(test_passage_article_tokenized),
    TEST_ARTICLE_BATCH_SIZE
):
    batch_end = min(
        batch_start + TEST_ARTICLE_BATCH_SIZE,
        len(test_passage_article_tokenized)
    )

    batch = test_passage_article_tokenized[
        batch_start:batch_end
    ]

    model_inputs = {
        input_name: torch.tensor(
            batch[input_name],
            dtype=torch.long,
            device=DEVICE
        )
        for input_name in model_input_names
    }

    with torch.inference_mode():
        outputs = test_article_model(
            **model_inputs
        )

    test_start_batches.append(
        outputs.start_logits
        .detach()
        .cpu()
        .numpy()
    )

    test_end_batches.append(
        outputs.end_logits
        .detach()
        .cpu()
        .numpy()
    )

    if (
        batch_end % 160 == 0
        or batch_end
        == len(test_passage_article_tokenized)
    ):
        elapsed = (
            time.time()
            - inference_start
        )

        print(
            f"Processed {batch_end}/"
            f"{len(test_passage_article_tokenized)} "
            f"features in {elapsed:.1f} seconds"
        )


test_article_start_logits = np.concatenate(
    test_start_batches,
    axis=0
)

test_article_end_logits = np.concatenate(
    test_end_batches,
    axis=0
)

np.savez_compressed(
    TEST_ARTICLE_LOGITS_PATH,
    start_logits=test_article_start_logits,
    end_logits=test_article_end_logits
)

print(
    "\nInference seconds:",
    round(
        time.time()
        - inference_start,
        2
    )
)

print(
    "Start-logit shape:",
    test_article_start_logits.shape
)

print(
    "End-logit shape:",
    test_article_end_logits.shape
)

print(
    "Test article logits saved:",
    TEST_ARTICLE_LOGITS_PATH
)

print(
    "Logits file size:",
    round(
        os.path.getsize(
            TEST_ARTICLE_LOGITS_PATH
        ) / 1024**2,
        2
    ),
    "MB"
)

# 7. Release the model from GPU

for variable_name in [
    "test_article_model",
    "model_inputs",
    "outputs",
    "test_start_batches",
    "test_end_batches"
]:
    globals().pop(
        variable_name,
        None
    )

gc.collect()
torch.cuda.empty_cache()

print(
    "\nGPU memory after cleanup:",
    round(
        torch.cuda.memory_allocated()
        / 1024**3,
        3
    ),
    "GB"
)

print(
    "\nTest article-QA checkpoint complete:",
    os.path.exists(
        TEST_ARTICLE_ROWS_PATH
    )
    and os.path.exists(
        TEST_ARTICLE_TOKENIZED_DIR
    )
    and os.path.exists(
        TEST_ARTICLE_LOGITS_PATH
    )
)

Predicted-passage test rows: 183

Test article word-count summary:
count     183.000000
mean      580.672131
std       653.446713
min        36.000000
25%       234.000000
50%       384.000000
75%       709.500000
max      5537.000000
Name: article_word_count, dtype: float64

Test article contexts saved:
/kaggle/working/task2_passage_v1/test_passage_article_contexts.pkl

Loading article QA model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Device: cuda
GPU: Tesla T4


Tokenizing test passage articles:   0%|          | 0/183 [00:00<?, ? examples/s]


Tokenization seconds: 1.12
Sliding-window features: 602
Average windows per article: 3.29


Saving the dataset (0/1 shards):   0%|          | 0/602 [00:00<?, ? examples/s]


Tokenized test features saved:
/kaggle/working/task2_passage_v1/test_passage_article_tokenized
Processed 160/602 features in 4.3 seconds
Processed 320/602 features in 8.7 seconds
Processed 480/602 features in 14.4 seconds
Processed 602/602 features in 18.3 seconds

Inference seconds: 18.4
Start-logit shape: (602, 384)
End-logit shape: (602, 384)
Test article logits saved: /kaggle/working/task2_passage_v1/test_passage_article_logits.npz
Logits file size: 1.4 MB

GPU memory after cleanup: 0.009 GB

Test article-QA checkpoint complete: True


In [22]:
import os
import re
import numpy as np
import pandas as pd

from datasets import load_from_disk
from transformers import AutoTokenizer


TEST_MAX_ANSWER_LENGTH = 100

TEST_ARTICLE_ROWS_PATH = os.path.join(
    WORK_DIR,
    "test_passage_article_contexts.pkl"
)

TEST_ARTICLE_TOKENIZED_DIR = os.path.join(
    WORK_DIR,
    "test_passage_article_tokenized"
)

TEST_ARTICLE_LOGITS_PATH = os.path.join(
    WORK_DIR,
    "test_passage_article_logits.npz"
)

NEW_SUBMISSION_PATH = os.path.join(
    WORK_DIR,
    "submission_passage_containing_sentence_v1.csv"
)

NEW_AUDIT_PATH = os.path.join(
    WORK_DIR,
    "submission_passage_containing_sentence_v1_audit.csv"
)

# 1. Reload saved test inference objects

if "test_passage_article_df" not in globals():
    test_passage_article_df = pd.read_pickle(
        TEST_ARTICLE_ROWS_PATH
    )

if "test_passage_article_tokenized" not in globals():
    test_passage_article_tokenized = load_from_disk(
        TEST_ARTICLE_TOKENIZED_DIR
    )

if "test_article_tokenizer" not in globals():
    test_article_tokenizer = (
        AutoTokenizer.from_pretrained(
            ARTICLE_MODEL_DIR,
            local_files_only=True,
            use_fast=True
        )
    )

if (
    "test_article_start_logits" not in globals()
    or "test_article_end_logits" not in globals()
):
    saved_test_logits = np.load(
        TEST_ARTICLE_LOGITS_PATH
    )

    test_article_start_logits = (
        saved_test_logits["start_logits"]
    )

    test_article_end_logits = (
        saved_test_logits["end_logits"]
    )


test_article_context_lookup = (
    test_passage_article_df
    .set_index("row_number")[
        "article_context"
    ]
    .astype(str)
    .to_dict()
)

expected_test_passage_rows = set(
    test_type_predictions_df.loc[
        test_type_predictions_df[
            "predicted_type"
        ].astype(str).eq("passage"),
        "row_number"
    ].astype(int)
)

print(
    "Expected passage rows:",
    len(expected_test_passage_rows)
)

print(
    "Tokenized features:",
    len(test_passage_article_tokenized)
)

print(
    "Start-logit shape:",
    test_article_start_logits.shape
)

print(
    "End-logit shape:",
    test_article_end_logits.shape
)

# 2. Text and sentence helpers

def clean_output_text(text):
    text = str(text)

    text = re.sub(
        r"^\s*Title:\s*",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"^\s*Paragraph\s+\d+:\s*",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\s+Paragraph\s+\d+:\s*",
        " ",
        text,
        flags=re.IGNORECASE
    )

    return re.sub(
        r"\s+",
        " ",
        text
    ).strip()


def get_article_sections(article_context):
    sections = []

    for match in re.finditer(
        r"(?:^|\n\n)(.*?)(?=\n\n|$)",
        article_context,
        flags=re.DOTALL
    ):
        raw_section = match.group(1)

        if not raw_section.strip():
            continue

        section_start = match.start(1)
        section_end = match.end(1)

        marker_match = re.match(
            r"^\s*(?:Title:|Paragraph\s+\d+:)\s*",
            raw_section,
            flags=re.IGNORECASE
        )

        marker_length = (
            marker_match.end()
            if marker_match
            else 0
        )

        content_start = (
            section_start
            + marker_length
        )

        raw_content = article_context[
            content_start:section_end
        ]

        leading_whitespace = (
            len(raw_content)
            - len(raw_content.lstrip())
        )

        trailing_whitespace = (
            len(raw_content)
            - len(raw_content.rstrip())
        )

        content_start += leading_whitespace
        content_end = (
            section_end
            - trailing_whitespace
        )

        if content_end <= content_start:
            continue

        sections.append({
            "section_start": section_start,
            "section_end": section_end,
            "content_start": content_start,
            "content_end": content_end,
            "content_text": article_context[
                content_start:content_end
            ]
        })

    return sections


def get_sentence_spans(text):
    sentence_spans = []

    sentence_pattern = re.compile(
        r"""
        \s*
        (
            .*?
            (?:
                [.!?]+
                ["'”’)]*
                (?=\s+|$)
                |
                $
            )
        )
        """,
        flags=re.DOTALL | re.VERBOSE
    )

    for match in sentence_pattern.finditer(text):
        sentence_text = clean_output_text(
            match.group(1)
        )

        if not sentence_text:
            continue

        sentence_spans.append({
            "start": match.start(1),
            "end": match.end(1),
            "text": sentence_text
        })

    if not sentence_spans and clean_output_text(text):
        sentence_spans.append({
            "start": 0,
            "end": len(text),
            "text": clean_output_text(text)
        })

    return sentence_spans


def expand_to_containing_sentence(
    article_context,
    character_start,
    character_end,
    raw_prediction
):
    sections = get_article_sections(
        article_context
    )

    if not sections:
        return clean_output_text(
            raw_prediction
        )

    span_midpoint = (
        character_start
        + character_end
    ) / 2

    selected_section = None

    for section in sections:
        if (
            section["section_start"]
            <= span_midpoint
            <= section["section_end"]
        ):
            selected_section = section
            break

    if selected_section is None:
        selected_section = min(
            sections,
            key=lambda section: abs(
                section["section_start"]
                - span_midpoint
            )
        )

    section_text = selected_section[
        "content_text"
    ]

    local_start = max(
        0,
        character_start
        - selected_section["content_start"]
    )

    local_end = min(
        len(section_text),
        max(
            local_start + 1,
            character_end
            - selected_section["content_start"]
        )
    )

    sentence_spans = get_sentence_spans(
        section_text
    )

    overlapping_sentences = [
        sentence
        for sentence in sentence_spans
        if (
            sentence["end"] > local_start
            and sentence["start"] < local_end
        )
    ]

    if overlapping_sentences:
        expanded_text = clean_output_text(
            " ".join(
                sentence["text"]
                for sentence
                in overlapping_sentences
            )
        )
    else:
        nearest_sentence = min(
            sentence_spans,
            key=lambda sentence: abs(
                sentence["start"]
                - local_start
            )
        )

        expanded_text = clean_output_text(
            nearest_sentence["text"]
        )

    if not expanded_text:
        expanded_text = clean_output_text(
            raw_prediction
        )

    return expanded_text

# 3. Decode the highest-margin article span per test row

best_result_by_row = {}

for feature_index in range(
    len(test_passage_article_tokenized)
):
    feature = test_passage_article_tokenized[
        feature_index
    ]

    row_number = int(
        feature["row_number"]
    )

    offsets = feature["offset_mapping"]
    input_ids = feature["input_ids"]

    start_logits = test_article_start_logits[
        feature_index
    ]

    end_logits = test_article_end_logits[
        feature_index
    ]

    try:
        cls_index = input_ids.index(
            test_article_tokenizer.cls_token_id
        )
    except ValueError:
        cls_index = 0

    null_score = float(
        start_logits[cls_index]
        + end_logits[cls_index]
    )

    valid_mask = np.array([
        offset is not None
        and len(offset) == 2
        and int(offset[1]) > int(offset[0])
        for offset in offsets
    ])

    valid_start_positions = np.flatnonzero(
        valid_mask
    )

    best_span_score = -np.inf
    best_start_position = None
    best_end_position = None

    for start_position in valid_start_positions:
        maximum_end_position = min(
            start_position
            + TEST_MAX_ANSWER_LENGTH
            - 1,
            len(offsets) - 1
        )

        allowed_end_mask = valid_mask[
            start_position:
            maximum_end_position + 1
        ]

        if not allowed_end_mask.any():
            continue

        local_end_logits = end_logits[
            start_position:
            maximum_end_position + 1
        ].copy()

        local_end_logits[
            ~allowed_end_mask
        ] = -np.inf

        relative_end_position = int(
            np.argmax(local_end_logits)
        )

        end_position = (
            start_position
            + relative_end_position
        )

        span_score = float(
            start_logits[start_position]
            + end_logits[end_position]
        )

        if span_score > best_span_score:
            best_span_score = span_score
            best_start_position = int(
                start_position
            )
            best_end_position = int(
                end_position
            )

    if (
        best_start_position is None
        or best_end_position is None
    ):
        continue

    character_start = int(
        offsets[best_start_position][0]
    )

    character_end = int(
        offsets[best_end_position][1]
    )

    article_context = (
        test_article_context_lookup[
            row_number
        ]
    )

    raw_prediction = clean_output_text(
        article_context[
            character_start:
            character_end
        ]
    )

    answerability_margin = float(
        best_span_score
        - null_score
    )

    current_result = {
        "row_number": row_number,
        "feature_index": feature_index,
        "character_start": character_start,
        "character_end": character_end,
        "raw_article_span": raw_prediction,
        "span_score": best_span_score,
        "null_score": null_score,
        "answerability_margin": (
            answerability_margin
        )
    }

    previous_result = best_result_by_row.get(
        row_number
    )

    if (
        previous_result is None
        or answerability_margin
        > previous_result[
            "answerability_margin"
        ]
    ):
        best_result_by_row[
            row_number
        ] = current_result


decoded_test_rows = set(
    best_result_by_row.keys()
)

missing_test_rows = sorted(
    expected_test_passage_rows
    - decoded_test_rows
)

unexpected_test_rows = sorted(
    decoded_test_rows
    - expected_test_passage_rows
)

print(
    "\nDecoded passage rows:",
    len(decoded_test_rows)
)

print(
    "Missing passage rows:",
    len(missing_test_rows)
)

print(
    "Unexpected decoded rows:",
    len(unexpected_test_rows)
)

if missing_test_rows or unexpected_test_rows:
    raise RuntimeError(
        "Decoded test rows do not match the "
        "183 expected passage rows."
    )

# 4. Expand rank-1 spans to containing sentences

new_prediction_records = []

for row_number in sorted(
    expected_test_passage_rows
):
    result = best_result_by_row[
        row_number
    ]

    article_context = (
        test_article_context_lookup[
            row_number
        ]
    )

    containing_sentence = (
        expand_to_containing_sentence(
            article_context=article_context,
            character_start=result[
                "character_start"
            ],
            character_end=result[
                "character_end"
            ],
            raw_prediction=result[
                "raw_article_span"
            ]
        )
    )

    if not containing_sentence:
        raise RuntimeError(
            f"Empty containing-sentence prediction "
            f"for row {row_number}."
        )

    new_prediction_records.append({
        **result,
        "containing_sentence": (
            containing_sentence
        ),
        "raw_word_count": len(
            result[
                "raw_article_span"
            ].split()
        ),
        "sentence_word_count": len(
            containing_sentence.split()
        )
    })


test_sentence_predictions_df = pd.DataFrame(
    new_prediction_records
)


# 5. Create a new submission from the frozen baseline

sample_columns = list(
    sample_df.columns
)

baseline_columns = list(
    baseline_submission_df.columns
)

if set(sample_columns) != set(
    baseline_columns
):
    raise RuntimeError(
        "Baseline submission columns do not match "
        "the sample submission."
    )

id_column = (
    "id"
    if "id" in sample_columns
    else sample_columns[0]
)

prediction_columns = [
    column
    for column in sample_columns
    if column != id_column
]

if len(prediction_columns) != 1:
    raise RuntimeError(
        "Could not identify exactly one prediction column."
    )

prediction_column = prediction_columns[0]

new_submission_df = (
    baseline_submission_df[
        sample_columns
    ]
    .copy()
)

baseline_ids = (
    new_submission_df[id_column]
    .astype(str)
    .tolist()
)

test_ids = (
    test_df["id"]
    .astype(str)
    .tolist()
)

if baseline_ids != test_ids:
    raise RuntimeError(
        "Baseline submission IDs are not aligned "
        "with test row order."
    )

baseline_prediction_lookup = (
    new_submission_df[
        prediction_column
    ]
    .astype(str)
    .to_dict()
)

sentence_prediction_lookup = (
    test_sentence_predictions_df
    .set_index("row_number")[
        "containing_sentence"
    ]
    .to_dict()
)

for row_number, prediction in (
    sentence_prediction_lookup.items()
):
    new_submission_df.at[
        int(row_number),
        prediction_column
    ] = prediction


# 6. Validate that only passage-routed rows changed


non_passage_rows = sorted(
    set(range(len(test_df)))
    - expected_test_passage_rows
)

non_passage_unchanged = bool(
    (
        new_submission_df.loc[
            non_passage_rows,
            prediction_column
        ].astype(str).to_numpy()
        ==
        baseline_submission_df.loc[
            non_passage_rows,
            prediction_column
        ].astype(str).to_numpy()
    ).all()
)

empty_prediction_count = int(
    new_submission_df[
        prediction_column
    ]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

changed_row_mask = (
    new_submission_df[
        prediction_column
    ].astype(str)
    != baseline_submission_df[
        prediction_column
    ].astype(str)
)

changed_row_numbers = set(
    np.flatnonzero(
        changed_row_mask.to_numpy()
    )
)

changes_outside_passage = (
    changed_row_numbers
    - expected_test_passage_rows
)

if len(new_submission_df) != 400:
    raise RuntimeError(
        "New submission does not contain 400 rows."
    )

if list(new_submission_df.columns) != sample_columns:
    raise RuntimeError(
        "New submission column order is incorrect."
    )

if empty_prediction_count != 0:
    raise RuntimeError(
        "New submission contains empty predictions."
    )

if not non_passage_unchanged:
    raise RuntimeError(
        "At least one non-passage row was modified."
    )

if changes_outside_passage:
    raise RuntimeError(
        "Changes were detected outside passage-routed rows."
    )


# 7. Build audit table and save files

audit_df = (
    test_type_predictions_df
    .copy()
    .sort_values("row_number")
    .reset_index(drop=True)
)

audit_df[
    "baseline_prediction"
] = baseline_submission_df[
    prediction_column
].astype(str).values

audit_df[
    "new_prediction"
] = new_submission_df[
    prediction_column
].astype(str).values

decoded_lookup = (
    test_sentence_predictions_df
    .set_index("row_number")
)

audit_df[
    "raw_article_span"
] = audit_df[
    "row_number"
].map(
    decoded_lookup[
        "raw_article_span"
    ]
)

audit_df[
    "containing_sentence"
] = audit_df[
    "row_number"
].map(
    decoded_lookup[
        "containing_sentence"
    ]
)

audit_df[
    "answerability_margin"
] = audit_df[
    "row_number"
].map(
    decoded_lookup[
        "answerability_margin"
    ]
)

audit_df[
    "baseline_word_count"
] = audit_df[
    "baseline_prediction"
].apply(
    lambda text: len(str(text).split())
)

audit_df[
    "new_word_count"
] = audit_df[
    "new_prediction"
].apply(
    lambda text: len(str(text).split())
)

audit_df[
    "changed_from_baseline"
] = (
    audit_df[
        "baseline_prediction"
    ]
    != audit_df[
        "new_prediction"
    ]
)

audit_df[
    "operator"
] = np.where(
    audit_df[
        "predicted_type"
    ].astype(str).eq("passage"),
    "article_rank1_containing_sentence",
    "frozen_baseline"
)

new_submission_df.to_csv(
    NEW_SUBMISSION_PATH,
    index=False
)

audit_df.to_csv(
    NEW_AUDIT_PATH,
    index=False
)

# 8. Final report

passage_audit_df = audit_df[
    audit_df[
        "predicted_type"
    ].astype(str).eq("passage")
]

print("\nSubmission validation:")
print("Rows:", len(new_submission_df))
print("Columns:", list(new_submission_df.columns))
print("Empty predictions:", empty_prediction_count)
print(
    "Non-passage rows unchanged:",
    non_passage_unchanged
)
print(
    "Changed passage rows:",
    int(
        passage_audit_df[
            "changed_from_baseline"
        ].sum()
    ),
    "/",
    len(passage_audit_df)
)

print(
    "\nBaseline passage mean words:",
    round(
        passage_audit_df[
            "baseline_word_count"
        ].mean(),
        2
    )
)

print(
    "New passage mean words:",
    round(
        passage_audit_df[
            "new_word_count"
        ].mean(),
        2
    )
)

print(
    "New passage median words:",
    float(
        passage_audit_df[
            "new_word_count"
        ].median()
    )
)

print(
    "New passage maximum words:",
    int(
        passage_audit_df[
            "new_word_count"
        ].max()
    )
)

print("\nNew submission:")
print(NEW_SUBMISSION_PATH)

print("\nNew audit:")
print(NEW_AUDIT_PATH)

print(
    "\nSubmission checkpoint complete:",
    os.path.exists(
        NEW_SUBMISSION_PATH
    )
    and os.path.exists(
        NEW_AUDIT_PATH
    )
)

print(
    "\nSample changed passage predictions:"
)

display(
    passage_audit_df[
        passage_audit_df[
            "changed_from_baseline"
        ]
    ][
        [
            "row_number",
            "baseline_prediction",
            "raw_article_span",
            "new_prediction",
            "baseline_word_count",
            "new_word_count",
            "answerability_margin"
        ]
    ].head(12)
)

Expected passage rows: 183
Tokenized features: 602
Start-logit shape: (602, 384)
End-logit shape: (602, 384)

Decoded passage rows: 183
Missing passage rows: 0
Unexpected decoded rows: 0

Submission validation:
Rows: 400
Columns: ['id', 'spoiler']
Empty predictions: 0
Non-passage rows unchanged: True
Changed passage rows: 147 / 183

Baseline passage mean words: 46.69
New passage mean words: 24.28
New passage median words: 24.0
New passage maximum words: 92

New submission:
/kaggle/working/task2_passage_v1/submission_passage_containing_sentence_v1.csv

New audit:
/kaggle/working/task2_passage_v1/submission_passage_containing_sentence_v1_audit.csv

Submission checkpoint complete: True

Sample changed passage predictions:


,row_number,baseline_prediction,raw_article_span,new_prediction,baseline_word_count,new_word_count,answerability_margin
1,1,Why you SHOULD be selfish at work: Helping oth...,Helping others will lead to 'generosity burnou...,Why you SHOULD be selfish at work: Helping oth...,67,19,1.168779
7,7,Why You Should Never Pet A Service Dog Flynn t...,Flynn the dog does a lot to help out his 17-ye...,Flynn the dog does a lot to help out his 17-ye...,69,31,-1.066707
10,10,But what was perhaps the most baffling part of...,"""Who is watching your kids?""",But what was perhaps the most baffling part of...,48,30,3.944452
11,11,Foursquare's Dennis Crowley: This Mistake Will...,trying too hard,Building loyalty with your customers is crucia...,37,24,0.463420
13,13,"""People talk a lot and they know little,"" De G...","""People talk a lot and they know little,"" De G...","""I know how all this works, but what I think i...",68,24,-1.190698
17,17,"Gulf Futurism ""Black Friday"" sits comfortably ...",Gulf Futurism,Gulf Futurism,73,2,-4.143710
21,21,Swedish police eventually work it out Police i...,Drunk or Danish? Swedish police eventually wor...,Police in western Sweden initially thought a m...,38,32,-0.265569
22,22,Now it’s possible! The starting price being ex...,Now it’s possible,Now it’s possible!,56,3,-3.674701
25,25,"Yesterday, Hamill was asked by a female Star W...",Comic-Con,"Yesterday, Hamill was asked by a female Star W...",89,21,1.530513
27,27,Americans Are Convinced Climate Change Is Conn...,Most Americans think climate change,"Most Americans think climate change, and more ...",83,36,3.787436


In [23]:
import os
import numpy as np
import pandas as pd


SAFEGUARD_SUMMARY_PATH = os.path.join(
    WORK_DIR,
    "val_passage_confidence_safeguard_summary.csv"
)

HISTORICAL_PASSAGE_ROUTE_SCORE = 0.387260

# 1. Assemble validation sentence, raw-span, and confidence data

validation_sentence_df = (
    boundary_results_df[
        boundary_results_df["rule"].eq(
            "article_rank1_containing_sentence"
        )
    ][
        [
            "row_number",
            "gold_type",
            "prediction",
            "word_count",
            "meteor"
        ]
    ]
    .rename(
        columns={
            "prediction": "sentence_prediction",
            "word_count": "sentence_word_count",
            "meteor": "sentence_meteor"
        }
    )
    .copy()
)

validation_raw_df = (
    boundary_results_df[
        boundary_results_df["rule"].eq(
            "article_rank1_raw_span"
        )
    ][
        [
            "row_number",
            "prediction",
            "word_count",
            "meteor"
        ]
    ]
    .rename(
        columns={
            "prediction": "raw_prediction",
            "word_count": "raw_word_count",
            "meteor": "raw_meteor"
        }
    )
    .copy()
)

validation_rank1_features_df = (
    article_span_candidates_df[
        article_span_candidates_df[
            "candidate_rank"
        ].eq(1)
    ][
        [
            "row_number",
            "answerability_margin",
            "span_score",
            "null_score"
        ]
    ]
    .copy()
)

validation_safeguard_df = (
    validation_sentence_df
    .merge(
        validation_raw_df,
        on="row_number",
        how="inner",
        validate="one_to_one"
    )
    .merge(
        validation_rank1_features_df,
        on="row_number",
        how="inner",
        validate="one_to_one"
    )
    .sort_values("row_number")
    .reset_index(drop=True)
)

if len(validation_safeguard_df) != 168:
    raise RuntimeError(
        "Expected 168 validation passage-route rows, "
        f"but found {len(validation_safeguard_df)}."
    )

# 2. Predetermine a small set of interpretable safeguards

safeguard_rules = [
    {
        "rule": "replace_all",
        "minimum_margin": -np.inf,
        "minimum_words": 0
    },
    {
        "rule": "margin_at_least_minus_1",
        "minimum_margin": -1.0,
        "minimum_words": 0
    },
    {
        "rule": "margin_at_least_0",
        "minimum_margin": 0.0,
        "minimum_words": 0
    },
    {
        "rule": "at_least_8_words",
        "minimum_margin": -np.inf,
        "minimum_words": 8
    },
    {
        "rule": "margin_minus_1_and_8_words",
        "minimum_margin": -1.0,
        "minimum_words": 8
    },
    {
        "rule": "margin_0_and_8_words",
        "minimum_margin": 0.0,
        "minimum_words": 8
    },
    {
        "rule": "margin_minus_1_and_10_words",
        "minimum_margin": -1.0,
        "minimum_words": 10
    },
    {
        "rule": "margin_0_and_10_words",
        "minimum_margin": 0.0,
        "minimum_words": 10
    }
]

# 3. Evaluate validation safeguards

summary_records = []

for safeguard in safeguard_rules:
    use_sentence = (
        validation_safeguard_df[
            "answerability_margin"
        ].ge(
            safeguard["minimum_margin"]
        )
        &
        validation_safeguard_df[
            "sentence_word_count"
        ].ge(
            safeguard["minimum_words"]
        )
    )

    accepted_count = int(
        use_sentence.sum()
    )

    rejected_count = int(
        (~use_sentence).sum()
    )

    # Exact measurable comparison:
    # use containing sentence when accepted,
    # otherwise use the article model's raw span.
    raw_fallback_scores = np.where(
        use_sentence,
        validation_safeguard_df[
            "sentence_meteor"
        ],
        validation_safeguard_df[
            "raw_meteor"
        ]
    )

    exact_raw_fallback_score = float(
        raw_fallback_scores.mean()
    )

    # Approximation only:
    # rejected rows retain the historical route and are
    # assigned its overall mean score of 0.387260.
    estimated_baseline_fallback_score = float(
        (
            validation_safeguard_df.loc[
                use_sentence,
                "sentence_meteor"
            ].sum()
            + rejected_count
            * HISTORICAL_PASSAGE_ROUTE_SCORE
        )
        / len(validation_safeguard_df)
    )

    accepted_sentence_score = (
        float(
            validation_safeguard_df.loc[
                use_sentence,
                "sentence_meteor"
            ].mean()
        )
        if accepted_count
        else np.nan
    )

    rejected_sentence_score = (
        float(
            validation_safeguard_df.loc[
                ~use_sentence,
                "sentence_meteor"
            ].mean()
        )
        if rejected_count
        else np.nan
    )

    # Apply the same deployable condition to test only
    # to count how many frozen-baseline rows would change.
    test_use_sentence = (
        test_sentence_predictions_df[
            "answerability_margin"
        ].ge(
            safeguard["minimum_margin"]
        )
        &
        test_sentence_predictions_df[
            "sentence_word_count"
        ].ge(
            safeguard["minimum_words"]
        )
    )

    summary_records.append({
        "rule": safeguard["rule"],
        "minimum_margin": (
            safeguard["minimum_margin"]
        ),
        "minimum_words": (
            safeguard["minimum_words"]
        ),
        "validation_replacements": (
            accepted_count
        ),
        "validation_fallbacks": (
            rejected_count
        ),
        "accepted_sentence_meteor": (
            accepted_sentence_score
        ),
        "rejected_sentence_meteor": (
            rejected_sentence_score
        ),
        "exact_score_with_raw_span_fallback": (
            exact_raw_fallback_score
        ),
        "estimated_score_with_baseline_fallback": (
            estimated_baseline_fallback_score
        ),
        "estimated_gain_vs_historical_route": (
            estimated_baseline_fallback_score
            - HISTORICAL_PASSAGE_ROUTE_SCORE
        ),
        "test_replacements": int(
            test_use_sentence.sum()
        ),
        "test_fallbacks_to_frozen_baseline": int(
            (~test_use_sentence).sum()
        )
    })


safeguard_summary_df = (
    pd.DataFrame(summary_records)
    .sort_values(
        "estimated_score_with_baseline_fallback",
        ascending=False
    )
    .reset_index(drop=True)
)

safeguard_summary_df.to_csv(
    SAFEGUARD_SUMMARY_PATH,
    index=False
)

# 4. Diagnose confidence and length groups

validation_safeguard_df[
    "margin_group"
] = pd.cut(
    validation_safeguard_df[
        "answerability_margin"
    ],
    bins=[
        -np.inf,
        -2,
        -1,
        0,
        1,
        2,
        np.inf
    ],
    right=False
)

validation_safeguard_df[
    "length_group"
] = pd.cut(
    validation_safeguard_df[
        "sentence_word_count"
    ],
    bins=[
        0,
        5,
        8,
        10,
        15,
        25,
        40,
        np.inf
    ],
    right=False
)


print("Safeguard comparison:")
display(
    safeguard_summary_df
)

print("\nContaining-sentence performance by margin:")
display(
    validation_safeguard_df
    .groupby(
        "margin_group",
        observed=True
    )
    .agg(
        rows=(
            "row_number",
            "count"
        ),
        mean_meteor=(
            "sentence_meteor",
            "mean"
        ),
        mean_words=(
            "sentence_word_count",
            "mean"
        )
    )
    .reset_index()
)

print("\nContaining-sentence performance by length:")
display(
    validation_safeguard_df
    .groupby(
        "length_group",
        observed=True
    )
    .agg(
        rows=(
            "row_number",
            "count"
        ),
        mean_meteor=(
            "sentence_meteor",
            "mean"
        ),
        mean_margin=(
            "answerability_margin",
            "mean"
        )
    )
    .reset_index()
)

print("\nSaved safeguard summary:")
print(SAFEGUARD_SUMMARY_PATH)

Safeguard comparison:


,rule,minimum_margin,minimum_words,validation_replacements,validation_fallbacks,accepted_sentence_meteor,rejected_sentence_meteor,exact_score_with_raw_span_fallback,estimated_score_with_baseline_fallback,estimated_gain_vs_historical_route,test_replacements,test_fallbacks_to_frozen_baseline
0,margin_at_least_0,0.0,0,98,70,0.459468,0.345949,0.389538,0.429381,0.042121,123,60
1,at_least_8_words,-inf,8,152,16,0.432128,0.222551,0.411427,0.427855,0.040595,167,16
2,margin_0_and_8_words,0.0,8,91,77,0.460995,0.354464,0.390383,0.427200,0.039940,113,70
3,margin_0_and_10_words,0.0,10,89,79,0.460431,0.357796,0.392466,0.426023,0.038763,109,74
4,margin_minus_1_and_8_words,-1.0,8,108,60,0.445934,0.351390,0.394932,0.424979,0.037719,130,53
5,margin_minus_1_and_10_words,-1.0,10,106,62,0.445176,0.355736,0.397015,0.423802,0.036542,126,57
6,margin_at_least_minus_1,-1.0,0,118,50,0.435043,0.358185,0.394535,0.420822,0.033562,140,43
7,replace_all,-inf,0,168,0,0.412168,NaN,0.412168,0.412168,0.024908,183,0



Containing-sentence performance by margin:


,margin_group,rows,mean_meteor,mean_words
0,"[-inf, -2.0)",37,0.359040,24.729730
1,"[-2.0, -1.0)",13,0.355751,20.615385
2,"[-1.0, 0.0)",20,0.315360,19.950000
3,"[0.0, 1.0)",14,0.284632,24.571429
4,"[1.0, 2.0)",14,0.398755,27.142857
5,"[2.0, inf)",70,0.506578,23.614286



Containing-sentence performance by length:


,length_group,rows,mean_meteor,mean_margin
0,"[0.0, 5.0)",6,0.172020,-0.050497
1,"[5.0, 8.0)",10,0.252871,0.770968
2,"[8.0, 10.0)",5,0.604874,-1.557031
3,"[10.0, 15.0)",18,0.442506,0.396848
4,"[15.0, 25.0)",59,0.436294,1.158258
5,"[25.0, 40.0)",54,0.434465,1.677933
6,"[40.0, inf)",16,0.343222,0.151302



Saved safeguard summary:
/kaggle/working/task2_passage_v1/val_passage_confidence_safeguard_summary.csv


In [24]:
import os
import numpy as np
import pandas as pd


LENGTH_SAFEGUARD_PATH = os.path.join(
    WORK_DIR,
    "val_passage_length_safeguard_analysis.csv"
)

HISTORICAL_PASSAGE_ROUTE_SCORE = 0.387260

validation_count = len(
    validation_safeguard_df
)

replace_all_total_score = float(
    validation_safeguard_df[
        "sentence_meteor"
    ].sum()
)

replace_all_mean_score = (
    replace_all_total_score
    / validation_count
)


# Fixed, interpretable length safeguards.
length_rules = [
    {
        "rule": "replace_all",
        "minimum_words": 0,
        "maximum_words": np.inf
    },
    {
        "rule": "minimum_8_words",
        "minimum_words": 8,
        "maximum_words": np.inf
    },
    {
        "rule": "minimum_10_words",
        "minimum_words": 10,
        "maximum_words": np.inf
    },
    {
        "rule": "between_8_and_40_words",
        "minimum_words": 8,
        "maximum_words": 40
    },
    {
        "rule": "between_8_and_50_words",
        "minimum_words": 8,
        "maximum_words": 50
    },
    {
        "rule": "between_10_and_40_words",
        "minimum_words": 10,
        "maximum_words": 40
    }
]


analysis_records = []

for rule_config in length_rules:
    validation_accept = (
        validation_safeguard_df[
            "sentence_word_count"
        ].ge(
            rule_config["minimum_words"]
        )
        &
        validation_safeguard_df[
            "sentence_word_count"
        ].le(
            rule_config["maximum_words"]
        )
    )

    accepted_scores = (
        validation_safeguard_df.loc[
            validation_accept,
            "sentence_meteor"
        ]
    )

    rejected_scores = (
        validation_safeguard_df.loc[
            ~validation_accept,
            "sentence_meteor"
        ]
    )

    accepted_count = int(
        validation_accept.sum()
    )

    fallback_count = int(
        (~validation_accept).sum()
    )

    accepted_score_sum = float(
        accepted_scores.sum()
    )

    # Guaranteed score when every fallback is assigned zero.
    conservative_lower_bound = (
        accepted_score_sum
        / validation_count
    )

    if fallback_count > 0:
        fallback_needed_for_historical = (
            (
                HISTORICAL_PASSAGE_ROUTE_SCORE
                * validation_count
                - accepted_score_sum
            )
            / fallback_count
        )

        fallback_needed_for_replace_all = (
            (
                replace_all_total_score
                - accepted_score_sum
            )
            / fallback_count
        )
    else:
        fallback_needed_for_historical = np.nan
        fallback_needed_for_replace_all = np.nan

    # This remains only a rough estimate.
    estimated_with_global_historical_mean = (
        accepted_score_sum
        + fallback_count
        * HISTORICAL_PASSAGE_ROUTE_SCORE
    ) / validation_count

    test_accept = (
        test_sentence_predictions_df[
            "sentence_word_count"
        ].ge(
            rule_config["minimum_words"]
        )
        &
        test_sentence_predictions_df[
            "sentence_word_count"
        ].le(
            rule_config["maximum_words"]
        )
    )

    analysis_records.append({
        "rule": rule_config["rule"],
        "minimum_words": (
            rule_config["minimum_words"]
        ),
        "maximum_words": (
            rule_config["maximum_words"]
        ),
        "validation_replacements": (
            accepted_count
        ),
        "validation_fallbacks": (
            fallback_count
        ),
        "accepted_sentence_meteor": (
            float(accepted_scores.mean())
            if accepted_count
            else np.nan
        ),
        "rejected_sentence_meteor": (
            float(rejected_scores.mean())
            if fallback_count
            else np.nan
        ),
        "conservative_score_if_fallbacks_score_zero": (
            conservative_lower_bound
        ),
        "guaranteed_gain_vs_historical": (
            conservative_lower_bound
            - HISTORICAL_PASSAGE_ROUTE_SCORE
        ),
        "fallback_mean_needed_to_beat_historical": (
            fallback_needed_for_historical
        ),
        "fallback_mean_needed_to_beat_replace_all": (
            fallback_needed_for_replace_all
        ),
        "rough_score_using_global_historical_mean": (
            estimated_with_global_historical_mean
        ),
        "test_replacements": int(
            test_accept.sum()
        ),
        "test_fallbacks": int(
            (~test_accept).sum()
        )
    })


length_safeguard_df = (
    pd.DataFrame(
        analysis_records
    )
    .sort_values(
        [
            "guaranteed_gain_vs_historical",
            "rough_score_using_global_historical_mean"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

length_safeguard_df.to_csv(
    LENGTH_SAFEGUARD_PATH,
    index=False
)

print(
    "Historical passage-route score:",
    HISTORICAL_PASSAGE_ROUTE_SCORE
)

print(
    "Replace-all sentence score:",
    round(
        replace_all_mean_score,
        6
    )
)

print(
    "\nLength-only safeguard analysis:"
)

display(
    length_safeguard_df
)

print(
    "\nRules with a guaranteed lower bound "
    "above the historical route:"
)

display(
    length_safeguard_df[
        length_safeguard_df[
            "guaranteed_gain_vs_historical"
        ].gt(0)
    ][
        [
            "rule",
            "validation_replacements",
            "validation_fallbacks",
            "conservative_score_if_fallbacks_score_zero",
            "guaranteed_gain_vs_historical",
            "fallback_mean_needed_to_beat_replace_all",
            "test_replacements",
            "test_fallbacks"
        ]
    ]
)

print("\nSaved analysis:")
print(LENGTH_SAFEGUARD_PATH)

Historical passage-route score: 0.38726
Replace-all sentence score: 0.412168

Length-only safeguard analysis:


,rule,minimum_words,maximum_words,validation_replacements,validation_fallbacks,accepted_sentence_meteor,rejected_sentence_meteor,conservative_score_if_fallbacks_score_zero,guaranteed_gain_vs_historical,fallback_mean_needed_to_beat_historical,fallback_mean_needed_to_beat_replace_all,rough_score_using_global_historical_mean,test_replacements,test_fallbacks
0,replace_all,0,inf,168,0,0.412168,NaN,0.412168,0.024908,NaN,NaN,0.412168,183,0
1,minimum_8_words,8,inf,152,16,0.432128,0.222551,0.390973,0.003713,-0.038986,0.222551,0.427855,167,16
2,minimum_10_words,10,inf,147,21,0.426252,0.313581,0.372971,-0.014289,0.114314,0.313581,0.421378,160,23
3,between_8_and_50_words,8,50.0,145,23,0.429346,0.303875,0.370566,-0.016694,0.121936,0.303875,0.423584,160,23
4,between_8_and_40_words,8,40.0,139,29,0.443168,0.263583,0.366669,-0.020591,0.119287,0.263583,0.433517,154,29
5,between_10_and_40_words,10,40.0,134,34,0.437134,0.313773,0.348667,-0.038593,0.190697,0.313773,0.427041,147,36



Rules with a guaranteed lower bound above the historical route:


,rule,validation_replacements,validation_fallbacks,conservative_score_if_fallbacks_score_zero,guaranteed_gain_vs_historical,fallback_mean_needed_to_beat_replace_all,test_replacements,test_fallbacks
0,replace_all,168,0,0.412168,0.024908,NaN,183,0
1,minimum_8_words,152,16,0.390973,0.003713,0.222551,167,16



Saved analysis:
/kaggle/working/task2_passage_v1/val_passage_length_safeguard_analysis.csv


In [25]:
import os
import numpy as np
import pandas as pd


MINIMUM_SENTENCE_WORDS = 8

V2_SUBMISSION_PATH = os.path.join(
    WORK_DIR,
    "submission_passage_containing_sentence_min8_v2.csv"
)

V2_AUDIT_PATH = os.path.join(
    WORK_DIR,
    "submission_passage_containing_sentence_min8_v2_audit.csv"
)

# 1. Identify submission columns

sample_columns = list(sample_df.columns)

id_column = (
    "id"
    if "id" in sample_columns
    else sample_columns[0]
)

prediction_columns = [
    column
    for column in sample_columns
    if column != id_column
]

if len(prediction_columns) != 1:
    raise RuntimeError(
        "Could not identify exactly one prediction column."
    )

prediction_column = prediction_columns[0]

# 2. Validate baseline alignment

v2_submission_df = (
    baseline_submission_df[
        sample_columns
    ]
    .copy()
)

if len(v2_submission_df) != len(test_df):
    raise RuntimeError(
        "Baseline and test row counts do not match."
    )

if (
    v2_submission_df[id_column]
    .astype(str)
    .tolist()
    != test_df["id"]
    .astype(str)
    .tolist()
):
    raise RuntimeError(
        "Baseline IDs are not aligned with test rows."
    )

# 3. Apply the minimum-eight-word sentence rule

test_sentence_rule_df = (
    test_sentence_predictions_df[
        [
            "row_number",
            "raw_article_span",
            "containing_sentence",
            "answerability_margin",
            "raw_word_count",
            "sentence_word_count"
        ]
    ]
    .copy()
    .sort_values("row_number")
    .reset_index(drop=True)
)

test_sentence_rule_df[
    "accepted_new_sentence"
] = (
    test_sentence_rule_df[
        "sentence_word_count"
    ].ge(MINIMUM_SENTENCE_WORDS)
)

accepted_sentence_df = (
    test_sentence_rule_df[
        test_sentence_rule_df[
            "accepted_new_sentence"
        ]
    ]
)

for row in accepted_sentence_df.itertuples(
    index=False
):
    v2_submission_df.at[
        int(row.row_number),
        prediction_column
    ] = row.containing_sentence

# 4. Build a complete audit table

sentence_lookup_df = (
    test_sentence_rule_df
    .set_index("row_number")
)

v2_audit_df = (
    test_type_predictions_df
    .copy()
    .sort_values("row_number")
    .reset_index(drop=True)
)

v2_audit_df[
    "baseline_prediction"
] = (
    baseline_submission_df[
        prediction_column
    ]
    .astype(str)
    .values
)

v2_audit_df[
    "v1_replace_all_prediction"
] = (
    new_submission_df[
        prediction_column
    ]
    .astype(str)
    .values
)

v2_audit_df[
    "v2_prediction"
] = (
    v2_submission_df[
        prediction_column
    ]
    .astype(str)
    .values
)

v2_audit_df[
    "raw_article_span"
] = (
    v2_audit_df["row_number"]
    .map(
        sentence_lookup_df[
            "raw_article_span"
        ]
    )
)

v2_audit_df[
    "containing_sentence"
] = (
    v2_audit_df["row_number"]
    .map(
        sentence_lookup_df[
            "containing_sentence"
        ]
    )
)

v2_audit_df[
    "answerability_margin"
] = (
    v2_audit_df["row_number"]
    .map(
        sentence_lookup_df[
            "answerability_margin"
        ]
    )
)

v2_audit_df[
    "sentence_word_count"
] = (
    v2_audit_df["row_number"]
    .map(
        sentence_lookup_df[
            "sentence_word_count"
        ]
    )
)

v2_audit_df[
    "accepted_new_sentence"
] = (
    v2_audit_df["row_number"]
    .map(
        sentence_lookup_df[
            "accepted_new_sentence"
        ]
    )
    .fillna(False)
    .astype(bool)
)

v2_audit_df[
    "changed_from_baseline"
] = (
    v2_audit_df[
        "v2_prediction"
    ]
    !=
    v2_audit_df[
        "baseline_prediction"
    ]
)

v2_audit_df[
    "changed_from_v1"
] = (
    v2_audit_df[
        "v2_prediction"
    ]
    !=
    v2_audit_df[
        "v1_replace_all_prediction"
    ]
)

v2_audit_df[
    "operator"
] = np.select(
    [
        ~v2_audit_df[
            "predicted_type"
        ].astype(str).eq("passage"),

        v2_audit_df[
            "accepted_new_sentence"
        ]
    ],
    [
        "frozen_baseline_non_passage",
        "article_rank1_containing_sentence_min8"
    ],
    default="frozen_baseline_short_sentence"
)

v2_audit_df[
    "v2_word_count"
] = (
    v2_audit_df[
        "v2_prediction"
    ]
    .apply(
        lambda text:
        len(str(text).split())
    )
)

# 5. Validate submission integrity

expected_passage_rows = set(
    test_type_predictions_df.loc[
        test_type_predictions_df[
            "predicted_type"
        ].astype(str).eq("passage"),
        "row_number"
    ].astype(int)
)

accepted_rows = set(
    accepted_sentence_df[
        "row_number"
    ].astype(int)
)

fallback_rows = (
    expected_passage_rows
    - accepted_rows
)

non_passage_rows = (
    set(range(len(test_df)))
    - expected_passage_rows
)

changed_rows = set(
    np.flatnonzero(
        (
            v2_submission_df[
                prediction_column
            ].astype(str)
            !=
            baseline_submission_df[
                prediction_column
            ].astype(str)
        ).to_numpy()
    )
)

changes_outside_passage = (
    changed_rows
    - expected_passage_rows
)

non_passage_unchanged = bool(
    (
        v2_submission_df.loc[
            sorted(non_passage_rows),
            prediction_column
        ].astype(str).to_numpy()
        ==
        baseline_submission_df.loc[
            sorted(non_passage_rows),
            prediction_column
        ].astype(str).to_numpy()
    ).all()
)

fallbacks_match_baseline = bool(
    (
        v2_submission_df.loc[
            sorted(fallback_rows),
            prediction_column
        ].astype(str).to_numpy()
        ==
        baseline_submission_df.loc[
            sorted(fallback_rows),
            prediction_column
        ].astype(str).to_numpy()
    ).all()
)

empty_predictions = int(
    v2_submission_df[
        prediction_column
    ]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

if len(v2_submission_df) != 400:
    raise RuntimeError(
        "The v2 submission does not contain 400 rows."
    )

if list(v2_submission_df.columns) != sample_columns:
    raise RuntimeError(
        "The v2 submission column order is incorrect."
    )

if empty_predictions:
    raise RuntimeError(
        "The v2 submission contains empty predictions."
    )

if changes_outside_passage:
    raise RuntimeError(
        "Rows outside the passage route were changed."
    )

if not non_passage_unchanged:
    raise RuntimeError(
        "A non-passage prediction was modified."
    )

if not fallbacks_match_baseline:
    raise RuntimeError(
        "At least one short-sentence fallback does not "
        "match the frozen baseline."
    )


# 6. Save v2

v2_submission_df.to_csv(
    V2_SUBMISSION_PATH,
    index=False
)

v2_audit_df.to_csv(
    V2_AUDIT_PATH,
    index=False
)

# 7. Report results

passage_v2_audit_df = (
    v2_audit_df[
        v2_audit_df[
            "predicted_type"
        ].astype(str).eq("passage")
    ]
)

fallback_audit_df = (
    passage_v2_audit_df[
        ~passage_v2_audit_df[
            "accepted_new_sentence"
        ]
    ]
)

print("V2 submission validation:")
print("Rows:", len(v2_submission_df))
print("Columns:", list(v2_submission_df.columns))
print("Empty predictions:", empty_predictions)
print(
    "Non-passage rows unchanged:",
    non_passage_unchanged
)
print(
    "Short-sentence fallbacks match baseline:",
    fallbacks_match_baseline
)

print(
    "\nAccepted containing sentences:",
    len(accepted_rows)
)

print(
    "Fallbacks to frozen baseline:",
    len(fallback_rows)
)

print(
    "Rows actually changed from baseline:",
    len(changed_rows)
)

print(
    "Rows changed compared with v1:",
    int(
        v2_audit_df[
            "changed_from_v1"
        ].sum()
    )
)

print(
    "\nV2 passage mean words:",
    round(
        passage_v2_audit_df[
            "v2_word_count"
        ].mean(),
        2
    )
)

print(
    "V2 passage median words:",
    float(
        passage_v2_audit_df[
            "v2_word_count"
        ].median()
    )
)

print(
    "V2 passage maximum words:",
    int(
        passage_v2_audit_df[
            "v2_word_count"
        ].max()
    )
)

print("\nV2 submission:")
print(V2_SUBMISSION_PATH)

print("\nV2 audit:")
print(V2_AUDIT_PATH)

print(
    "\nV2 checkpoint complete:",
    os.path.exists(
        V2_SUBMISSION_PATH
    )
    and os.path.exists(
        V2_AUDIT_PATH
    )
)

print(
    "\nThe 16 short-sentence fallbacks:"
)

display(
    fallback_audit_df[
        [
            "row_number",
            "raw_article_span",
            "containing_sentence",
            "sentence_word_count",
            "baseline_prediction",
            "v2_prediction",
            "answerability_margin"
        ]
    ]
)

V2 submission validation:
Rows: 400
Columns: ['id', 'spoiler']
Empty predictions: 0
Non-passage rows unchanged: True
Short-sentence fallbacks match baseline: True

Accepted containing sentences: 167
Fallbacks to frozen baseline: 16
Rows actually changed from baseline: 134
Rows changed compared with v1: 13

V2 passage mean words: 27.44
V2 passage median words: 25.0
V2 passage maximum words: 92

V2 submission:
/kaggle/working/task2_passage_v1/submission_passage_containing_sentence_min8_v2.csv

V2 audit:
/kaggle/working/task2_passage_v1/submission_passage_containing_sentence_min8_v2_audit.csv

V2 checkpoint complete: True

The 16 short-sentence fallbacks:


/tmp/ipykernel_58/4002282170.py:206: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


,row_number,raw_article_span,containing_sentence,sentence_word_count,baseline_prediction,v2_prediction,answerability_margin
17,17,Gulf Futurism,Gulf Futurism,2.0,"Gulf Futurism ""Black Friday"" sits comfortably ...","Gulf Futurism ""Black Friday"" sits comfortably ...",-4.143710
22,22,Now it’s possible,Now it’s possible!,3.0,Now it’s possible! The starting price being ex...,Now it’s possible! The starting price being ex...,-3.674701
46,46,Harry Potter and the Philosopher's Stone illus...,Harry Potter and the Philosopher's Stone illus...,7.0,"""Harry pocketed it"" was the exact phrase in th...","""Harry pocketed it"" was the exact phrase in th...",-5.056290
48,48,Russell Wilson Listens To Gospel Before Kickoff,Russell Wilson Listens To Gospel Before Kickoff,7.0,Russell Wilson Listens To Gospel Before Kickof...,Russell Wilson Listens To Gospel Before Kickof...,-4.862952
52,52,1) Readers wanted to read about it,1) Readers wanted to read about it,7.0,1) Readers wanted to read about it In my pre-V...,1) Readers wanted to read about it In my pre-V...,1.184960
54,54,Pennsylvania,At least in Pennsylvania.,4.0,At least in Pennsylvania.,At least in Pennsylvania.,7.597549
85,85,Try It Yourself,Try It Yourself,3.0,Try It Yourself While there are tons of great ...,Try It Yourself While there are tons of great ...,-2.952123
97,97,"She loves eating, and she hates Trump","She loves eating, and she hates Trump!",7.0,"She loves eating, and she hates Trump!","She loves eating, and she hates Trump!",2.486162
165,165,"synaptic pruning.""","It’s called ""synaptic pruning.""",4.0,"synaptic pruning."" when we sleep When the micr...","synaptic pruning."" when we sleep When the micr...",0.351618
200,200,An e-Passport contains an electronic chip,An e-Passport contains an electronic chip.,6.0,An e-Passport contains an electronic chip.,An e-Passport contains an electronic chip.,1.385751


In [26]:
import os
import re
import json
import hashlib
import shutil
import pandas as pd


FINAL_SUBMISSION_PATH = os.path.join(
    WORK_DIR,
    "submission_passage_containing_sentence_min8_v2.csv"
)

FINAL_AUDIT_PATH = os.path.join(
    WORK_DIR,
    "submission_passage_containing_sentence_min8_v2_audit.csv"
)

EASY_ACCESS_PATH = (
    "/kaggle/working/"
    "FINAL_submission_passage_containing_sentence_min8_v2.csv"
)

MANIFEST_PATH = os.path.join(
    WORK_DIR,
    "submission_passage_containing_sentence_min8_v2_manifest.json"
)

# 1. Reload the saved files from disk

final_submission_df = pd.read_csv(
    FINAL_SUBMISSION_PATH
)

final_audit_df = pd.read_csv(
    FINAL_AUDIT_PATH
)

sample_columns = list(
    sample_df.columns
)

id_column = (
    "id"
    if "id" in sample_columns
    else sample_columns[0]
)

prediction_column = [
    column
    for column in sample_columns
    if column != id_column
][0]

# 2. Basic submission-format checks

format_checks = {
    "file_exists":
        os.path.exists(
            FINAL_SUBMISSION_PATH
        ),

    "row_count_is_400":
        len(final_submission_df) == 400,

    "columns_match_sample":
        list(final_submission_df.columns)
        == sample_columns,

    "ids_match_test_order":
        final_submission_df[id_column]
        .astype(str)
        .tolist()
        ==
        test_df["id"]
        .astype(str)
        .tolist(),

    "ids_are_unique":
        final_submission_df[
            id_column
        ].is_unique,

    "no_missing_predictions":
        final_submission_df[
            prediction_column
        ].notna().all(),

    "no_empty_predictions":
        final_submission_df[
            prediction_column
        ]
        .astype(str)
        .str.strip()
        .ne("")
        .all()
}

# 3. Confirm frozen-baseline protection

passage_mask = (
    test_type_predictions_df[
        "predicted_type"
    ]
    .astype(str)
    .eq("passage")
)

non_passage_indices = (
    test_type_predictions_df.loc[
        ~passage_mask,
        "row_number"
    ]
    .astype(int)
    .tolist()
)

short_fallback_indices = (
    final_audit_df.loc[
        final_audit_df[
            "operator"
        ].eq(
            "frozen_baseline_short_sentence"
        ),
        "row_number"
    ]
    .astype(int)
    .tolist()
)

non_passage_unchanged = (
    final_submission_df.loc[
        non_passage_indices,
        prediction_column
    ]
    .astype(str)
    .to_numpy()
    ==
    baseline_submission_df.loc[
        non_passage_indices,
        prediction_column
    ]
    .astype(str)
    .to_numpy()
).all()

short_fallbacks_unchanged = (
    final_submission_df.loc[
        short_fallback_indices,
        prediction_column
    ]
    .astype(str)
    .to_numpy()
    ==
    baseline_submission_df.loc[
        short_fallback_indices,
        prediction_column
    ]
    .astype(str)
    .to_numpy()
).all()

format_checks[
    "non_passage_rows_unchanged"
] = bool(non_passage_unchanged)

format_checks[
    "short_fallbacks_match_baseline"
] = bool(short_fallbacks_unchanged)


# 4. Check accepted predictions against source articles

def normalize_for_source_check(text):
    return re.sub(
        r"\s+",
        " ",
        str(text)
    ).strip().lower()


accepted_audit_df = final_audit_df[
    final_audit_df[
        "operator"
    ].eq(
        "article_rank1_containing_sentence_min8"
    )
].copy()

source_match_results = []

for row in accepted_audit_df.itertuples(
    index=False
):
    row_number = int(
        row.row_number
    )

    article_text = " ".join(
        [
            str(
                test_df.iloc[
                    row_number
                ].get(
                    "targetTitle",
                    ""
                )
                or ""
            )
        ]
        +
        [
            str(paragraph)
            for paragraph in (
                test_df.iloc[
                    row_number
                ][
                    "targetParagraphs"
                ]
                or []
            )
        ]
    )

    normalized_article = (
        normalize_for_source_check(
            article_text
        )
    )

    normalized_prediction = (
        normalize_for_source_check(
            row.v2_prediction
        )
    )

    source_match_results.append(
        normalized_prediction
        in normalized_article
    )

accepted_source_match_rate = (
    sum(source_match_results)
    / len(source_match_results)
    if source_match_results
    else 0.0
)

format_checks[
    "all_accepted_sentences_found_in_source"
] = all(
    source_match_results
)

# 5. Search for malformed output markers


prediction_series = (
    final_submission_df[
        prediction_column
    ]
    .astype(str)
)

paragraph_marker_count = int(
    prediction_series
    .str.contains(
        r"\bParagraph\s+\d+\s*:",
        case=False,
        regex=True
    )
    .sum()
)

title_marker_count = int(
    prediction_series
    .str.contains(
        r"^\s*Title\s*:",
        case=False,
        regex=True
    )
    .sum()
)

newline_count = int(
    prediction_series
    .str.contains(
        r"[\r\n]",
        regex=True
    )
    .sum()
)

word_counts = (
    prediction_series
    .str.split()
    .str.len()
)

format_checks[
    "no_paragraph_markers"
] = paragraph_marker_count == 0

format_checks[
    "no_title_markers"
] = title_marker_count == 0

format_checks[
    "no_embedded_newlines"
] = newline_count == 0

# 6. Compare final file with frozen baseline

changed_mask = (
    final_submission_df[
        prediction_column
    ].astype(str)
    !=
    baseline_submission_df[
        prediction_column
    ].astype(str)
)

changed_rows = int(
    changed_mask.sum()
)

changed_outside_passage = int(
    (
        changed_mask
        &
        ~passage_mask.to_numpy()
    ).sum()
)

# 7. Create hash, manifest, and easy-access copy

with open(
    FINAL_SUBMISSION_PATH,
    "rb"
) as file:
    submission_hash = hashlib.sha256(
        file.read()
    ).hexdigest()

manifest = {
    "submission_filename":
        os.path.basename(
            FINAL_SUBMISSION_PATH
        ),

    "frozen_public_baseline_score":
        0.45453,

    "experiment":
        (
            "Replace passage-routed predictions "
            "with the article model's rank-1 "
            "containing sentence when the sentence "
            "contains at least eight words."
        ),

    "validation_containing_sentence_score":
        0.412168,

    "historical_passage_route_score":
        0.387260,

    "repeated_heldout_mean_score":
        0.397689,

    "repeated_heldout_minimum_score":
        0.389238,

    "estimated_full_validation_gain":
        0.004380,

    "predicted_passage_test_rows":
        int(passage_mask.sum()),

    "accepted_new_sentences":
        int(
            len(
                accepted_audit_df
            )
        ),

    "fallbacks_to_frozen_baseline":
        int(
            len(
                short_fallback_indices
            )
        ),

    "rows_changed_from_baseline":
        changed_rows,

    "sha256":
        submission_hash
}

with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        manifest,
        file,
        indent=2
    )

shutil.copy2(
    FINAL_SUBMISSION_PATH,
    EASY_ACCESS_PATH
)

# 8. Final result

all_checks_passed = all(
    format_checks.values()
)

print("FINAL PRE-SUBMISSION AUDIT")
print("=" * 50)

for check_name, passed in (
    format_checks.items()
):
    print(
        f"{check_name}: {passed}"
    )

print("\nPrediction statistics:")
print("Rows:", len(final_submission_df))
print("Passage-routed rows:", int(passage_mask.sum()))
print(
    "Accepted containing sentences:",
    len(accepted_audit_df)
)
print(
    "Short-sentence fallbacks:",
    len(short_fallback_indices)
)
print(
    "Rows changed from baseline:",
    changed_rows
)
print(
    "Changes outside passage route:",
    changed_outside_passage
)
print(
    "Accepted source-match rate:",
    round(
        accepted_source_match_rate,
        6
    )
)

print("\nWord-count statistics:")
print("Mean:", round(word_counts.mean(), 2))
print("Median:", float(word_counts.median()))
print("Maximum:", int(word_counts.max()))

print("\nMalformed-output counts:")
print("Paragraph markers:", paragraph_marker_count)
print("Title markers:", title_marker_count)
print("Embedded newlines:", newline_count)

print("\nSHA-256:")
print(submission_hash)

print("\nEasy-access submission:")
print(EASY_ACCESS_PATH)

print("\nManifest:")
print(MANIFEST_PATH)

print(
    "\nREADY TO SUBMIT:",
    all_checks_passed
    and changed_outside_passage == 0
)

FINAL PRE-SUBMISSION AUDIT
file_exists: True
row_count_is_400: True
columns_match_sample: True
ids_match_test_order: True
ids_are_unique: True
no_missing_predictions: True
no_empty_predictions: True
non_passage_rows_unchanged: True
short_fallbacks_match_baseline: True
all_accepted_sentences_found_in_source: True
no_paragraph_markers: True
no_title_markers: True
no_embedded_newlines: True

Prediction statistics:
Rows: 400
Passage-routed rows: 183
Accepted containing sentences: 167
Short-sentence fallbacks: 16
Rows changed from baseline: 134
Changes outside passage route: 0
Accepted source-match rate: 1.0

Word-count statistics:
Mean: 27.56
Median: 21.0
Maximum: 139

Malformed-output counts:
Paragraph markers: 0
Title markers: 0
Embedded newlines: 0

SHA-256:
237b984683fa00994e49bddd3deb618f2f008c60540ced1f66f3bd40a4f56b40

Easy-access submission:
/kaggle/working/FINAL_submission_passage_containing_sentence_min8_v2.csv

Manifest:
/kaggle/working/task2_passage_v1/submission_passage_contai

In [29]:
import os
import re
import time
import numpy as np
import pandas as pd

from nltk.tokenize import TreebankWordTokenizer
from nltk.translate.meteor_score import meteor_score


TRAIN_RERANKER_POOL_PATH = os.path.join(
    WORK_DIR,
    "train_passage_sentence_reranker_candidates.pkl"
)

TRAIN_RERANKER_SUMMARY_PATH = os.path.join(
    WORK_DIR,
    "train_passage_sentence_reranker_oracle_summary.csv"
)

meteor_tokenizer = TreebankWordTokenizer()

# 1. Helpers

def normalize_whitespace(value):
    if value is None:
        return ""

    if isinstance(value, list):
        value = " ".join(
            str(item)
            for item in value
            if item is not None
        )

    return re.sub(
        r"\s+",
        " ",
        str(value)
    ).strip()


def combine_gold_spoiler(value):
    if isinstance(value, list):
        return normalize_whitespace(
            " ".join(
                normalize_whitespace(item)
                for item in value
                if normalize_whitespace(item)
            )
        )

    return normalize_whitespace(value)


def split_paragraph_into_sentences(paragraph):
    paragraph = normalize_whitespace(
        paragraph
    )

    if not paragraph:
        return []

    sentence_pattern = re.compile(
        r"""
        (
            .*?
            (?:
                [.!?]+
                ["'”’)]*
                (?=\s+|$)
                |
                $
            )
        )
        """,
        flags=re.DOTALL | re.VERBOSE
    )

    sentences = []

    for match in sentence_pattern.finditer(
        paragraph
    ):
        sentence = normalize_whitespace(
            match.group(1)
        )

        if sentence:
            sentences.append(sentence)

    if not sentences:
        sentences = [paragraph]

    return sentences


def calculate_candidate_meteor(
    gold_text,
    candidate_text
):
    gold_tokens = meteor_tokenizer.tokenize(
        normalize_whitespace(
            gold_text
        )
    )

    candidate_tokens = meteor_tokenizer.tokenize(
        normalize_whitespace(
            candidate_text
        )
    )

    if not gold_tokens or not candidate_tokens:
        return 0.0

    return float(
        meteor_score(
            [gold_tokens],
            candidate_tokens
        )
    )


def add_unique_candidate(
    candidate_dictionary,
    variant,
    candidate_text
):
    candidate_text = normalize_whitespace(
        candidate_text
    )

    normalized_key = re.sub(
        r"[^\w]+",
        " ",
        candidate_text.lower()
    )

    normalized_key = normalize_whitespace(
        normalized_key
    )

    if not normalized_key:
        return

    if normalized_key not in candidate_dictionary:
        candidate_dictionary[
            normalized_key
        ] = {
            "variant": variant,
            "candidate_text": candidate_text
        }

# 2. Select passage-labelled training examples

train_passage_mask = train_df[
    "tags"
].apply(
    lambda tags:
    isinstance(tags, list)
    and len(tags) > 0
    and str(tags[0]) == "passage"
)

train_passage_rows = (
    train_df[
        train_passage_mask
    ]
    .copy()
)

print(
    "Passage-labelled training rows:",
    len(train_passage_rows)
)

# 3. Generate sentence-boundary candidates

candidate_records = []
summary_records = []

generation_start = time.time()

for processed_number, (
    row_number,
    row
) in enumerate(
    train_passage_rows.iterrows(),
    start=1
):
    question = normalize_whitespace(
        row.get("postText", "")
    )

    title = normalize_whitespace(
        row.get("targetTitle", "")
    )

    description = normalize_whitespace(
        row.get(
            "targetDescription",
            ""
        )
    )

    gold_text = combine_gold_spoiler(
        row.get("spoiler", "")
    )

    paragraphs = row.get(
        "targetParagraphs",
        []
    )

    if not isinstance(paragraphs, list):
        paragraphs = []

    article_candidates = {}

    for paragraph_index, paragraph in enumerate(
        paragraphs
    ):
        sentences = (
            split_paragraph_into_sentences(
                paragraph
            )
        )

        for sentence_index, sentence in enumerate(
            sentences
        ):
            previous_sentence = (
                sentences[sentence_index - 1]
                if sentence_index > 0
                else ""
            )

            next_sentence = (
                sentences[sentence_index + 1]
                if sentence_index + 1
                < len(sentences)
                else ""
            )

            variants = {
                "sentence":
                    sentence,

                "previous_current":
                    " ".join(
                        part
                        for part in [
                            previous_sentence,
                            sentence
                        ]
                        if part
                    ),

                "current_next":
                    " ".join(
                        part
                        for part in [
                            sentence,
                            next_sentence
                        ]
                        if part
                    ),

                "previous_current_next":
                    " ".join(
                        part
                        for part in [
                            previous_sentence,
                            sentence,
                            next_sentence
                        ]
                        if part
                    )
            }

            for variant, candidate_text in (
                variants.items()
            ):
                normalized_key_before = set(
                    article_candidates.keys()
                )

                add_unique_candidate(
                    article_candidates,
                    variant,
                    candidate_text
                )

                normalized_key_after = set(
                    article_candidates.keys()
                )

                new_keys = (
                    normalized_key_after
                    - normalized_key_before
                )

                for new_key in new_keys:
                    article_candidates[
                        new_key
                    ][
                        "paragraph_index"
                    ] = paragraph_index

                    article_candidates[
                        new_key
                    ][
                        "sentence_index"
                    ] = sentence_index

    row_candidate_records = []

    for candidate in (
        article_candidates.values()
    ):
        candidate_text = candidate[
            "candidate_text"
        ]

        candidate_meteor = (
            calculate_candidate_meteor(
                gold_text,
                candidate_text
            )
        )

        row_candidate_records.append({
            "train_row_number": int(
                row_number
            ),
            "article_id": row["id"],
            "question": question,
            "title": title,
            "description": description,
            "gold_text": gold_text,
            "paragraph_index": int(
                candidate[
                    "paragraph_index"
                ]
            ),
            "sentence_index": int(
                candidate[
                    "sentence_index"
                ]
            ),
            "variant": candidate[
                "variant"
            ],
            "candidate_text": (
                candidate_text
            ),
            "word_count": len(
                candidate_text.split()
            ),
            "meteor": candidate_meteor
        })

    if not row_candidate_records:
        continue

    candidate_records.extend(
        row_candidate_records
    )

    row_candidate_df = pd.DataFrame(
        row_candidate_records
    )

    sentence_only_df = (
        row_candidate_df[
            row_candidate_df[
                "variant"
            ].eq("sentence")
        ]
    )

    best_row = (
        row_candidate_df
        .sort_values(
            "meteor",
            ascending=False
        )
        .iloc[0]
    )

    summary_records.append({
        "train_row_number": int(
            row_number
        ),
        "article_id": row["id"],
        "candidate_count": len(
            row_candidate_df
        ),
        "sentence_candidate_count": len(
            sentence_only_df
        ),
        "sentence_oracle_meteor": float(
            sentence_only_df[
                "meteor"
            ].max()
            if len(sentence_only_df)
            else 0.0
        ),
        "all_variant_oracle_meteor": float(
            best_row["meteor"]
        ),
        "best_variant": best_row[
            "variant"
        ],
        "best_candidate_text": best_row[
            "candidate_text"
        ],
        "gold_text": gold_text
    })

    if (
        processed_number % 200 == 0
        or processed_number
        == len(train_passage_rows)
    ):
        elapsed = (
            time.time()
            - generation_start
        )

        print(
            f"Processed {processed_number}/"
            f"{len(train_passage_rows)} "
            f"passage rows in {elapsed:.1f}s"
        )

# 4. Save the candidate pool and oracle summary

train_sentence_candidates_df = (
    pd.DataFrame(
        candidate_records
    )
)

train_sentence_oracle_df = (
    pd.DataFrame(
        summary_records
    )
)

train_sentence_candidates_df[
    "candidate_rank_by_meteor"
] = (
    train_sentence_candidates_df
    .groupby(
        "train_row_number"
    )[
        "meteor"
    ]
    .rank(
        method="first",
        ascending=False
    )
    .astype(int)
)

train_sentence_candidates_df.to_pickle(
    TRAIN_RERANKER_POOL_PATH
)

train_sentence_oracle_df.to_csv(
    TRAIN_RERANKER_SUMMARY_PATH,
    index=False
)

# 5. Report candidate coverage

print(
    "\nGeneration seconds:",
    round(
        time.time()
        - generation_start,
        2
    )
)

print(
    "Training articles with candidates:",
    train_sentence_oracle_df[
        "train_row_number"
    ].nunique()
)

print(
    "Total unique candidates:",
    len(
        train_sentence_candidates_df
    )
)

print(
    "Average candidates per article:",
    round(
        train_sentence_oracle_df[
            "candidate_count"
        ].mean(),
        2
    )
)

print(
    "\nMean single-sentence oracle:",
    round(
        train_sentence_oracle_df[
            "sentence_oracle_meteor"
        ].mean(),
        6
    )
)

print(
    "Mean all-variant oracle:",
    round(
        train_sentence_oracle_df[
            "all_variant_oracle_meteor"
        ].mean(),
        6
    )
)

print(
    "\nAll-variant oracle coverage:"
)

for threshold in [
    0.30,
    0.50,
    0.70,
    0.90,
    0.99
]:
    coverage = (
        train_sentence_oracle_df[
            "all_variant_oracle_meteor"
        ]
        .ge(threshold)
        .mean()
    )

    print(
        f"METEOR >= {threshold:.2f}: "
        f"{coverage:.3%}"
    )

print(
    "\nBest-variant counts:"
)

print(
    train_sentence_oracle_df[
        "best_variant"
    ].value_counts()
)

print(
    "\nSaved candidate pool:"
)

print(TRAIN_RERANKER_POOL_PATH)

print(
    "\nSaved oracle summary:"
)

print(TRAIN_RERANKER_SUMMARY_PATH)

print(
    "\nCheckpoint complete:",
    os.path.exists(
        TRAIN_RERANKER_POOL_PATH
    )
    and os.path.exists(
        TRAIN_RERANKER_SUMMARY_PATH
    )
)

Passage-labelled training rows: 1274
Processed 200/1274 passage rows in 26.8s
Processed 400/1274 passage rows in 41.6s
Processed 600/1274 passage rows in 61.5s
Processed 800/1274 passage rows in 79.6s
Processed 1000/1274 passage rows in 94.7s
Processed 1200/1274 passage rows in 112.9s
Processed 1274/1274 passage rows in 118.9s

Generation seconds: 119.16
Training articles with candidates: 1274
Total unique candidates: 71364
Average candidates per article: 56.02

Mean single-sentence oracle: 0.845814
Mean all-variant oracle: 0.909791

All-variant oracle coverage:
METEOR >= 0.30: 99.294%
METEOR >= 0.50: 98.195%
METEOR >= 0.70: 92.465%
METEOR >= 0.90: 68.210%
METEOR >= 0.99: 40.110%

Best-variant counts:
best_variant
sentence                 1032
current_next              174
previous_current_next      68
Name: count, dtype: int64

Saved candidate pool:
/kaggle/working/task2_passage_v1/train_passage_sentence_reranker_candidates.pkl

Saved oracle summary:
/kaggle/working/task2_passage_v1/t

In [30]:
import os
import re
import time
import pandas as pd


VAL_RERANKER_POOL_PATH = os.path.join(
    WORK_DIR,
    "val_routed_sentence_reranker_candidates.pkl"
)

VAL_RERANKER_SUMMARY_PATH = os.path.join(
    WORK_DIR,
    "val_routed_sentence_reranker_oracle_summary.csv"
)


def build_sentence_candidate_dictionary(paragraphs):
    candidate_dictionary = {}

    if not isinstance(paragraphs, list):
        paragraphs = []

    for paragraph_index, paragraph in enumerate(
        paragraphs
    ):
        sentences = split_paragraph_into_sentences(
            paragraph
        )

        for sentence_index, sentence in enumerate(
            sentences
        ):
            previous_sentence = (
                sentences[sentence_index - 1]
                if sentence_index > 0
                else ""
            )

            next_sentence = (
                sentences[sentence_index + 1]
                if sentence_index + 1
                < len(sentences)
                else ""
            )

            variants = {
                "sentence": sentence,

                "previous_current": " ".join(
                    part
                    for part in [
                        previous_sentence,
                        sentence
                    ]
                    if part
                ),

                "current_next": " ".join(
                    part
                    for part in [
                        sentence,
                        next_sentence
                    ]
                    if part
                ),

                "previous_current_next": " ".join(
                    part
                    for part in [
                        previous_sentence,
                        sentence,
                        next_sentence
                    ]
                    if part
                )
            }

            for variant, candidate_text in variants.items():
                candidate_text = normalize_whitespace(
                    candidate_text
                )

                normalized_key = re.sub(
                    r"[^\w]+",
                    " ",
                    candidate_text.lower()
                )

                normalized_key = normalize_whitespace(
                    normalized_key
                )

                if not normalized_key:
                    continue

                if normalized_key not in candidate_dictionary:
                    candidate_dictionary[
                        normalized_key
                    ] = {
                        "variant": variant,
                        "candidate_text": candidate_text,
                        "paragraph_index": paragraph_index,
                        "sentence_index": sentence_index
                    }

    return candidate_dictionary

# Build candidates for all 168 validation rows routed as passage by the type classifier.
validation_candidate_records = []
validation_summary_records = []

generation_start = time.time()

sorted_passage_rows = sorted(
    int(row_number)
    for row_number in passage_row_numbers
)

for processed_number, row_number in enumerate(
    sorted_passage_rows,
    start=1
):
    row = val_df.iloc[row_number]

    question = normalize_whitespace(
        row.get("postText", "")
    )

    title = normalize_whitespace(
        row.get("targetTitle", "")
    )

    description = normalize_whitespace(
        row.get(
            "targetDescription",
            ""
        )
    )

    gold_text = combine_gold_spoiler(
        row.get("spoiler", "")
    )

    gold_type = normalize_whitespace(
        row.get("tags", [""])[0]
    )

    candidate_dictionary = (
        build_sentence_candidate_dictionary(
            row.get(
                "targetParagraphs",
                []
            )
        )
    )

    row_candidate_records = []

    for candidate in candidate_dictionary.values():
        candidate_text = candidate[
            "candidate_text"
        ]

        candidate_meteor = (
            calculate_candidate_meteor(
                gold_text,
                candidate_text
            )
        )

        row_candidate_records.append({
            "row_number": row_number,
            "article_id": row["id"],
            "gold_type": gold_type,
            "question": question,
            "title": title,
            "description": description,
            "gold_text": gold_text,
            "paragraph_index": int(
                candidate[
                    "paragraph_index"
                ]
            ),
            "sentence_index": int(
                candidate[
                    "sentence_index"
                ]
            ),
            "variant": candidate[
                "variant"
            ],
            "candidate_text": candidate_text,
            "word_count": len(
                candidate_text.split()
            ),
            "meteor": candidate_meteor
        })

    if not row_candidate_records:
        raise RuntimeError(
            f"No sentence candidates generated "
            f"for validation row {row_number}."
        )

    validation_candidate_records.extend(
        row_candidate_records
    )

    row_candidate_df = pd.DataFrame(
        row_candidate_records
    )

    sentence_only_df = row_candidate_df[
        row_candidate_df[
            "variant"
        ].eq("sentence")
    ]

    best_row = (
        row_candidate_df
        .sort_values(
            "meteor",
            ascending=False
        )
        .iloc[0]
    )

    validation_summary_records.append({
        "row_number": row_number,
        "article_id": row["id"],
        "gold_type": gold_type,
        "candidate_count": len(
            row_candidate_df
        ),
        "sentence_candidate_count": len(
            sentence_only_df
        ),
        "sentence_oracle_meteor": float(
            sentence_only_df[
                "meteor"
            ].max()
            if len(sentence_only_df)
            else 0.0
        ),
        "all_variant_oracle_meteor": float(
            best_row["meteor"]
        ),
        "best_variant": best_row[
            "variant"
        ],
        "best_candidate_text": best_row[
            "candidate_text"
        ],
        "gold_text": gold_text
    })

    if (
        processed_number % 40 == 0
        or processed_number
        == len(sorted_passage_rows)
    ):
        print(
            f"Processed {processed_number}/"
            f"{len(sorted_passage_rows)} rows"
        )

# Save candidate pool

val_sentence_candidates_df = pd.DataFrame(
    validation_candidate_records
)

val_sentence_oracle_df = pd.DataFrame(
    validation_summary_records
)

val_sentence_candidates_df[
    "candidate_rank_by_meteor"
] = (
    val_sentence_candidates_df
    .groupby("row_number")[
        "meteor"
    ]
    .rank(
        method="first",
        ascending=False
    )
    .astype(int)
)

val_sentence_candidates_df.to_pickle(
    VAL_RERANKER_POOL_PATH
)

val_sentence_oracle_df.to_csv(
    VAL_RERANKER_SUMMARY_PATH,
    index=False
)

# Report validation candidate coverage

true_passage_oracle_df = (
    val_sentence_oracle_df[
        val_sentence_oracle_df[
            "gold_type"
        ].eq("passage")
    ]
)

print(
    "\nGeneration seconds:",
    round(
        time.time()
        - generation_start,
        2
    )
)

print(
    "Validation routed rows:",
    val_sentence_oracle_df[
        "row_number"
    ].nunique()
)

print(
    "True-passage rows:",
    len(true_passage_oracle_df)
)

print(
    "Total unique candidates:",
    len(val_sentence_candidates_df)
)

print(
    "Average candidates per article:",
    round(
        val_sentence_oracle_df[
            "candidate_count"
        ].mean(),
        2
    )
)

print(
    "\nAll 168 rows:"
)

print(
    "Mean sentence oracle:",
    round(
        val_sentence_oracle_df[
            "sentence_oracle_meteor"
        ].mean(),
        6
    )
)

print(
    "Mean all-variant oracle:",
    round(
        val_sentence_oracle_df[
            "all_variant_oracle_meteor"
        ].mean(),
        6
    )
)

print(
    "\nTrue-passage subset:"
)

print(
    "Mean sentence oracle:",
    round(
        true_passage_oracle_df[
            "sentence_oracle_meteor"
        ].mean(),
        6
    )
)

print(
    "Mean all-variant oracle:",
    round(
        true_passage_oracle_df[
            "all_variant_oracle_meteor"
        ].mean(),
        6
    )
)

print(
    "\nAll-variant oracle coverage "
    "across the 168 routed rows:"
)

for threshold in [
    0.30,
    0.50,
    0.70,
    0.90,
    0.99
]:
    coverage = (
        val_sentence_oracle_df[
            "all_variant_oracle_meteor"
        ]
        .ge(threshold)
        .mean()
    )

    print(
        f"METEOR >= {threshold:.2f}: "
        f"{coverage:.3%}"
    )

print(
    "\nBest-variant counts:"
)

print(
    val_sentence_oracle_df[
        "best_variant"
    ].value_counts()
)

print(
    "\nSaved validation candidate pool:"
)

print(VAL_RERANKER_POOL_PATH)

print(
    "\nCheckpoint complete:",
    os.path.exists(
        VAL_RERANKER_POOL_PATH
    )
    and os.path.exists(
        VAL_RERANKER_SUMMARY_PATH
    )
)

Processed 40/168 rows
Processed 80/168 rows
Processed 120/168 rows
Processed 160/168 rows
Processed 168/168 rows

Generation seconds: 17.13
Validation routed rows: 168
True-passage rows: 127
Total unique candidates: 10384
Average candidates per article: 61.81

All 168 rows:
Mean sentence oracle: 0.777262
Mean all-variant oracle: 0.833572

True-passage subset:
Mean sentence oracle: 0.849025
Mean all-variant oracle: 0.920231

All-variant oracle coverage across the 168 routed rows:
METEOR >= 0.30: 95.238%
METEOR >= 0.50: 91.071%
METEOR >= 0.70: 77.976%
METEOR >= 0.90: 54.167%
METEOR >= 0.99: 29.762%

Best-variant counts:
best_variant
sentence                 136
current_next              26
previous_current_next      6
Name: count, dtype: int64

Saved validation candidate pool:
/kaggle/working/task2_passage_v1/val_routed_sentence_reranker_candidates.pkl

Checkpoint complete: True


In [ ]:
import os
import re
import numpy as np
import pandas as pd


RERANKER_TRAIN_SAMPLE_PATH = os.path.join(
    WORK_DIR,
    "train_passage_reranker_balanced_sample.pkl"
)

RERANKER_VAL_READY_PATH = os.path.join(
    WORK_DIR,
    "val_routed_sentence_reranker_ready.pkl"
)

RANDOM_SEED = 641
MAX_SELECTED_PER_ARTICLE = 18


# ---------------------------------------------------------
# 1. Lexical-overlap helpers for hard-negative selection
# ---------------------------------------------------------

def reranker_word_set(text):
    return set(
        re.findall(
            r"\b\w+\b",
            normalize_whitespace(text).lower()
        )
    )


def lexical_overlap_score(
    candidate_text,
    question,
    title
):
    candidate_words = reranker_word_set(
        candidate_text
    )

    query_words = (
        reranker_word_set(question)
        |
        reranker_word_set(title)
    )

    if not candidate_words or not query_words:
        return 0.0

    intersection_size = len(
        candidate_words & query_words
    )

    union_size = len(
        candidate_words | query_words
    )

    return (
        intersection_size / union_size
        if union_size
        else 0.0
    )


# ---------------------------------------------------------
# 2. Add model-input text and sampling features
# ---------------------------------------------------------

train_reranker_pool_df = (
    train_sentence_candidates_df
    .copy()
    .reset_index(drop=True)
)

train_reranker_pool_df[
    "lexical_overlap"
] = [
    lexical_overlap_score(
        candidate_text,
        question,
        title
    )
    for candidate_text, question, title
    in zip(
        train_reranker_pool_df[
            "candidate_text"
        ],
        train_reranker_pool_df[
            "question"
        ],
        train_reranker_pool_df[
            "title"
        ]
    )
]

train_reranker_pool_df[
    "model_text_a"
] = (
    "CLICKBAIT: "
    + train_reranker_pool_df[
        "question"
    ].fillna("").astype(str)
    + " TITLE: "
    + train_reranker_pool_df[
        "title"
    ].fillna("").astype(str)
)

train_reranker_pool_df[
    "model_text_b"
] = (
    "CANDIDATE: "
    + train_reranker_pool_df[
        "candidate_text"
    ].fillna("").astype(str)
    + " DESCRIPTION: "
    + train_reranker_pool_df[
        "description"
    ].fillna("").astype(str)
)


# ---------------------------------------------------------
# 3. Deterministically sample a balanced set per article
# ---------------------------------------------------------

random_generator = np.random.default_rng(
    RANDOM_SEED
)

selected_index_records = []

for train_row_number, group in (
    train_reranker_pool_df.groupby(
        "train_row_number",
        sort=True
    )
):
    group = group.copy()

    selected_indices = []

    def add_indices(indices):
        for index in indices:
            index = int(index)

            if index not in selected_indices:
                selected_indices.append(index)

            if (
                len(selected_indices)
                >= MAX_SELECTED_PER_ARTICLE
            ):
                break

    # Always retain the strongest two candidates.
    strongest_indices = (
        group
        .sort_values(
            [
                "meteor",
                "word_count"
            ],
            ascending=[
                False,
                True
            ]
        )
        .head(2)
        .index
        .tolist()
    )

    add_indices(strongest_indices)

    # Additional high-quality candidates.
    high_quality_df = (
        group[
            group["meteor"].ge(0.70)
        ]
        .drop(
            index=selected_indices,
            errors="ignore"
        )
        .sort_values(
            "meteor",
            ascending=False
        )
        .head(3)
    )

    add_indices(
        high_quality_df.index.tolist()
    )

    # Medium candidates give the regression model
    # useful distinctions between plausible answers.
    medium_df = (
        group[
            group["meteor"].between(
                0.30,
                0.70,
                inclusive="left"
            )
        ]
        .drop(
            index=selected_indices,
            errors="ignore"
        )
    )

    if len(medium_df):
        medium_target_values = np.linspace(
            0.65,
            0.30,
            num=min(
                4,
                len(medium_df)
            )
        )

        available_medium_df = medium_df.copy()

        medium_indices = []

        for target_value in medium_target_values:
            if available_medium_df.empty:
                break

            nearest_index = (
                (
                    available_medium_df[
                        "meteor"
                    ]
                    - target_value
                )
                .abs()
                .idxmin()
            )

            medium_indices.append(
                nearest_index
            )

            available_medium_df = (
                available_medium_df.drop(
                    index=nearest_index
                )
            )

        add_indices(medium_indices)

    # Hard negatives have poor gold similarity but high
    # lexical overlap with the clickbait post or title.
    hard_negative_df = (
        group[
            group["meteor"].lt(0.30)
        ]
        .drop(
            index=selected_indices,
            errors="ignore"
        )
        .sort_values(
            [
                "lexical_overlap",
                "meteor"
            ],
            ascending=[
                False,
                False
            ]
        )
        .head(5)
    )

    add_indices(
        hard_negative_df.index.tolist()
    )

    # Add random low-quality negatives so the model does
    # not learn only from lexically deceptive negatives.
    remaining_negative_df = (
        group[
            group["meteor"].lt(0.30)
        ]
        .drop(
            index=selected_indices,
            errors="ignore"
        )
    )

    remaining_slots = (
        MAX_SELECTED_PER_ARTICLE
        - len(selected_indices)
    )

    if (
        remaining_slots > 0
        and len(remaining_negative_df) > 0
    ):
        sample_size = min(
            remaining_slots,
            len(remaining_negative_df)
        )

        sampled_negative_indices = (
            random_generator.choice(
                remaining_negative_df.index.to_numpy(),
                size=sample_size,
                replace=False
            )
            .tolist()
        )

        add_indices(
            sampled_negative_indices
        )

    # Fill any remaining slots from the highest unselected
    # candidates so article coverage remains consistent.
    remaining_slots = (
        MAX_SELECTED_PER_ARTICLE
        - len(selected_indices)
    )

    if remaining_slots > 0:
        remaining_df = (
            group
            .drop(
                index=selected_indices,
                errors="ignore"
            )
            .sort_values(
                "meteor",
                ascending=False
            )
            .head(remaining_slots)
        )

        add_indices(
            remaining_df.index.tolist()
        )

    for selected_index in selected_indices:
        selected_index_records.append({
            "train_row_number": int(
                train_row_number
            ),
            "source_index": int(
                selected_index
            )
        })


selected_index_df = pd.DataFrame(
    selected_index_records
)

balanced_train_df = (
    train_reranker_pool_df.loc[
        selected_index_df[
            "source_index"
        ].to_numpy()
    ]
    .copy()
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 4. Add candidate relevance classes for diagnostics
# ---------------------------------------------------------

balanced_train_df[
    "relevance_band"
] = pd.cut(
    balanced_train_df["meteor"],
    bins=[
        -0.000001,
        0.10,
        0.30,
        0.50,
        0.70,
        0.90,
        1.000001
    ],
    labels=[
        "0.00-0.10",
        "0.10-0.30",
        "0.30-0.50",
        "0.50-0.70",
        "0.70-0.90",
        "0.90-1.00"
    ],
    include_lowest=True
)


# ---------------------------------------------------------
# 5. Prepare the full validation candidate pool
# ---------------------------------------------------------

val_reranker_ready_df = (
    val_sentence_candidates_df
    .copy()
    .reset_index(drop=True)
)

val_reranker_ready_df[
    "lexical_overlap"
] = [
    lexical_overlap_score(
        candidate_text,
        question,
        title
    )
    for candidate_text, question, title
    in zip(
        val_reranker_ready_df[
            "candidate_text"
        ],
        val_reranker_ready_df[
            "question"
        ],
        val_reranker_ready_df[
            "title"
        ]
    )
]

val_reranker_ready_df[
    "model_text_a"
] = (
    "CLICKBAIT: "
    + val_reranker_ready_df[
        "question"
    ].fillna("").astype(str)
    + " TITLE: "
    + val_reranker_ready_df[
        "title"
    ].fillna("").astype(str)
)

val_reranker_ready_df[
    "model_text_b"
] = (
    "CANDIDATE: "
    + val_reranker_ready_df[
        "candidate_text"
    ].fillna("").astype(str)
    + " DESCRIPTION: "
    + val_reranker_ready_df[
        "description"
    ].fillna("").astype(str)
)


# ---------------------------------------------------------
# 6. Save both prepared datasets
# ---------------------------------------------------------

balanced_train_df.to_pickle(
    RERANKER_TRAIN_SAMPLE_PATH
)

val_reranker_ready_df.to_pickle(
    RERANKER_VAL_READY_PATH
)


# ---------------------------------------------------------
# 7. Integrity checks and report
# ---------------------------------------------------------

training_article_counts = (
    balanced_train_df
    .groupby("train_row_number")
    .size()
)

missing_training_articles = (
    set(
        train_sentence_candidates_df[
            "train_row_number"
        ].unique()
    )
    -
    set(
        balanced_train_df[
            "train_row_number"
        ].unique()
    )
)

best_candidate_retained = (
    balanced_train_df
    .groupby("train_row_number")[
        "meteor"
    ]
    .max()
    .sort_index()
    .equals(
        train_sentence_candidates_df
        .groupby("train_row_number")[
            "meteor"
        ]
        .max()
        .sort_index()
    )
)

print(
    "Balanced training candidates:",
    len(balanced_train_df)
)

print(
    "Training articles represented:",
    balanced_train_df[
        "train_row_number"
    ].nunique()
)

print(
    "Missing training articles:",
    len(missing_training_articles)
)

print(
    "Best candidate retained for every article:",
    best_candidate_retained
)

print(
    "\nCandidates per training article:"
)

print(
    training_article_counts.describe()
)

print(
    "\nTraining target distribution:"
)

print(
    balanced_train_df[
        "meteor"
    ].describe()
)

print(
    "\nRelevance-band counts:"
)

print(
    balanced_train_df[
        "relevance_band"
    ].value_counts().sort_index()
)

print(
    "\nSelected variant counts:"
)

print(
    balanced_train_df[
        "variant"
    ].value_counts()
)

print(
    "\nCandidate word-count summary:"
)

print(
    balanced_train_df[
        "word_count"
    ].describe()
)

print(
    "\nFull validation candidates prepared:",
    len(val_reranker_ready_df)
)

print(
    "Validation rows represented:",
    val_reranker_ready_df[
        "row_number"
    ].nunique()
)

print(
    "\nSaved balanced training set:"
)

print(RERANKER_TRAIN_SAMPLE_PATH)

print(
    "\nSaved validation set:"
)

print(RERANKER_VAL_READY_PATH)

print(
    "\nCheckpoint complete:",
    os.path.exists(
        RERANKER_TRAIN_SAMPLE_PATH
    )
    and os.path.exists(
        RERANKER_VAL_READY_PATH
    )
    and len(missing_training_articles) == 0
    and best_candidate_retained
)

In [31]:
import os
import re
import numpy as np
import pandas as pd


RERANKER_TRAIN_SAMPLE_PATH = os.path.join(
    WORK_DIR,
    "train_passage_reranker_balanced_sample.pkl"
)

RERANKER_VAL_READY_PATH = os.path.join(
    WORK_DIR,
    "val_routed_sentence_reranker_ready.pkl"
)

RANDOM_SEED = 641
MAX_SELECTED_PER_ARTICLE = 18

# 1. Lexical-overlap helpers for hard-negative selection

def reranker_word_set(text):
    return set(
        re.findall(
            r"\b\w+\b",
            normalize_whitespace(text).lower()
        )
    )


def lexical_overlap_score(
    candidate_text,
    question,
    title
):
    candidate_words = reranker_word_set(
        candidate_text
    )

    query_words = (
        reranker_word_set(question)
        |
        reranker_word_set(title)
    )

    if not candidate_words or not query_words:
        return 0.0

    intersection_size = len(
        candidate_words & query_words
    )

    union_size = len(
        candidate_words | query_words
    )

    return (
        intersection_size / union_size
        if union_size
        else 0.0
    )

# 2. Add model-input text and sampling features
train_reranker_pool_df = (
    train_sentence_candidates_df
    .copy()
    .reset_index(drop=True)
)

train_reranker_pool_df[
    "lexical_overlap"
] = [
    lexical_overlap_score(
        candidate_text,
        question,
        title
    )
    for candidate_text, question, title
    in zip(
        train_reranker_pool_df[
            "candidate_text"
        ],
        train_reranker_pool_df[
            "question"
        ],
        train_reranker_pool_df[
            "title"
        ]
    )
]

train_reranker_pool_df[
    "model_text_a"
] = (
    "CLICKBAIT: "
    + train_reranker_pool_df[
        "question"
    ].fillna("").astype(str)
    + " TITLE: "
    + train_reranker_pool_df[
        "title"
    ].fillna("").astype(str)
)

train_reranker_pool_df[
    "model_text_b"
] = (
    "CANDIDATE: "
    + train_reranker_pool_df[
        "candidate_text"
    ].fillna("").astype(str)
    + " DESCRIPTION: "
    + train_reranker_pool_df[
        "description"
    ].fillna("").astype(str)
)

# 3. Deterministically sample a balanced set per article

random_generator = np.random.default_rng(
    RANDOM_SEED
)

selected_index_records = []

for train_row_number, group in (
    train_reranker_pool_df.groupby(
        "train_row_number",
        sort=True
    )
):
    group = group.copy()

    selected_indices = []

    def add_indices(indices):
        for index in indices:
            index = int(index)

            if index not in selected_indices:
                selected_indices.append(index)

            if (
                len(selected_indices)
                >= MAX_SELECTED_PER_ARTICLE
            ):
                break

    # Always retain the strongest two candidates.
    strongest_indices = (
        group
        .sort_values(
            [
                "meteor",
                "word_count"
            ],
            ascending=[
                False,
                True
            ]
        )
        .head(2)
        .index
        .tolist()
    )

    add_indices(strongest_indices)

    # Additional high-quality candidates.
    high_quality_df = (
        group[
            group["meteor"].ge(0.70)
        ]
        .drop(
            index=selected_indices,
            errors="ignore"
        )
        .sort_values(
            "meteor",
            ascending=False
        )
        .head(3)
    )

    add_indices(
        high_quality_df.index.tolist()
    )

    # Medium candidates give the regression model
    # useful distinctions between plausible answers.
    medium_df = (
        group[
            group["meteor"].between(
                0.30,
                0.70,
                inclusive="left"
            )
        ]
        .drop(
            index=selected_indices,
            errors="ignore"
        )
    )

    if len(medium_df):
        medium_target_values = np.linspace(
            0.65,
            0.30,
            num=min(
                4,
                len(medium_df)
            )
        )

        available_medium_df = medium_df.copy()

        medium_indices = []

        for target_value in medium_target_values:
            if available_medium_df.empty:
                break

            nearest_index = (
                (
                    available_medium_df[
                        "meteor"
                    ]
                    - target_value
                )
                .abs()
                .idxmin()
            )

            medium_indices.append(
                nearest_index
            )

            available_medium_df = (
                available_medium_df.drop(
                    index=nearest_index
                )
            )

        add_indices(medium_indices)

    # Hard negatives have poor gold similarity but high
    # lexical overlap with the clickbait post or title.
    hard_negative_df = (
        group[
            group["meteor"].lt(0.30)
        ]
        .drop(
            index=selected_indices,
            errors="ignore"
        )
        .sort_values(
            [
                "lexical_overlap",
                "meteor"
            ],
            ascending=[
                False,
                False
            ]
        )
        .head(5)
    )

    add_indices(
        hard_negative_df.index.tolist()
    )

    # Add random low-quality negatives so the model does
    # not learn only from lexically deceptive negatives.
    remaining_negative_df = (
        group[
            group["meteor"].lt(0.30)
        ]
        .drop(
            index=selected_indices,
            errors="ignore"
        )
    )

    remaining_slots = (
        MAX_SELECTED_PER_ARTICLE
        - len(selected_indices)
    )

    if (
        remaining_slots > 0
        and len(remaining_negative_df) > 0
    ):
        sample_size = min(
            remaining_slots,
            len(remaining_negative_df)
        )

        sampled_negative_indices = (
            random_generator.choice(
                remaining_negative_df.index.to_numpy(),
                size=sample_size,
                replace=False
            )
            .tolist()
        )

        add_indices(
            sampled_negative_indices
        )

    # Fill any remaining slots from the highest unselected
    # candidates so article coverage remains consistent.
    remaining_slots = (
        MAX_SELECTED_PER_ARTICLE
        - len(selected_indices)
    )

    if remaining_slots > 0:
        remaining_df = (
            group
            .drop(
                index=selected_indices,
                errors="ignore"
            )
            .sort_values(
                "meteor",
                ascending=False
            )
            .head(remaining_slots)
        )

        add_indices(
            remaining_df.index.tolist()
        )

    for selected_index in selected_indices:
        selected_index_records.append({
            "train_row_number": int(
                train_row_number
            ),
            "source_index": int(
                selected_index
            )
        })


selected_index_df = pd.DataFrame(
    selected_index_records
)

balanced_train_df = (
    train_reranker_pool_df.loc[
        selected_index_df[
            "source_index"
        ].to_numpy()
    ]
    .copy()
    .reset_index(drop=True)
)

# 4. Add candidate relevance classes for diagnostics

balanced_train_df[
    "relevance_band"
] = pd.cut(
    balanced_train_df["meteor"],
    bins=[
        -0.000001,
        0.10,
        0.30,
        0.50,
        0.70,
        0.90,
        1.000001
    ],
    labels=[
        "0.00-0.10",
        "0.10-0.30",
        "0.30-0.50",
        "0.50-0.70",
        "0.70-0.90",
        "0.90-1.00"
    ],
    include_lowest=True
)

# 5. Prepare the full validation candidate pool

val_reranker_ready_df = (
    val_sentence_candidates_df
    .copy()
    .reset_index(drop=True)
)

val_reranker_ready_df[
    "lexical_overlap"
] = [
    lexical_overlap_score(
        candidate_text,
        question,
        title
    )
    for candidate_text, question, title
    in zip(
        val_reranker_ready_df[
            "candidate_text"
        ],
        val_reranker_ready_df[
            "question"
        ],
        val_reranker_ready_df[
            "title"
        ]
    )
]

val_reranker_ready_df[
    "model_text_a"
] = (
    "CLICKBAIT: "
    + val_reranker_ready_df[
        "question"
    ].fillna("").astype(str)
    + " TITLE: "
    + val_reranker_ready_df[
        "title"
    ].fillna("").astype(str)
)

val_reranker_ready_df[
    "model_text_b"
] = (
    "CANDIDATE: "
    + val_reranker_ready_df[
        "candidate_text"
    ].fillna("").astype(str)
    + " DESCRIPTION: "
    + val_reranker_ready_df[
        "description"
    ].fillna("").astype(str)
)

# 6. Save both prepared datasets

balanced_train_df.to_pickle(
    RERANKER_TRAIN_SAMPLE_PATH
)

val_reranker_ready_df.to_pickle(
    RERANKER_VAL_READY_PATH
)

# 7. Integrity checks and report

training_article_counts = (
    balanced_train_df
    .groupby("train_row_number")
    .size()
)

missing_training_articles = (
    set(
        train_sentence_candidates_df[
            "train_row_number"
        ].unique()
    )
    -
    set(
        balanced_train_df[
            "train_row_number"
        ].unique()
    )
)

best_candidate_retained = (
    balanced_train_df
    .groupby("train_row_number")[
        "meteor"
    ]
    .max()
    .sort_index()
    .equals(
        train_sentence_candidates_df
        .groupby("train_row_number")[
            "meteor"
        ]
        .max()
        .sort_index()
    )
)

print(
    "Balanced training candidates:",
    len(balanced_train_df)
)

print(
    "Training articles represented:",
    balanced_train_df[
        "train_row_number"
    ].nunique()
)

print(
    "Missing training articles:",
    len(missing_training_articles)
)

print(
    "Best candidate retained for every article:",
    best_candidate_retained
)

print(
    "\nCandidates per training article:"
)

print(
    training_article_counts.describe()
)

print(
    "\nTraining target distribution:"
)

print(
    balanced_train_df[
        "meteor"
    ].describe()
)

print(
    "\nRelevance-band counts:"
)

print(
    balanced_train_df[
        "relevance_band"
    ].value_counts().sort_index()
)

print(
    "\nSelected variant counts:"
)

print(
    balanced_train_df[
        "variant"
    ].value_counts()
)

print(
    "\nCandidate word-count summary:"
)

print(
    balanced_train_df[
        "word_count"
    ].describe()
)

print(
    "\nFull validation candidates prepared:",
    len(val_reranker_ready_df)
)

print(
    "Validation rows represented:",
    val_reranker_ready_df[
        "row_number"
    ].nunique()
)

print(
    "\nSaved balanced training set:"
)

print(RERANKER_TRAIN_SAMPLE_PATH)

print(
    "\nSaved validation set:"
)

print(RERANKER_VAL_READY_PATH)

print(
    "\nCheckpoint complete:",
    os.path.exists(
        RERANKER_TRAIN_SAMPLE_PATH
    )
    and os.path.exists(
        RERANKER_VAL_READY_PATH
    )
    and len(missing_training_articles) == 0
    and best_candidate_retained
)

Balanced training candidates: 21205
Training articles represented: 1274
Missing training articles: 0
Best candidate retained for every article: True

Candidates per training article:
count    1274.000000
mean       16.644427
std         3.371234
min         1.000000
25%        18.000000
50%        18.000000
75%        18.000000
max        18.000000
dtype: float64

Training target distribution:
count    21205.000000
mean         0.218218
std          0.267385
min          0.000000
25%          0.056818
50%          0.114068
75%          0.225904
max          1.000000
Name: meteor, dtype: float64

Relevance-band counts:
relevance_band
0.00-0.10    9374
0.10-0.30    7548
0.30-0.50    1244
0.50-0.70     747
0.70-0.90    1163
0.90-1.00    1129
Name: count, dtype: int64

Selected variant counts:
variant
sentence                 12557
current_next              6019
previous_current_next     2629
Name: count, dtype: int64

Candidate word-count summary:
count    21205.000000
mean        27.0241

In [32]:
import os
import re
import gc
import json
import math
import time
import random
import shutil
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from datasets import Dataset
from torch.utils.data import DataLoader
from transformers import (
    AutoConfig,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)


# =========================================================
# Configuration
# =========================================================

RANDOM_SEED = 641
MAX_LENGTH = 224
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
GRADIENT_ACCUMULATION_STEPS = 2
NUM_EPOCHS = 2
LEARNING_RATE = 1.5e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.08
MAX_GRAD_NORM = 1.0
DEV_ARTICLE_FRACTION = 0.15

RERANKER_MODEL_DIR = os.path.join(
    WORK_DIR,
    "passage_sentence_cross_encoder_v1"
)

TOKENIZED_TRAIN_DIR = os.path.join(
    WORK_DIR,
    "reranker_tokenized_train_v1"
)

TOKENIZED_DEV_DIR = os.path.join(
    WORK_DIR,
    "reranker_tokenized_internal_dev_v1"
)

TRAINING_HISTORY_PATH = os.path.join(
    WORK_DIR,
    "passage_sentence_cross_encoder_v1_history.json"
)

ARTICLE_SPLIT_PATH = os.path.join(
    WORK_DIR,
    "passage_sentence_cross_encoder_v1_article_split.csv"
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

USE_AMP = (
    DEVICE.type == "cuda"
)

print("Device:", DEVICE)

if USE_AMP:
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(
        RANDOM_SEED
    )

torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True


# 1. Prepare model inputs for the full training pool

full_train_pool_df = (
    train_sentence_candidates_df
    .copy()
    .reset_index(drop=True)
)

if "model_text_a" not in full_train_pool_df.columns:
    full_train_pool_df[
        "model_text_a"
    ] = (
        "CLICKBAIT: "
        + full_train_pool_df[
            "question"
        ].fillna("").astype(str)
        + " TITLE: "
        + full_train_pool_df[
            "title"
        ].fillna("").astype(str)
    )

if "model_text_b" not in full_train_pool_df.columns:
    full_train_pool_df[
        "model_text_b"
    ] = (
        "CANDIDATE: "
        + full_train_pool_df[
            "candidate_text"
        ].fillna("").astype(str)
        + " DESCRIPTION: "
        + full_train_pool_df[
            "description"
        ].fillna("").astype(str)
    )


# 2. Split by article, never by candidate

all_article_ids = np.array(
    sorted(
        balanced_train_df[
            "train_row_number"
        ]
        .astype(int)
        .unique()
    )
)

split_generator = np.random.default_rng(
    RANDOM_SEED
)

shuffled_article_ids = (
    split_generator.permutation(
        all_article_ids
    )
)

number_of_dev_articles = int(
    round(
        len(shuffled_article_ids)
        * DEV_ARTICLE_FRACTION
    )
)

dev_article_ids = set(
    shuffled_article_ids[
        :number_of_dev_articles
    ].tolist()
)

training_article_ids = set(
    shuffled_article_ids[
        number_of_dev_articles:
    ].tolist()
)

training_cross_encoder_df = (
    balanced_train_df[
        balanced_train_df[
            "train_row_number"
        ]
        .astype(int)
        .isin(training_article_ids)
    ]
    .copy()
    .reset_index(drop=True)
)

# Evaluate on every available candidate from held-out articles.
internal_dev_df = (
    full_train_pool_df[
        full_train_pool_df[
            "train_row_number"
        ]
        .astype(int)
        .isin(dev_article_ids)
    ]
    .copy()
    .reset_index(drop=True)
)

article_split_df = pd.DataFrame({
    "train_row_number": all_article_ids
})

article_split_df[
    "split"
] = article_split_df[
    "train_row_number"
].apply(
    lambda row_number:
    "internal_dev"
    if int(row_number) in dev_article_ids
    else "train"
)

article_split_df.to_csv(
    ARTICLE_SPLIT_PATH,
    index=False
)

print("\nArticle split:")
print(
    "Training articles:",
    len(training_article_ids)
)
print(
    "Internal-dev articles:",
    len(dev_article_ids)
)
print(
    "Balanced training candidates:",
    len(training_cross_encoder_df)
)
print(
    "Full internal-dev candidates:",
    len(internal_dev_df)
)

# 3. Candidate training weights

training_cross_encoder_df[
    "sample_weight"
] = (
    1.0
    + 3.0
    * training_cross_encoder_df[
        "meteor"
    ].astype(float)
    + 2.0
    * training_cross_encoder_df[
        "meteor"
    ].ge(0.70).astype(float)
)

training_cross_encoder_df[
    "group_id"
] = (
    training_cross_encoder_df[
        "train_row_number"
    ].astype(int)
)

internal_dev_df[
    "group_id"
] = (
    internal_dev_df[
        "train_row_number"
    ].astype(int)
)

internal_dev_df[
    "sample_weight"
] = 1.0

# 4. Load tokenizer

reranker_tokenizer = (
    AutoTokenizer.from_pretrained(
        ARTICLE_MODEL_DIR,
        local_files_only=True,
        use_fast=True
    )
)


def build_tokenized_dataset(
    dataframe,
    output_directory
):
    source_df = dataframe[
        [
            "model_text_a",
            "model_text_b",
            "meteor",
            "sample_weight",
            "group_id"
        ]
    ].copy()

    source_df = source_df.rename(
        columns={
            "meteor": "labels"
        }
    )

    source_df[
        "labels"
    ] = source_df[
        "labels"
    ].astype(float)

    source_df[
        "sample_weight"
    ] = source_df[
        "sample_weight"
    ].astype(float)

    source_df[
        "group_id"
    ] = source_df[
        "group_id"
    ].astype(int)

    dataset = Dataset.from_pandas(
        source_df,
        preserve_index=False
    )

    def tokenize_batch(batch):
        return reranker_tokenizer(
            batch["model_text_a"],
            batch["model_text_b"],
            truncation="longest_first",
            max_length=MAX_LENGTH,
            padding="max_length"
        )

    tokenized_dataset = dataset.map(
        tokenize_batch,
        batched=True,
        remove_columns=[
            "model_text_a",
            "model_text_b"
        ],
        desc="Tokenizing cross-encoder inputs"
    )

    if os.path.exists(output_directory):
        shutil.rmtree(
            output_directory
        )

    tokenized_dataset.save_to_disk(
        output_directory
    )

    tensor_columns = [
        "input_ids",
        "attention_mask",
        "labels",
        "sample_weight",
        "group_id"
    ]

    if (
        "token_type_ids"
        in tokenized_dataset.column_names
    ):
        tensor_columns.append(
            "token_type_ids"
        )

    tokenized_dataset.set_format(
        type="torch",
        columns=tensor_columns
    )

    return tokenized_dataset


tokenization_start = time.time()

tokenized_train_dataset = (
    build_tokenized_dataset(
        training_cross_encoder_df,
        TOKENIZED_TRAIN_DIR
    )
)

tokenized_dev_dataset = (
    build_tokenized_dataset(
        internal_dev_df,
        TOKENIZED_DEV_DIR
    )
)

print(
    "\nTokenization seconds:",
    round(
        time.time()
        - tokenization_start,
        2
    )
)

print(
    "Tokenized training rows:",
    len(tokenized_train_dataset)
)

print(
    "Tokenized dev rows:",
    len(tokenized_dev_dataset)
)

# 5. DataLoaders

train_generator = torch.Generator()

train_generator.manual_seed(
    RANDOM_SEED
)

train_loader = DataLoader(
    tokenized_train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    generator=train_generator,
    num_workers=2,
    pin_memory=USE_AMP
)

dev_loader = DataLoader(
    tokenized_dev_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=USE_AMP
)

# 6. Initialize regression cross-encoder

reranker_config = AutoConfig.from_pretrained(
    ARTICLE_MODEL_DIR,
    local_files_only=True,
    num_labels=1
)

reranker_config.problem_type = "regression"

reranker_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        ARTICLE_MODEL_DIR,
        config=reranker_config,
        local_files_only=True,
        ignore_mismatched_sizes=True
    )
    .to(DEVICE)
)

optimizer = torch.optim.AdamW(
    reranker_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

updates_per_epoch = math.ceil(
    len(train_loader)
    / GRADIENT_ACCUMULATION_STEPS
)

total_training_updates = (
    updates_per_epoch
    * NUM_EPOCHS
)

warmup_updates = int(
    total_training_updates
    * WARMUP_RATIO
)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_updates,
    num_training_steps=total_training_updates
)

scaler = torch.cuda.amp.GradScaler(
    enabled=USE_AMP
)

print(
    "\nOptimizer updates:",
    total_training_updates
)

print(
    "Warmup updates:",
    warmup_updates
)

# 7. Evaluation function

def evaluate_reranker(
    model,
    data_loader,
    metadata_df
):
    model.eval()

    all_predictions = []
    all_labels = []
    all_group_ids = []

    total_loss = 0.0
    total_examples = 0

    with torch.inference_mode():
        for batch in data_loader:
            labels = (
                batch["labels"]
                .float()
                .to(DEVICE)
            )

            group_ids = (
                batch["group_id"]
                .cpu()
                .numpy()
            )

            model_inputs = {
                key: value.to(
                    DEVICE,
                    non_blocking=True
                )
                for key, value in batch.items()
                if key in {
                    "input_ids",
                    "attention_mask",
                    "token_type_ids"
                }
            }

            with torch.cuda.amp.autocast(
                enabled=USE_AMP
            ):
                logits = model(
                    **model_inputs
                ).logits.squeeze(-1)

                loss = F.smooth_l1_loss(
                    logits,
                    labels,
                    reduction="mean",
                    beta=0.10
                )

            batch_size = labels.size(0)

            total_loss += (
                float(loss.item())
                * batch_size
            )

            total_examples += batch_size

            all_predictions.append(
                logits.detach()
                .cpu()
                .numpy()
            )

            all_labels.append(
                labels.detach()
                .cpu()
                .numpy()
            )

            all_group_ids.append(
                group_ids
            )

    predictions = np.concatenate(
        all_predictions
    )

    labels = np.concatenate(
        all_labels
    )

    group_ids = np.concatenate(
        all_group_ids
    )

    evaluation_df = pd.DataFrame({
        "group_id": group_ids,
        "predicted_score": predictions,
        "meteor": labels
    })

    selected_indices = (
        evaluation_df
        .groupby("group_id")[
            "predicted_score"
        ]
        .idxmax()
    )

    selected_df = (
        evaluation_df.loc[
            selected_indices
        ]
    )

    top1_meteor = float(
        selected_df[
            "meteor"
        ].mean()
    )

    candidate_correlation = float(
        pd.Series(predictions).corr(
            pd.Series(labels),
            method="spearman"
        )
    )

    return {
        "loss": (
            total_loss
            / max(total_examples, 1)
        ),
        "top1_meteor": top1_meteor,
        "candidate_spearman": (
            candidate_correlation
        ),
        "selected_rows": len(
            selected_df
        )
    }

# 8. Training loop

training_history = []

best_dev_top1_meteor = -np.inf
best_epoch = None

training_start = time.time()

optimizer.zero_grad(
    set_to_none=True
)

for epoch_number in range(
    1,
    NUM_EPOCHS + 1
):
    reranker_model.train()

    epoch_loss_sum = 0.0
    epoch_weight_sum = 0.0
    epoch_start = time.time()

    for batch_number, batch in enumerate(
        train_loader,
        start=1
    ):
        labels = (
            batch["labels"]
            .float()
            .to(
                DEVICE,
                non_blocking=True
            )
        )

        sample_weight = (
            batch["sample_weight"]
            .float()
            .to(
                DEVICE,
                non_blocking=True
            )
        )

        model_inputs = {
            key: value.to(
                DEVICE,
                non_blocking=True
            )
            for key, value in batch.items()
            if key in {
                "input_ids",
                "attention_mask",
                "token_type_ids"
            }
        }

        with torch.cuda.amp.autocast(
            enabled=USE_AMP
        ):
            predicted_scores = (
                reranker_model(
                    **model_inputs
                )
                .logits
                .squeeze(-1)
            )

            per_candidate_loss = (
                F.smooth_l1_loss(
                    predicted_scores,
                    labels,
                    reduction="none",
                    beta=0.10
                )
            )

            weighted_loss = (
                (
                    per_candidate_loss
                    * sample_weight
                ).sum()
                /
                sample_weight.sum()
            )

            scaled_loss = (
                weighted_loss
                /
                GRADIENT_ACCUMULATION_STEPS
            )

        scaler.scale(
            scaled_loss
        ).backward()

        epoch_loss_sum += float(
            (
                per_candidate_loss
                * sample_weight
            ).sum().item()
        )

        epoch_weight_sum += float(
            sample_weight.sum().item()
        )

        should_update = (
            batch_number
            % GRADIENT_ACCUMULATION_STEPS
            == 0
            or batch_number
            == len(train_loader)
        )

        if should_update:
            scaler.unscale_(
                optimizer
            )

            torch.nn.utils.clip_grad_norm_(
                reranker_model.parameters(),
                MAX_GRAD_NORM
            )

            scaler.step(
                optimizer
            )

            scaler.update()

            scheduler.step()

            optimizer.zero_grad(
                set_to_none=True
            )

        if (
            batch_number % 200 == 0
            or batch_number
            == len(train_loader)
        ):
            elapsed = (
                time.time()
                - epoch_start
            )

            print(
                f"Epoch {epoch_number} | "
                f"batch {batch_number}/"
                f"{len(train_loader)} | "
                f"{elapsed:.1f}s"
            )

    training_loss = (
        epoch_loss_sum
        / max(
            epoch_weight_sum,
            1.0
        )
    )

    dev_metrics = evaluate_reranker(
        reranker_model,
        dev_loader,
        internal_dev_df
    )

    epoch_record = {
        "epoch": epoch_number,
        "training_loss": float(
            training_loss
        ),
        "dev_loss": float(
            dev_metrics["loss"]
        ),
        "dev_top1_meteor": float(
            dev_metrics[
                "top1_meteor"
            ]
        ),
        "dev_candidate_spearman": float(
            dev_metrics[
                "candidate_spearman"
            ]
        ),
        "epoch_seconds": float(
            time.time()
            - epoch_start
        )
    }

    training_history.append(
        epoch_record
    )

    print(
        f"\nEpoch {epoch_number} summary:"
    )

    print(
        "Training loss:",
        round(
            training_loss,
            6
        )
    )

    print(
        "Dev loss:",
        round(
            dev_metrics["loss"],
            6
        )
    )

    print(
        "Dev article top-1 METEOR:",
        round(
            dev_metrics[
                "top1_meteor"
            ],
            6
        )
    )

    print(
        "Dev candidate Spearman:",
        round(
            dev_metrics[
                "candidate_spearman"
            ],
            6
        )
    )

    if (
        dev_metrics[
            "top1_meteor"
        ]
        >
        best_dev_top1_meteor
    ):
        best_dev_top1_meteor = float(
            dev_metrics[
                "top1_meteor"
            ]
        )

        best_epoch = epoch_number

        if os.path.exists(
            RERANKER_MODEL_DIR
        ):
            shutil.rmtree(
                RERANKER_MODEL_DIR
            )

        reranker_model.save_pretrained(
            RERANKER_MODEL_DIR
        )

        reranker_tokenizer.save_pretrained(
            RERANKER_MODEL_DIR
        )

        print(
            "Saved new best checkpoint:",
            RERANKER_MODEL_DIR
        )


# 9. Save history and release GPU memory

with open(
    TRAINING_HISTORY_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        {
            "configuration": {
                "random_seed":
                    RANDOM_SEED,
                "max_length":
                    MAX_LENGTH,
                "training_articles":
                    len(
                        training_article_ids
                    ),
                "internal_dev_articles":
                    len(dev_article_ids),
                "training_candidates":
                    len(
                        training_cross_encoder_df
                    ),
                "internal_dev_candidates":
                    len(internal_dev_df),
                "epochs":
                    NUM_EPOCHS,
                "learning_rate":
                    LEARNING_RATE,
                "gradient_accumulation_steps":
                    GRADIENT_ACCUMULATION_STEPS
            },
            "best_epoch":
                best_epoch,
            "best_dev_top1_meteor":
                best_dev_top1_meteor,
            "history":
                training_history
        },
        file,
        indent=2
    )

total_seconds = (
    time.time()
    - training_start
)

for variable_name in [
    "reranker_model",
    "optimizer",
    "scheduler",
    "scaler",
    "train_loader",
    "dev_loader"
]:
    globals().pop(
        variable_name,
        None
    )

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\nTraining complete.")
print(
    "Total training seconds:",
    round(
        total_seconds,
        2
    )
)

print(
    "Best epoch:",
    best_epoch
)

print(
    "Best internal-dev top-1 METEOR:",
    round(
        best_dev_top1_meteor,
        6
    )
)

print(
    "Model checkpoint:",
    RERANKER_MODEL_DIR
)

print(
    "History:",
    TRAINING_HISTORY_PATH
)

print(
    "Checkpoint complete:",
    os.path.exists(
        os.path.join(
            RERANKER_MODEL_DIR,
            "model.safetensors"
        )
    )
    and os.path.exists(
        TRAINING_HISTORY_PATH
    )
)

Device: cuda
GPU: Tesla T4

Article split:
Training articles: 1083
Internal-dev articles: 191
Balanced training candidates: 18041
Full internal-dev candidates: 15852


Tokenizing cross-encoder inputs:   0%|          | 0/18041 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/18041 [00:00<?, ? examples/s]

Tokenizing cross-encoder inputs:   0%|          | 0/15852 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/15852 [00:00<?, ? examples/s]

...


Epoch 1 summary:
Training loss: 0.215817
Dev loss: 0.090538
Dev article top-1 METEOR: 0.449332
Dev candidate Spearman: 0.728061


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved new best checkpoint: /kaggle/working/task2_passage_v1/passage_sentence_cross_encoder_v1
Epoch 2 | batch 200/1128 | 37.2s
Epoch 2 | batch 400/1128 | 74.5s
Epoch 2 | batch 600/1128 | 111.8s
Epoch 2 | batch 800/1128 | 149.0s
Epoch 2 | batch 1000/1128 | 186.3s
Epoch 2 | batch 1128/1128 | 210.1s

Epoch 2 summary:
Training loss: 0.138272
Dev loss: 0.068343
Dev article top-1 METEOR: 0.473354
Dev candidate Spearman: 0.726034


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved new best checkpoint: /kaggle/working/task2_passage_v1/passage_sentence_cross_encoder_v1

Training complete.
Total training seconds: 523.01
Best epoch: 2
Best internal-dev top-1 METEOR: 0.473354
Model checkpoint: /kaggle/working/task2_passage_v1/passage_sentence_cross_encoder_v1
History: /kaggle/working/task2_passage_v1/passage_sentence_cross_encoder_v1_history.json
Checkpoint complete: True


## decisive test to evaluate the saved checkpoint on the 168 row validation candidate pool

In [33]:
import os
import gc
import time
import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)


# =========================================================
# Configuration
# =========================================================

RERANKER_MODEL_DIR = os.path.join(
    WORK_DIR,
    "passage_sentence_cross_encoder_v1"
)

VAL_READY_PATH = os.path.join(
    WORK_DIR,
    "val_routed_sentence_reranker_ready.pkl"
)

VAL_SCORED_PATH = os.path.join(
    WORK_DIR,
    "val_passage_cross_encoder_v1_scored.pkl"
)

VAL_SELECTIONS_PATH = os.path.join(
    WORK_DIR,
    "val_passage_cross_encoder_v1_selections.csv"
)

MAX_LENGTH = 224
INFERENCE_BATCH_SIZE = 64

HISTORICAL_PASSAGE_ROUTE_SCORE = 0.387260
BOUNDARY_SENTENCE_SCORE = 0.412168

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", DEVICE)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


# 1. Load validation candidates

if "val_reranker_ready_df" not in globals():
    val_reranker_ready_df = pd.read_pickle(
        VAL_READY_PATH
    )

validation_scored_df = (
    val_reranker_ready_df
    .copy()
    .reset_index(drop=True)
)

print(
    "\nValidation candidates:",
    len(validation_scored_df)
)

print(
    "Validation rows:",
    validation_scored_df[
        "row_number"
    ].nunique()
)

if (
    validation_scored_df[
        "row_number"
    ].nunique()
    != 168
):
    raise RuntimeError(
        "Expected 168 validation groups."
    )


# 2. Load saved cross-encoder

reranker_tokenizer = (
    AutoTokenizer.from_pretrained(
        RERANKER_MODEL_DIR,
        local_files_only=True,
        use_fast=True
    )
)

reranker_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        RERANKER_MODEL_DIR,
        local_files_only=True
    )
    .to(DEVICE)
)

reranker_model.eval()

print(
    "\nModel loaded:",
    os.path.exists(
        os.path.join(
            RERANKER_MODEL_DIR,
            "model.safetensors"
        )
    )
)

# 3. Score all validation candidates

all_candidate_scores = []

inference_start = time.time()

for batch_start in range(
    0,
    len(validation_scored_df),
    INFERENCE_BATCH_SIZE
):
    batch_end = min(
        batch_start + INFERENCE_BATCH_SIZE,
        len(validation_scored_df)
    )

    batch_df = validation_scored_df.iloc[
        batch_start:batch_end
    ]

    encoded = reranker_tokenizer(
        batch_df[
            "model_text_a"
        ].astype(str).tolist(),
        batch_df[
            "model_text_b"
        ].astype(str).tolist(),
        truncation="longest_first",
        max_length=MAX_LENGTH,
        padding=True,
        return_tensors="pt"
    )

    encoded = {
        key: value.to(
            DEVICE,
            non_blocking=True
        )
        for key, value in encoded.items()
    }

    with torch.inference_mode():
        with torch.cuda.amp.autocast(
            enabled=(
                DEVICE.type == "cuda"
            )
        ):
            scores = (
                reranker_model(
                    **encoded
                )
                .logits
                .squeeze(-1)
            )

    all_candidate_scores.append(
        scores
        .detach()
        .cpu()
        .numpy()
    )

    if (
        batch_end % 1024 == 0
        or batch_end
        == len(validation_scored_df)
    ):
        print(
            f"Scored {batch_end}/"
            f"{len(validation_scored_df)} "
            "candidates"
        )


validation_scored_df[
    "cross_encoder_score"
] = np.concatenate(
    all_candidate_scores
)

print(
    "\nInference seconds:",
    round(
        time.time()
        - inference_start,
        2
    )
)

# 4. Select the highest-scoring candidate per article

selected_indices = (
    validation_scored_df
    .groupby("row_number")[
        "cross_encoder_score"
    ]
    .idxmax()
)

validation_selected_df = (
    validation_scored_df.loc[
        selected_indices
    ]
    .copy()
    .sort_values("row_number")
    .reset_index(drop=True)
)

validation_selected_df[
    "candidate_rank_by_model"
] = 1

all_168_score = float(
    validation_selected_df[
        "meteor"
    ].mean()
)

true_passage_selected_df = (
    validation_selected_df[
        validation_selected_df[
            "gold_type"
        ].eq("passage")
    ]
)

misrouted_selected_df = (
    validation_selected_df[
        ~validation_selected_df[
            "gold_type"
        ].eq("passage")
    ]
)

true_passage_score = float(
    true_passage_selected_df[
        "meteor"
    ].mean()
)

misrouted_score = float(
    misrouted_selected_df[
        "meteor"
    ].mean()
)

candidate_spearman = float(
    validation_scored_df[
        "cross_encoder_score"
    ].corr(
        validation_scored_df[
            "meteor"
        ],
        method="spearman"
    )
)

# 5. Ranking diagnostics

validation_scored_df[
    "model_rank"
] = (
    validation_scored_df
    .groupby("row_number")[
        "cross_encoder_score"
    ]
    .rank(
        method="first",
        ascending=False
    )
    .astype(int)
)

validation_scored_df[
    "gold_rank"
] = (
    validation_scored_df
    .groupby("row_number")[
        "meteor"
    ]
    .rank(
        method="first",
        ascending=False
    )
    .astype(int)
)

best_gold_candidates_df = (
    validation_scored_df[
        validation_scored_df[
            "gold_rank"
        ].eq(1)
    ]
)

gold_best_model_ranks = (
    best_gold_candidates_df[
        "model_rank"
    ]
)

top_1_hit_rate = float(
    gold_best_model_ranks.le(1).mean()
)

top_3_hit_rate = float(
    gold_best_model_ranks.le(3).mean()
)

top_5_hit_rate = float(
    gold_best_model_ranks.le(5).mean()
)

top_10_hit_rate = float(
    gold_best_model_ranks.le(10).mean()
)

# 6. Estimate full-validation impact


gain_vs_historical_route = (
    all_168_score
    - HISTORICAL_PASSAGE_ROUTE_SCORE
)

estimated_full_validation_gain = (
    gain_vs_historical_route
    * 168
    / 400
)

estimated_full_validation_score = (
    0.416904
    + estimated_full_validation_gain
)

# 7. Save scored candidates and selections

validation_scored_df.to_pickle(
    VAL_SCORED_PATH
)

validation_selected_df[
    [
        "row_number",
        "article_id",
        "gold_type",
        "question",
        "title",
        "variant",
        "candidate_text",
        "word_count",
        "cross_encoder_score",
        "meteor"
    ]
].to_csv(
    VAL_SELECTIONS_PATH,
    index=False
)

# 8. Report external validation results
print("\nEXTERNAL VALIDATION RESULTS")
print("=" * 55)

print(
    "All 168 routed rows METEOR:",
    round(
        all_168_score,
        6
    )
)

print(
    "True-passage subset METEOR:",
    round(
        true_passage_score,
        6
    )
)

print(
    "Misrouted phrase/multi METEOR:",
    round(
        misrouted_score,
        6
    )
)

print(
    "Candidate-level Spearman:",
    round(
        candidate_spearman,
        6
    )
)

print("\nComparisons:")
print(
    "Historical passage route:",
    HISTORICAL_PASSAGE_ROUTE_SCORE
)

print(
    "Containing-sentence heuristic:",
    BOUNDARY_SENTENCE_SCORE
)

print(
    "Cross-encoder gain vs historical:",
    round(
        gain_vs_historical_route,
        6
    )
)

print(
    "Cross-encoder gain vs sentence heuristic:",
    round(
        all_168_score
        - BOUNDARY_SENTENCE_SCORE,
        6
    )
)

print(
    "Estimated full-validation gain:",
    round(
        estimated_full_validation_gain,
        6
    )
)

print(
    "Estimated full-validation score:",
    round(
        estimated_full_validation_score,
        6
    )
)

print("\nOracle-candidate ranking:")
print(
    "Gold-best candidate ranked top 1:",
    round(
        top_1_hit_rate,
        6
    )
)

print(
    "Gold-best candidate ranked top 3:",
    round(
        top_3_hit_rate,
        6
    )
)

print(
    "Gold-best candidate ranked top 5:",
    round(
        top_5_hit_rate,
        6
    )
)

print(
    "Gold-best candidate ranked top 10:",
    round(
        top_10_hit_rate,
        6
    )
)

print("\nSelected variant counts:")

print(
    validation_selected_df[
        "variant"
    ].value_counts()
)

print("\nSelected word-count summary:")

print(
    validation_selected_df[
        "word_count"
    ].describe()
)

print("\nSaved scored candidates:")
print(VAL_SCORED_PATH)

print("\nSaved selections:")
print(VAL_SELECTIONS_PATH)

print(
    "\nCheckpoint complete:",
    os.path.exists(
        VAL_SCORED_PATH
    )
    and os.path.exists(
        VAL_SELECTIONS_PATH
    )
)

print(
    "\nHighest-scoring selected examples:"
)

display(
    validation_selected_df[
        [
            "row_number",
            "gold_type",
            "candidate_text",
            "cross_encoder_score",
            "meteor",
            "variant",
            "word_count"
        ]
    ]
    .sort_values(
        "cross_encoder_score",
        ascending=False
    )
    .head(15)
)

# 9. Release GPU memory


for variable_name in [
    "reranker_model",
    "encoded",
    "scores",
    "all_candidate_scores"
]:
    globals().pop(
        variable_name,
        None
    )

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(
    "\nGPU memory after cleanup:",
    round(
        torch.cuda.memory_allocated()
        / 1024**3,
        3
    ),
    "GB"
)

Device: cuda
GPU: Tesla T4

Validation candidates: 10384
Validation rows: 168


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Model loaded: True


/tmp/ipykernel_58/1875630583.py:167: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(
/tmp/ipykernel_58/1875630583.py:167: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(
/tmp/ipykernel_58/1875630583.py:167: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(
/tmp/ipykernel_58/1875630583.py:167: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(
/tmp/ipykernel_58/1875630583.py:167: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(
/tmp/ipykernel_58/1875630583.py:167: Fut

Scored 1024/10384 candidates


...

/tmp/ipykernel_58/1875630583.py:167: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(
/tmp/ipykernel_58/1875630583.py:167: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(


Scored 10384/10384 candidates

Inference seconds: 28.06

EXTERNAL VALIDATION RESULTS
All 168 routed rows METEOR: 0.37541
True-passage subset METEOR: 0.425566
Misrouted phrase/multi METEOR: 0.22005
Candidate-level Spearman: 0.462798

Comparisons:
Historical passage route: 0.38726
Containing-sentence heuristic: 0.412168
Cross-encoder gain vs historical: -0.01185
Cross-encoder gain vs sentence heuristic: -0.036758
Estimated full-validation gain: -0.004977
Estimated full-validation score: 0.411927

Oracle-candidate ranking:
Gold-best candidate ranked top 1: 0.190476
Gold-best candidate ranked top 3: 0.422619
Gold-best candidate ranked top 5: 0.565476
Gold-best candidate ranked top 10: 0.744048

Selected variant counts:
variant
sentence                 101
current_next              47
previous_current_next     20
Name: count, dtype: int64

Selected word-count summary:
count    168.000000
mean      34.339286
std       16.531165
min       11.000000
25%       23.000000
50%       31.000000
75% 

/tmp/ipykernel_58/1875630583.py:167: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(
/usr/local/lib/python3.12/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,row_number,gold_type,candidate_text,cross_encoder_score,meteor,variant,word_count
123,299,passage,The Pulitzer Prize has broadened eligibility i...,0.964844,0.887333,sentence,29
162,390,passage,"Sure, you’ve got 16-ish hours away from the jo...",0.951660,0.839955,current_next,53
27,66,passage,A hospital in Argentina is reportedly using as...,0.930176,0.118694,sentence,28
82,213,phrase,The conservative pundit blasted House Republic...,0.924805,0.000000,sentence,41
6,16,phrase,There’s no reason to believe that birds are so...,0.923828,0.000000,current_next,49
129,315,passage,"Juan Gabriel responded saying, ""Art is feminin...",0.916016,0.187793,current_next,25
11,21,passage,"Your Unfortunately, there's not enough solid r...",0.916016,0.684306,previous_current_next,92
149,361,passage,Americans are more likely to approve than disa...,0.915527,0.188172,sentence,44
28,73,passage,The majority of people no longer assume that s...,0.912109,0.169492,current_next,26
110,274,passage,"Michael Jhung, medical officer at the CDC, is ...",0.910156,0.112360,sentence,31



GPU memory after cleanup: 0.483 GB


In [37]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold


GATED_HYBRID_SUMMARY_PATH = os.path.join(
    WORK_DIR,
    "val_cross_encoder_gated_hybrid_summary.csv"
)

GATED_HYBRID_CV_PATH = os.path.join(
    WORK_DIR,
    "val_cross_encoder_gated_hybrid_repeated_cv.csv"
)

GATED_HYBRID_FREQUENCY_PATH = os.path.join(
    WORK_DIR,
    "val_cross_encoder_gated_hybrid_rule_frequency.csv"
)

CONTAINING_SENTENCE_SCORE = 0.412168

# 1. Row-level containing-sentence results

containing_sentence_df = (
    boundary_results_df[
        boundary_results_df["rule"].eq(
            "article_rank1_containing_sentence"
        )
    ][
        [
            "row_number",
            "gold_type",
            "prediction",
            "meteor"
        ]
    ]
    .rename(
        columns={
            "prediction":
                "containing_sentence_prediction",
            "meteor":
                "containing_sentence_meteor"
        }
    )
    .copy()
)

# 2. Row-level cross-encoder selections

cross_encoder_selection_df = (
    validation_selected_df[
        [
            "row_number",
            "candidate_text",
            "meteor",
            "cross_encoder_score",
            "variant",
            "word_count"
        ]
    ]
    .rename(
        columns={
            "candidate_text":
                "cross_encoder_prediction",
            "meteor":
                "cross_encoder_meteor",
            "word_count":
                "cross_encoder_word_count"
        }
    )
    .copy()
)

# 3. Calculate top-1 versus top-2 score margin

ranked_cross_encoder_df = (
    validation_scored_df
    .sort_values(
        [
            "row_number",
            "cross_encoder_score"
        ],
        ascending=[
            True,
            False
        ]
    )
    .copy()
)

ranked_cross_encoder_df[
    "score_rank"
] = (
    ranked_cross_encoder_df
    .groupby("row_number")
    .cumcount()
    + 1
)

top_two_score_df = (
    ranked_cross_encoder_df[
        ranked_cross_encoder_df[
            "score_rank"
        ].le(2)
    ]
    .pivot(
        index="row_number",
        columns="score_rank",
        values="cross_encoder_score"
    )
    .rename(
        columns={
            1: "cross_encoder_top1_score",
            2: "cross_encoder_top2_score"
        }
    )
    .reset_index()
)

top_two_score_df[
    "cross_encoder_score_margin"
] = (
    top_two_score_df[
        "cross_encoder_top1_score"
    ]
    -
    top_two_score_df[
        "cross_encoder_top2_score"
    ]
)

# 4. Merge deployable confidence features


hybrid_row_df = (
    containing_sentence_df
    .merge(
        cross_encoder_selection_df,
        on="row_number",
        how="inner",
        validate="one_to_one"
    )
    .merge(
        top_two_score_df,
        on="row_number",
        how="inner",
        validate="one_to_one"
    )
    .merge(
        val_type_predictions_df[
            [
                "row_number",
                "prob_phrase",
                "prob_passage",
                "prob_multi",
                "confidence"
            ]
        ],
        on="row_number",
        how="inner",
        validate="one_to_one"
    )
    .sort_values("row_number")
    .reset_index(drop=True)
)

if len(hybrid_row_df) != 168:
    raise RuntimeError(
        "Expected 168 routed validation rows, "
        f"but found {len(hybrid_row_df)}."
    )

# 5. Report the available switching ceiling

hybrid_row_df[
    "cross_encoder_better"
] = (
    hybrid_row_df[
        "cross_encoder_meteor"
    ]
    >
    hybrid_row_df[
        "containing_sentence_meteor"
    ]
)

hybrid_row_df[
    "oracle_switch_meteor"
] = hybrid_row_df[
    [
        "cross_encoder_meteor",
        "containing_sentence_meteor"
    ]
].max(axis=1)

print("Row-level comparison:")
print(
    "Containing-sentence score:",
    round(
        hybrid_row_df[
            "containing_sentence_meteor"
        ].mean(),
        6
    )
)

print(
    "Cross-encoder score:",
    round(
        hybrid_row_df[
            "cross_encoder_meteor"
        ].mean(),
        6
    )
)

print(
    "Oracle switch score:",
    round(
        hybrid_row_df[
            "oracle_switch_meteor"
        ].mean(),
        6
    )
)

print(
    "Rows where cross-encoder is better:",
    int(
        hybrid_row_df[
            "cross_encoder_better"
        ].sum()
    ),
    "/ 168"
)

# 6. Create a limited interpretable gating grid

passage_probability_thresholds = [
    0.50,
    0.60,
    0.70,
    0.80,
    0.90
]

top1_score_thresholds = [
    -np.inf,
    0.60,
    0.75,
    0.85
]

score_margin_thresholds = [
    -np.inf,
    0.02,
    0.05,
    0.10
]

rule_records = []

# Containing sentence only.
rule_records.append({
    "rule": "containing_sentence_only",
    "minimum_prob_passage": np.inf,
    "minimum_top1_score": np.inf,
    "minimum_score_margin": np.inf
})

# Cross-encoder only, included as a reference.
rule_records.append({
    "rule": "cross_encoder_only",
    "minimum_prob_passage": -np.inf,
    "minimum_top1_score": -np.inf,
    "minimum_score_margin": -np.inf
})

for prob_threshold in passage_probability_thresholds:
    for score_threshold in top1_score_thresholds:
        for margin_threshold in score_margin_thresholds:
            rule_records.append({
                "rule": (
                    f"prob{prob_threshold:.2f}"
                    f"_score{score_threshold}"
                    f"_margin{margin_threshold}"
                ),
                "minimum_prob_passage":
                    prob_threshold,
                "minimum_top1_score":
                    score_threshold,
                "minimum_score_margin":
                    margin_threshold
            })


# 7. Build row × rule score matrix

rule_score_dictionary = {}
rule_usage_dictionary = {}

for rule_config in rule_records:
    rule_name = rule_config["rule"]

    if rule_name == "containing_sentence_only":
        use_cross_encoder = pd.Series(
            False,
            index=hybrid_row_df.index
        )

    elif rule_name == "cross_encoder_only":
        use_cross_encoder = pd.Series(
            True,
            index=hybrid_row_df.index
        )

    else:
        use_cross_encoder = (
            hybrid_row_df[
                "prob_passage"
            ].ge(
                rule_config[
                    "minimum_prob_passage"
                ]
            )
            &
            hybrid_row_df[
                "cross_encoder_top1_score"
            ].ge(
                rule_config[
                    "minimum_top1_score"
                ]
            )
            &
            hybrid_row_df[
                "cross_encoder_score_margin"
            ].ge(
                rule_config[
                    "minimum_score_margin"
                ]
            )
        )

    hybrid_scores = np.where(
        use_cross_encoder,
        hybrid_row_df[
            "cross_encoder_meteor"
        ],
        hybrid_row_df[
            "containing_sentence_meteor"
        ]
    )

    rule_score_dictionary[
        rule_name
    ] = hybrid_scores

    rule_usage_dictionary[
        rule_name
    ] = use_cross_encoder.to_numpy()


rule_score_matrix = pd.DataFrame(
    rule_score_dictionary,
    index=hybrid_row_df[
        "row_number"
    ].astype(int)
)

rule_usage_matrix = pd.DataFrame(
    rule_usage_dictionary,
    index=hybrid_row_df[
        "row_number"
    ].astype(int)
)


# 8. Full-validation diagnostic table
full_summary_records = []

for rule_config in rule_records:
    rule_name = rule_config["rule"]

    scores = rule_score_matrix[
        rule_name
    ]

    usage = rule_usage_matrix[
        rule_name
    ]

    true_passage_mask = (
        hybrid_row_df[
            "gold_type"
        ].eq("passage")
        .to_numpy()
    )

    full_summary_records.append({
        "rule": rule_name,
        "minimum_prob_passage":
            rule_config[
                "minimum_prob_passage"
            ],
        "minimum_top1_score":
            rule_config[
                "minimum_top1_score"
            ],
        "minimum_score_margin":
            rule_config[
                "minimum_score_margin"
            ],
        "all_168_meteor":
            float(scores.mean()),
        "true_passage_meteor":
            float(
                scores.to_numpy()[
                    true_passage_mask
                ].mean()
            ),
        "misrouted_meteor":
            float(
                scores.to_numpy()[
                    ~true_passage_mask
                ].mean()
            ),
        "cross_encoder_replacements":
            int(usage.sum()),
        "containing_sentence_fallbacks":
            int((~usage).sum())
    })


full_summary_df = (
    pd.DataFrame(
        full_summary_records
    )
    .sort_values(
        "all_168_meteor",
        ascending=False
    )
    .reset_index(drop=True)
)

full_summary_df.to_csv(
    GATED_HYBRID_SUMMARY_PATH,
    index=False
)

# 9. Repeated held-out rule selection


SPLIT_SEEDS = [
    17,
    83,
    641,
    2026,
    9917,
    44,
    112,
    333,
    777,
    1234
]

row_metadata_df = (
    hybrid_row_df[
        [
            "row_number",
            "gold_type"
        ]
    ]
    .set_index("row_number")
    .loc[
        rule_score_matrix.index
    ]
)

seed_summary_records = []
fold_selection_records = []

for split_seed in SPLIT_SEEDS:
    splitter = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=split_seed
    )

    heldout_scores = pd.Series(
        np.nan,
        index=rule_score_matrix.index,
        dtype=float
    )

    print(f"\nSeed {split_seed}")

    for fold_number, (
        training_positions,
        validation_positions
    ) in enumerate(
        splitter.split(
            rule_score_matrix,
            row_metadata_df[
                "gold_type"
            ]
        ),
        start=1
    ):
        training_rows = (
            rule_score_matrix.index[
                training_positions
            ]
        )

        validation_rows = (
            rule_score_matrix.index[
                validation_positions
            ]
        )

        training_rule_means = (
            rule_score_matrix.loc[
                training_rows
            ].mean(axis=0)
        )

        selected_rule = (
            training_rule_means.idxmax()
        )

        selected_heldout_scores = (
            rule_score_matrix.loc[
                validation_rows,
                selected_rule
            ]
        )

        heldout_scores.loc[
            validation_rows
        ] = selected_heldout_scores

        fold_selection_records.append({
            "seed": split_seed,
            "fold": fold_number,
            "selected_rule": selected_rule,
            "training_score": float(
                training_rule_means[
                    selected_rule
                ]
            ),
            "heldout_score": float(
                selected_heldout_scores.mean()
            ),
            "heldout_rows": len(
                validation_rows
            )
        })

        print(
            f"  Fold {fold_number}: "
            f"{selected_rule} | "
            f"held-out="
            f"{selected_heldout_scores.mean():.6f}"
        )

    if heldout_scores.isna().any():
        raise RuntimeError(
            f"Incomplete OOF scores for seed "
            f"{split_seed}."
        )

    seed_score = float(
        heldout_scores.mean()
    )

    seed_summary_records.append({
        "seed": split_seed,
        "heldout_all_168_meteor":
            seed_score,
        "gain_vs_containing_sentence":
            seed_score
            - CONTAINING_SENTENCE_SCORE,
        "estimated_full_validation_gain":
            (
                seed_score
                - CONTAINING_SENTENCE_SCORE
            )
            * 168
            / 400
    })


repeated_cv_df = pd.DataFrame(
    seed_summary_records
)

fold_selection_df = pd.DataFrame(
    fold_selection_records
)

rule_frequency_df = (
    fold_selection_df[
        "selected_rule"
    ]
    .value_counts()
    .rename_axis("rule")
    .reset_index(
        name="selected_folds"
    )
)

rule_frequency_df[
    "selection_rate"
] = (
    rule_frequency_df[
        "selected_folds"
    ]
    / len(fold_selection_df)
)

repeated_cv_df.to_csv(
    GATED_HYBRID_CV_PATH,
    index=False
)

rule_frequency_df.to_csv(
    GATED_HYBRID_FREQUENCY_PATH,
    index=False
)

# 10. Final report


print("\nTop full-validation hybrid rules:")

display(
    full_summary_df.head(15)
)

print("\nRepeated held-out results:")

display(
    repeated_cv_df
)

print("\nRule-selection frequency:")

display(
    rule_frequency_df.head(15)
)

print("\nRepeated-CV summary:")

print(
    "Mean held-out score:",
    round(
        repeated_cv_df[
            "heldout_all_168_meteor"
        ].mean(),
        6
    )
)

print(
    "Minimum held-out score:",
    round(
        repeated_cv_df[
            "heldout_all_168_meteor"
        ].min(),
        6
    )
)

print(
    "Mean gain versus containing sentence:",
    round(
        repeated_cv_df[
            "gain_vs_containing_sentence"
        ].mean(),
        6
    )
)

print(
    "Estimated full-validation gain:",
    round(
        repeated_cv_df[
            "estimated_full_validation_gain"
        ].mean(),
        6
    )
)

print(
    "\nCheckpoint complete:",
    os.path.exists(
        GATED_HYBRID_SUMMARY_PATH
    )
    and os.path.exists(
        GATED_HYBRID_CV_PATH
    )
    and os.path.exists(
        GATED_HYBRID_FREQUENCY_PATH
    )
)

Row-level comparison:
Containing-sentence score: 0.412168
Cross-encoder score: 0.37541
Oracle switch score: 0.503349
Rows where cross-encoder is better: 76 / 168

Seed 17
  Fold 1: prob0.60_score0.75_margin0.1 | held-out=0.450783
  Fold 2: prob0.60_score0.75_margin0.1 | held-out=0.340015
  Fold 3: prob0.80_score0.75_margin-inf | held-out=0.384083
  Fold 4: prob0.60_score0.75_margin0.1 | held-out=0.421743
  Fold 5: prob0.80_score0.75_margin-inf | held-out=0.445643

Seed 83
  Fold 1: prob0.60_score0.75_margin0.1 | held-out=0.360490
  Fold 2: prob0.60_score0.75_margin0.1 | held-out=0.402570
  Fold 3: prob0.60_score0.75_margin0.1 | held-out=0.393419
  Fold 4: prob0.80_score0.85_margin-inf | held-out=0.461305
  Fold 5: prob0.80_score0.6_margin-inf | held-out=0.423618

Seed 641
  Fold 1: prob0.60_score-inf_margin0.1 | held-out=0.508651
  Fold 2: prob0.60_score0.75_margin0.1 | held-out=0.434052
  Fold 3: prob0.60_score0.75_margin0.1 | held-out=0.479844
  Fold 4: prob0.60_score0.75_margin0.1 |

,rule,minimum_prob_passage,minimum_top1_score,minimum_score_margin,all_168_meteor,true_passage_meteor,misrouted_meteor,cross_encoder_replacements,containing_sentence_fallbacks
0,prob0.60_score0.75_margin0.1,0.6,0.75,0.1,0.428958,0.443129,0.385062,13,155
1,prob0.60_score-inf_margin0.1,0.6,-inf,0.1,0.427819,0.441550,0.385286,24,144
2,prob0.60_score0.6_margin0.1,0.6,0.60,0.1,0.427819,0.441550,0.385286,20,148
3,prob0.80_score0.75_margin-inf,0.8,0.75,-inf,0.424845,0.443872,0.365909,42,126
4,prob0.80_score0.6_margin-inf,0.8,0.60,-inf,0.424734,0.443725,0.365909,53,115
5,prob0.70_score0.6_margin0.1,0.7,0.60,0.1,0.424619,0.437317,0.385286,17,151
6,prob0.70_score-inf_margin0.1,0.7,-inf,0.1,0.424619,0.437317,0.385286,18,150
7,prob0.70_score0.75_margin0.1,0.7,0.75,0.1,0.424592,0.437354,0.385062,11,157
8,prob0.80_score0.75_margin0.1,0.8,0.75,0.1,0.423931,0.437354,0.382352,8,160
9,prob0.80_score0.6_margin0.1,0.8,0.60,0.1,0.423903,0.437317,0.382352,12,156



Repeated held-out results:


,seed,heldout_all_168_meteor,gain_vs_containing_sentence,estimated_full_validation_gain
0,17,0.408153,-0.004015,-0.001686
1,83,0.407873,-0.004295,-0.001804
2,641,0.422064,0.009896,0.004156
3,2026,0.418556,0.006388,0.002683
4,9917,0.413830,0.001662,0.000698
5,44,0.420867,0.008699,0.003654
6,112,0.415905,0.003737,0.001570
7,333,0.409815,-0.002353,-0.000988
8,777,0.405073,-0.007095,-0.002980
9,1234,0.422710,0.010542,0.004428



Rule-selection frequency:


,rule,selected_folds,selection_rate
0,prob0.60_score0.75_margin0.1,29,0.58
1,prob0.80_score0.75_margin-inf,8,0.16
2,prob0.60_score-inf_margin0.1,5,0.10
3,prob0.80_score0.6_margin-inf,4,0.08
4,prob0.80_score0.85_margin-inf,2,0.04
5,prob0.70_score0.75_margin0.1,1,0.02
6,prob0.70_score0.6_margin0.1,1,0.02



Repeated-CV summary:
Mean held-out score: 0.414485
Minimum held-out score: 0.405073
Mean gain versus containing sentence: 0.002317
Estimated full-validation gain: 0.000973

Checkpoint complete: True


In [38]:
import os
import time
import pandas as pd


TEST_RERANKER_READY_PATH = os.path.join(
    WORK_DIR,
    "test_routed_sentence_reranker_ready.pkl"
)

# 1. Identify test rows routed as passage

test_passage_rows = (
    test_type_predictions_df.loc[
        test_type_predictions_df[
            "predicted_type"
        ].astype(str).eq("passage"),
        "row_number"
    ]
    .astype(int)
    .sort_values()
    .tolist()
)

print(
    "Passage-routed test rows:",
    len(test_passage_rows)
)

if len(test_passage_rows) != 183:
    raise RuntimeError(
        "Expected 183 passage-routed test rows, "
        f"but found {len(test_passage_rows)}."
    )


# 2. Generate the same candidate variants used in training

test_candidate_records = []

generation_start = time.time()

for processed_number, row_number in enumerate(
    test_passage_rows,
    start=1
):
    row = test_df.iloc[
        row_number
    ]

    question = normalize_whitespace(
        row.get("postText", "")
    )

    title = normalize_whitespace(
        row.get("targetTitle", "")
    )

    description = normalize_whitespace(
        row.get(
            "targetDescription",
            ""
        )
    )

    candidate_dictionary = (
        build_sentence_candidate_dictionary(
            row.get(
                "targetParagraphs",
                []
            )
        )
    )

    if not candidate_dictionary:
        raise RuntimeError(
            f"No sentence candidates generated "
            f"for test row {row_number}."
        )

    for candidate in candidate_dictionary.values():
        candidate_text = candidate[
            "candidate_text"
        ]

        test_candidate_records.append({
            "row_number": int(
                row_number
            ),
            "article_id": row["id"],
            "question": question,
            "title": title,
            "description": description,
            "paragraph_index": int(
                candidate[
                    "paragraph_index"
                ]
            ),
            "sentence_index": int(
                candidate[
                    "sentence_index"
                ]
            ),
            "variant": candidate[
                "variant"
            ],
            "candidate_text": candidate_text,
            "word_count": len(
                candidate_text.split()
            )
        })

    if (
        processed_number % 40 == 0
        or processed_number
        == len(test_passage_rows)
    ):
        print(
            f"Processed {processed_number}/"
            f"{len(test_passage_rows)} rows"
        )

# 3. Prepare the exact cross-encoder input format

test_reranker_ready_df = pd.DataFrame(
    test_candidate_records
)

test_reranker_ready_df[
    "candidate_row_id"
] = range(
    len(test_reranker_ready_df)
)

test_reranker_ready_df[
    "model_text_a"
] = (
    "CLICKBAIT: "
    + test_reranker_ready_df[
        "question"
    ].fillna("").astype(str)
    + " TITLE: "
    + test_reranker_ready_df[
        "title"
    ].fillna("").astype(str)
)

test_reranker_ready_df[
    "model_text_b"
] = (
    "CANDIDATE: "
    + test_reranker_ready_df[
        "candidate_text"
    ].fillna("").astype(str)
    + " DESCRIPTION: "
    + test_reranker_ready_df[
        "description"
    ].fillna("").astype(str)
)

# 4. Validate coverage and save

candidate_counts = (
    test_reranker_ready_df
    .groupby("row_number")
    .size()
)

covered_rows = set(
    test_reranker_ready_df[
        "row_number"
    ].astype(int)
)

missing_rows = sorted(
    set(test_passage_rows)
    - covered_rows
)

unexpected_rows = sorted(
    covered_rows
    - set(test_passage_rows)
)

duplicate_candidates = int(
    test_reranker_ready_df.duplicated(
        subset=[
            "row_number",
            "candidate_text"
        ]
    ).sum()
)

if missing_rows or unexpected_rows:
    raise RuntimeError(
        "Test candidate row coverage is incorrect."
    )

test_reranker_ready_df.to_pickle(
    TEST_RERANKER_READY_PATH
)

# 5. Report

print(
    "\nGeneration seconds:",
    round(
        time.time()
        - generation_start,
        2
    )
)

print(
    "Total test candidates:",
    len(test_reranker_ready_df)
)

print(
    "Test rows represented:",
    test_reranker_ready_df[
        "row_number"
    ].nunique()
)

print(
    "Missing rows:",
    len(missing_rows)
)

print(
    "Unexpected rows:",
    len(unexpected_rows)
)

print(
    "Duplicate row-text candidates:",
    duplicate_candidates
)

print(
    "\nCandidates per article:"
)

print(
    candidate_counts.describe()
)

print(
    "\nVariant counts:"
)

print(
    test_reranker_ready_df[
        "variant"
    ].value_counts()
)

print(
    "\nCandidate word-count summary:"
)

print(
    test_reranker_ready_df[
        "word_count"
    ].describe()
)

print(
    "\nSaved test candidate pool:"
)

print(TEST_RERANKER_READY_PATH)

print(
    "\nCheckpoint complete:",
    os.path.exists(
        TEST_RERANKER_READY_PATH
    )
    and len(missing_rows) == 0
    and len(unexpected_rows) == 0
)

Passage-routed test rows: 183
Processed 40/183 rows
Processed 80/183 rows
Processed 120/183 rows
Processed 160/183 rows
Processed 183/183 rows

Generation seconds: 1.13
Total test candidates: 9883
Test rows represented: 183
Missing rows: 0
Unexpected rows: 0
Duplicate row-text candidates: 0

Candidates per article:
count    183.000000
mean      54.005464
std       64.142569
min        2.000000
25%       18.000000
50%       34.000000
75%       62.500000
max      503.000000
dtype: float64

Variant counts:
variant
sentence                 5426
current_next             2918
previous_current_next    1539
Name: count, dtype: int64

Candidate word-count summary:
count    9883.000000
mean       27.929576
std        19.264972
min         1.000000
25%        14.000000
50%        24.000000
75%        38.000000
max       152.000000
Name: word_count, dtype: float64

Saved test candidate pool:
/kaggle/working/task2_passage_v1/test_routed_sentence_reranker_ready.pkl

Checkpoint complete: True


In [39]:
import os
import gc
import time
import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)


RERANKER_MODEL_DIR = os.path.join(
    WORK_DIR,
    "passage_sentence_cross_encoder_v1"
)

TEST_READY_PATH = os.path.join(
    WORK_DIR,
    "test_routed_sentence_reranker_ready.pkl"
)

TEST_SCORED_PATH = os.path.join(
    WORK_DIR,
    "test_passage_cross_encoder_v1_scored.pkl"
)

TEST_TOP1_PATH = os.path.join(
    WORK_DIR,
    "test_passage_cross_encoder_v1_top1.csv"
)

MAX_LENGTH = 224
INFERENCE_BATCH_SIZE = 64

MINIMUM_PASSAGE_PROBABILITY = 0.60
MINIMUM_TOP1_SCORE = 0.75
MINIMUM_SCORE_MARGIN = 0.10

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

# 1. Load candidates and model

if "test_reranker_ready_df" not in globals():
    test_reranker_ready_df = pd.read_pickle(
        TEST_READY_PATH
    )

test_cross_encoder_df = (
    test_reranker_ready_df
    .copy()
    .reset_index(drop=True)
)

reranker_tokenizer = (
    AutoTokenizer.from_pretrained(
        RERANKER_MODEL_DIR,
        local_files_only=True,
        use_fast=True
    )
)

reranker_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        RERANKER_MODEL_DIR,
        local_files_only=True
    )
    .to(DEVICE)
)

reranker_model.eval()

print(
    "\nCandidates:",
    len(test_cross_encoder_df)
)

print(
    "Passage-routed rows:",
    test_cross_encoder_df[
        "row_number"
    ].nunique()
)

# 2. Score every candidate

candidate_scores = []

inference_start = time.time()

for batch_start in range(
    0,
    len(test_cross_encoder_df),
    INFERENCE_BATCH_SIZE
):
    batch_end = min(
        batch_start + INFERENCE_BATCH_SIZE,
        len(test_cross_encoder_df)
    )

    batch_df = test_cross_encoder_df.iloc[
        batch_start:batch_end
    ]

    encoded = reranker_tokenizer(
        batch_df[
            "model_text_a"
        ].astype(str).tolist(),
        batch_df[
            "model_text_b"
        ].astype(str).tolist(),
        truncation="longest_first",
        max_length=MAX_LENGTH,
        padding=True,
        return_tensors="pt"
    )

    encoded = {
        key: value.to(
            DEVICE,
            non_blocking=True
        )
        for key, value in encoded.items()
    }

    with torch.inference_mode():
        with torch.amp.autocast(
            device_type=DEVICE.type,
            enabled=(
                DEVICE.type == "cuda"
            )
        ):
            scores = (
                reranker_model(
                    **encoded
                )
                .logits
                .squeeze(-1)
            )

    candidate_scores.append(
        scores
        .detach()
        .float()
        .cpu()
        .numpy()
    )

    if (
        batch_end % 1024 == 0
        or batch_end
        == len(test_cross_encoder_df)
    ):
        print(
            f"Scored {batch_end}/"
            f"{len(test_cross_encoder_df)} "
            "candidates"
        )


test_cross_encoder_df[
    "cross_encoder_score"
] = (
    np.concatenate(
        candidate_scores
    )
    .astype(np.float32)
)

print(
    "\nInference seconds:",
    round(
        time.time()
        - inference_start,
        2
    )
)

# 3. Rank candidates within each article

test_cross_encoder_df = (
    test_cross_encoder_df
    .sort_values(
        [
            "row_number",
            "cross_encoder_score"
        ],
        ascending=[
            True,
            False
        ]
    )
    .reset_index(drop=True)
)

test_cross_encoder_df[
    "model_rank"
] = (
    test_cross_encoder_df
    .groupby("row_number")
    .cumcount()
    + 1
)

top_two_df = (
    test_cross_encoder_df[
        test_cross_encoder_df[
            "model_rank"
        ].le(2)
    ][
        [
            "row_number",
            "model_rank",
            "cross_encoder_score"
        ]
    ]
    .pivot(
        index="row_number",
        columns="model_rank",
        values="cross_encoder_score"
    )
    .rename(
        columns={
            1: "top1_score",
            2: "top2_score"
        }
    )
    .reset_index()
)

top_two_df[
    "score_margin"
] = (
    top_two_df["top1_score"]
    - top_two_df["top2_score"]
)

top1_selection_df = (
    test_cross_encoder_df[
        test_cross_encoder_df[
            "model_rank"
        ].eq(1)
    ]
    .copy()
    .merge(
        top_two_df[
            [
                "row_number",
                "top2_score",
                "score_margin"
            ]
        ],
        on="row_number",
        how="left",
        validate="one_to_one"
    )
)


# 4. Add type-classifier probabilities and baseline text

top1_selection_df = (
    top1_selection_df
    .merge(
        test_type_predictions_df[
            [
                "row_number",
                "predicted_type",
                "prob_phrase",
                "prob_passage",
                "prob_multi"
            ]
        ],
        on="row_number",
        how="left",
        validate="one_to_one"
    )
)

baseline_prediction_column = (
    "spoiler"
)

top1_selection_df[
    "baseline_prediction"
] = (
    top1_selection_df[
        "row_number"
    ].map(
        baseline_submission_df[
            baseline_prediction_column
        ].astype(str)
    )
)


# 5. Apply the fixed conservative gate

top1_selection_df[
    "passes_conservative_gate"
] = (
    top1_selection_df[
        "prob_passage"
    ].ge(
        MINIMUM_PASSAGE_PROBABILITY
    )
    &
    top1_selection_df[
        "cross_encoder_score"
    ].ge(
        MINIMUM_TOP1_SCORE
    )
    &
    top1_selection_df[
        "score_margin"
    ].ge(
        MINIMUM_SCORE_MARGIN
    )
)

gated_test_df = (
    top1_selection_df[
        top1_selection_df[
            "passes_conservative_gate"
        ]
    ]
    .copy()
    .sort_values(
        "cross_encoder_score",
        ascending=False
    )
    .reset_index(drop=True)
)

gated_test_df[
    "candidate_differs_from_baseline"
] = (
    gated_test_df[
        "candidate_text"
    ].astype(str)
    !=
    gated_test_df[
        "baseline_prediction"
    ].astype(str)
)

gated_test_df[
    "baseline_word_count"
] = (
    gated_test_df[
        "baseline_prediction"
    ]
    .astype(str)
    .str.split()
    .str.len()
)

gated_test_df[
    "candidate_word_count"
] = (
    gated_test_df[
        "candidate_text"
    ]
    .astype(str)
    .str.split()
    .str.len()
)


# 6. Save scoring checkpoints

test_cross_encoder_df.to_pickle(
    TEST_SCORED_PATH
)

top1_selection_df.to_csv(
    TEST_TOP1_PATH,
    index=False
)

# 7. Report exactly what the gate would change

print("\nCONSERVATIVE TEST GATE")
print("=" * 55)

print(
    "Passage-routed rows evaluated:",
    len(top1_selection_df)
)

print(
    "Rows passing gate:",
    len(gated_test_df)
)

print(
    "Passing rows different from baseline:",
    int(
        gated_test_df[
            "candidate_differs_from_baseline"
        ].sum()
    )
)

print(
    "Passing rows identical to baseline:",
    int(
        (
            ~gated_test_df[
                "candidate_differs_from_baseline"
            ]
        ).sum()
    )
)

print("\nGate thresholds:")
print(
    "Minimum passage probability:",
    MINIMUM_PASSAGE_PROBABILITY
)
print(
    "Minimum top-1 score:",
    MINIMUM_TOP1_SCORE
)
print(
    "Minimum top1-top2 margin:",
    MINIMUM_SCORE_MARGIN
)

print("\nPassing-row score summary:")

print(
    gated_test_df[
        [
            "prob_passage",
            "cross_encoder_score",
            "score_margin",
            "candidate_word_count"
        ]
    ].describe()
)

print("\nSaved scored candidates:")
print(TEST_SCORED_PATH)

print("\nSaved top-1 selections:")
print(TEST_TOP1_PATH)

print(
    "\nCheckpoint complete:",
    os.path.exists(
        TEST_SCORED_PATH
    )
    and os.path.exists(
        TEST_TOP1_PATH
    )
)

print(
    "\nRows that would replace the frozen baseline:"
)

display(
    gated_test_df[
        gated_test_df[
            "candidate_differs_from_baseline"
        ]
    ][
        [
            "row_number",
            "prob_passage",
            "cross_encoder_score",
            "score_margin",
            "variant",
            "baseline_prediction",
            "candidate_text",
            "baseline_word_count",
            "candidate_word_count"
        ]
    ]
)

Device: cuda
GPU: Tesla T4


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Candidates: 9883
Passage-routed rows: 183
Scored 1024/9883 candidates
Scored 2048/9883 candidates
Scored 3072/9883 candidates
Scored 4096/9883 candidates
Scored 5120/9883 candidates
Scored 6144/9883 candidates
Scored 7168/9883 candidates
Scored 8192/9883 candidates
Scored 9216/9883 candidates
Scored 9883/9883 candidates

Inference seconds: 26.83

CONSERVATIVE TEST GATE
Passage-routed rows evaluated: 183
Rows passing gate: 24
Passing rows different from baseline: 21
Passing rows identical to baseline: 3

Gate thresholds:
Minimum passage probability: 0.6
Minimum top-1 score: 0.75
Minimum top1-top2 margin: 0.1

Passing-row score summary:
       prob_passage  cross_encoder_score  score_margin  candidate_word_count
count     24.000000            24.000000     24.000000             24.000000
mean       0.789849             0.847066      0.370720             30.291667
std        0.077721             0.052609      0.190915             11.195803
min        0.643515             0.769043      0.

,row_number,prob_passage,cross_encoder_score,score_margin,variant,baseline_prediction,candidate_text,baseline_word_count,candidate_word_count
0,154,0.704557,0.956543,0.693848,sentence,MIT researchers can read a book without openin...,Researchers from the Massachusetts Institute o...,35,25
1,223,0.817307,0.946289,0.247070,sentence,Police: Four Women Arrested for Spraying Anti-...,Police arrested four women in North Carolina f...,36,24
2,332,0.879119,0.932617,0.665527,sentence,"Hizbullah, an Iran-backed Lebanese Shia group,...",(AP) – The evacuation of civilians and opposit...,91,37
3,83,0.681752,0.914551,0.418701,sentence,she had plastic surgery years ago to make her ...,Julie Chen revealed on Wednesday that she had ...,57,22
4,93,0.643515,0.895020,0.242676,sentence,The 1937 Bugatti Type 57S was found in the gar...,The 1937 Bugatti Type 57S was found in the gar...,50,29
5,103,0.699235,0.869141,0.325684,sentence,"Xbox One's Terraria Facing This ""Serious Issue...",A new update for the A new update for the Xbox...,34,27
6,357,0.704158,0.862793,0.488770,sentence,The National Safety Council has some advice fo...,Men are more likely to have a heart attack aft...,66,25
8,50,0.881565,0.857910,0.470215,sentence,You won’t believe why ESPN said they hired Jor...,The former Vanderbilt quarterback is set to ap...,75,32
11,313,0.866267,0.853027,0.732727,sentence,"co/e8jsXhLBbj In a hilarious scene, Priyanka i...","In a hilarious scene, Priyanka is seen calling...",45,44
12,395,0.756985,0.844238,0.473389,sentence,This Is What Happens When You Leave A Hotel Cl...,Instead of encountering a mound of dirty towel...,40,26


In [40]:
import os
import re
import itertools
import numpy as np
import pandas as pd


HISTORICAL_PASSAGE_SCORE = 0.387260

RECONSTRUCTION_RESULTS_PATH = os.path.join(
    WORK_DIR,
    "historical_passage_route_reconstruction_grid.csv"
)

RECONSTRUCTED_PREDICTIONS_PATH = os.path.join(
    WORK_DIR,
    "historical_passage_route_reconstructed_predictions.csv"
)

# 1. Helpers

def normalize_reconstruction_text(text):
    text = clean_text(text).lower()

    text = re.sub(
        r"[^\w]+",
        " ",
        text
    )

    return re.sub(
        r"\s+",
        " ",
        text
    ).strip()


def deduplicate_texts(texts):
    retained = []
    seen = set()

    for text in texts:
        text = clean_text(text)
        normalized = normalize_reconstruction_text(
            text
        )

        if not normalized:
            continue

        if normalized in seen:
            continue

        seen.add(normalized)
        retained.append(text)

    return retained


def concatenate_predictions(texts):
    return clean_text(
        " ".join(
            deduplicate_texts(texts)
        )
    )


def cap_prediction_words(
    text,
    maximum_words
):
    words = clean_text(text).split()

    if maximum_words is None:
        return " ".join(words)

    return " ".join(
        words[:maximum_words]
    )

# 2. Reconstruct paragraph-model top candidates

paragraph_reconstruction_df = (
    paragraph_candidate_predictions_df[
        paragraph_candidate_predictions_df[
            "source_kind"
        ].eq("paragraph")
    ]
    .copy()
)

paragraph_prediction_lookup = {}

for row_number, group in (
    paragraph_reconstruction_df.groupby(
        "row_number"
    )
):
    group = group.copy()

    score_ranked_group = (
        group.sort_values(
            [
                "answerability_margin",
                "best_span_score"
            ],
            ascending=[
                False,
                False
            ]
        )
    )

    unique_score_rows = []
    seen_predictions = set()

    for row in score_ranked_group.itertuples(
        index=False
    ):
        normalized = (
            normalize_reconstruction_text(
                row.predicted_text
            )
        )

        if not normalized:
            continue

        if normalized in seen_predictions:
            continue

        seen_predictions.add(normalized)
        unique_score_rows.append({
            "text": clean_text(
                row.predicted_text
            ),
            "source_index": int(
                row.source_index
            )
        })

        if len(unique_score_rows) == 3:
            break

    score_order_texts = [
        item["text"]
        for item in unique_score_rows
    ]

    source_order_texts = [
        item["text"]
        for item in sorted(
            unique_score_rows,
            key=lambda item:
            item["source_index"]
        )
    ]

    paragraph_prediction_lookup[
        int(row_number)
    ] = {
        "paragraph_top1":
            concatenate_predictions(
                score_order_texts[:1]
            ),

        "paragraph_top3_score_order":
            concatenate_predictions(
                score_order_texts
            ),

        "paragraph_top3_source_order":
            concatenate_predictions(
                source_order_texts
            )
    }

# 3. Reconstruct article-model sentence candidates

article_rank_sentence_lookup = {}

for article_rank in [1, 2, 3]:
    rule_name = (
        f"article_rank{article_rank}"
        "_containing_sentence"
    )

    rank_df = (
        boundary_results_df[
            boundary_results_df[
                "rule"
            ].eq(rule_name)
        ][
            [
                "row_number",
                "prediction"
            ]
        ]
        .drop_duplicates(
            subset=["row_number"],
            keep="first"
        )
    )

    article_rank_sentence_lookup[
        article_rank
    ] = (
        rank_df
        .set_index("row_number")[
            "prediction"
        ]
        .to_dict()
    )


article_rank_position_df = (
    article_span_candidates_df[
        article_span_candidates_df[
            "candidate_rank"
        ].le(3)
    ][
        [
            "row_number",
            "candidate_rank",
            "character_start"
        ]
    ]
    .copy()
)

article_position_lookup = {
    (
        int(row.row_number),
        int(row.candidate_rank)
    ): int(row.character_start)
    for row in (
        article_rank_position_df.itertuples(
            index=False
        )
    )
}


article_prediction_lookup = {}

for row_number in sorted(
    int(value)
    for value in passage_row_numbers
):
    ranked_items = []

    for article_rank in [1, 2, 3]:
        prediction = (
            article_rank_sentence_lookup[
                article_rank
            ].get(
                row_number,
                ""
            )
        )

        if not clean_text(prediction):
            continue

        ranked_items.append({
            "rank": article_rank,
            "text": clean_text(
                prediction
            ),
            "character_start":
                article_position_lookup.get(
                    (
                        row_number,
                        article_rank
                    ),
                    10**12
                )
        })

    score_order_texts = [
        item["text"]
        for item in sorted(
            ranked_items,
            key=lambda item:
            item["rank"]
        )
    ]

    source_order_texts = [
        item["text"]
        for item in sorted(
            ranked_items,
            key=lambda item:
            item["character_start"]
        )
    ]

    article_prediction_lookup[
        row_number
    ] = {
        "article_sentence_top1":
            concatenate_predictions(
                score_order_texts[:1]
            ),

        "article_sentence_top2_score_order":
            concatenate_predictions(
                score_order_texts[:2]
            ),

        "article_sentence_top3_score_order":
            concatenate_predictions(
                score_order_texts[:3]
            ),

        "article_sentence_top2_source_order":
            concatenate_predictions(
                source_order_texts[:2]
            ),

        "article_sentence_top3_source_order":
            concatenate_predictions(
                source_order_texts[:3]
            )
    }

# 4. Evaluate plausible historical routing variants

type_probability_lookup = (
    val_type_predictions_df
    .set_index("row_number")[
        "prob_passage"
    ]
    .astype(float)
    .to_dict()
)

gold_text_lookup = {
    int(row_number):
        gold_lookup[
            int(row_number)
        ]["gold_text"]
    for row_number in passage_row_numbers
}


article_variants = [
    "article_sentence_top1",
    "article_sentence_top2_score_order",
    "article_sentence_top3_score_order",
    "article_sentence_top2_source_order",
    "article_sentence_top3_source_order"
]

paragraph_variants = [
    "paragraph_top1",
    "paragraph_top3_score_order",
    "paragraph_top3_source_order"
]

passage_probability_thresholds = [
    0.40,
    0.50,
    0.60
]

word_caps = [
    None,
    100
]

result_records = []
prediction_records_by_configuration = {}


for (
    article_variant,
    paragraph_variant,
    probability_threshold,
    word_cap
) in itertools.product(
    article_variants,
    paragraph_variants,
    passage_probability_thresholds,
    word_caps
):
    configuration_name = (
        f"{article_variant}"
        f"__{paragraph_variant}"
        f"__threshold_{probability_threshold:.2f}"
        f"__cap_{word_cap}"
    )

    row_records = []

    for row_number in sorted(
        int(value)
        for value in passage_row_numbers
    ):
        passage_probability = (
            type_probability_lookup[
                row_number
            ]
        )

        use_article_model = (
            passage_probability
            >= probability_threshold
        )

        if use_article_model:
            raw_prediction = (
                article_prediction_lookup[
                    row_number
                ][article_variant]
            )

            source_model = "article"

        else:
            raw_prediction = (
                paragraph_prediction_lookup[
                    row_number
                ][paragraph_variant]
            )

            source_model = "paragraph"

        prediction = cap_prediction_words(
            raw_prediction,
            word_cap
        )

        meteor_value = calculate_meteor(
            gold_text_lookup[
                row_number
            ],
            prediction
        )

        row_records.append({
            "row_number": row_number,
            "passage_probability":
                passage_probability,
            "source_model": source_model,
            "prediction": prediction,
            "meteor": meteor_value
        })

    configuration_df = pd.DataFrame(
        row_records
    )

    configuration_score = float(
        configuration_df[
            "meteor"
        ].mean()
    )

    distance_from_historical = abs(
        configuration_score
        - HISTORICAL_PASSAGE_SCORE
    )

    result_records.append({
        "configuration":
            configuration_name,
        "article_variant":
            article_variant,
        "paragraph_variant":
            paragraph_variant,
        "passage_probability_threshold":
            probability_threshold,
        "word_cap":
            (
                "none"
                if word_cap is None
                else word_cap
            ),
        "reconstructed_meteor":
            configuration_score,
        "distance_from_historical":
            distance_from_historical,
        "article_rows":
            int(
                configuration_df[
                    "source_model"
                ].eq("article").sum()
            ),
        "paragraph_rows":
            int(
                configuration_df[
                    "source_model"
                ].eq("paragraph").sum()
            ),
        "mean_word_count":
            float(
                configuration_df[
                    "prediction"
                ]
                .str.split()
                .str.len()
                .mean()
            )
    })

    prediction_records_by_configuration[
        configuration_name
    ] = configuration_df


# 5. Select the closest reconstruction

reconstruction_results_df = (
    pd.DataFrame(
        result_records
    )
    .sort_values(
        [
            "distance_from_historical",
            "reconstructed_meteor"
        ],
        ascending=[
            True,
            False
        ]
    )
    .reset_index(drop=True)
)

best_reconstruction = (
    reconstruction_results_df.iloc[0]
)

best_configuration_name = (
    best_reconstruction[
        "configuration"
    ]
)

best_reconstructed_predictions_df = (
    prediction_records_by_configuration[
        best_configuration_name
    ]
    .copy()
)

best_reconstructed_predictions_df[
    "gold_text"
] = (
    best_reconstructed_predictions_df[
        "row_number"
    ].map(
        gold_text_lookup
    )
)

best_reconstructed_predictions_df[
    "configuration"
] = (
    best_configuration_name
)

# 6. Save results

reconstruction_results_df.to_csv(
    RECONSTRUCTION_RESULTS_PATH,
    index=False
)

best_reconstructed_predictions_df.to_csv(
    RECONSTRUCTED_PREDICTIONS_PATH,
    index=False
)

# 7. Report

print(
    "Known historical passage score:",
    HISTORICAL_PASSAGE_SCORE
)

print(
    "\nClosest reconstruction configurations:"
)

display(
    reconstruction_results_df.head(15)
)

print("\nBest reconstruction:")

print(
    "Configuration:",
    best_configuration_name
)

print(
    "Reconstructed METEOR:",
    round(
        float(
            best_reconstruction[
                "reconstructed_meteor"
            ]
        ),
        6
    )
)

print(
    "Absolute difference:",
    round(
        float(
            best_reconstruction[
                "distance_from_historical"
            ]
        ),
        6
    )
)

print(
    "Article-routed rows:",
    int(
        best_reconstruction[
            "article_rows"
        ]
    )
)

print(
    "Paragraph fallback rows:",
    int(
        best_reconstruction[
            "paragraph_rows"
        ]
    )
)

print(
    "\nSaved reconstruction grid:",
    RECONSTRUCTION_RESULTS_PATH
)

print(
    "Saved best row-level predictions:",
    RECONSTRUCTED_PREDICTIONS_PATH
)

print(
    "\nClose enough to use as the frozen "
    "validation baseline:",
    float(
        best_reconstruction[
            "distance_from_historical"
        ]
    ) <= 0.002
)

Known historical passage score: 0.38726

Closest reconstruction configurations:


,configuration,article_variant,paragraph_variant,passage_probability_threshold,word_cap,reconstructed_meteor,distance_from_historical,article_rows,paragraph_rows,mean_word_count
0,article_sentence_top1__paragraph_top3_score_or...,article_sentence_top1,paragraph_top3_score_order,0.6,none,0.386986,0.000274,128,40,25.660714
1,article_sentence_top1__paragraph_top3_score_or...,article_sentence_top1,paragraph_top3_score_order,0.6,100,0.386986,0.000274,128,40,25.660714
2,article_sentence_top1__paragraph_top3_source_o...,article_sentence_top1,paragraph_top3_source_order,0.6,none,0.385730,0.001530,128,40,25.660714
3,article_sentence_top1__paragraph_top3_source_o...,article_sentence_top1,paragraph_top3_source_order,0.6,100,0.385730,0.001530,128,40,25.660714
4,article_sentence_top2_source_order__paragraph_...,article_sentence_top2_source_order,paragraph_top3_source_order,0.6,none,0.389615,0.002355,128,40,34.630952
5,article_sentence_top2_source_order__paragraph_...,article_sentence_top2_source_order,paragraph_top3_source_order,0.6,100,0.389615,0.002355,128,40,34.630952
6,article_sentence_top2_source_order__paragraph_...,article_sentence_top2_source_order,paragraph_top3_score_order,0.6,none,0.390871,0.003611,128,40,34.630952
7,article_sentence_top2_source_order__paragraph_...,article_sentence_top2_source_order,paragraph_top3_score_order,0.6,100,0.390871,0.003611,128,40,34.630952
8,article_sentence_top3_score_order__paragraph_t...,article_sentence_top3_score_order,paragraph_top3_source_order,0.6,100,0.393301,0.006041,128,40,41.309524
9,article_sentence_top3_score_order__paragraph_t...,article_sentence_top3_score_order,paragraph_top3_source_order,0.6,none,0.394201,0.006941,128,40,41.851190



Best reconstruction:
Configuration: article_sentence_top1__paragraph_top3_score_order__threshold_0.60__cap_None
Reconstructed METEOR: 0.386986
Absolute difference: 0.000274
Article-routed rows: 128
Paragraph fallback rows: 40

Saved reconstruction grid: /kaggle/working/task2_passage_v1/historical_passage_route_reconstruction_grid.csv
Saved best row-level predictions: /kaggle/working/task2_passage_v1/historical_passage_route_reconstructed_predictions.csv

Close enough to use as the frozen validation baseline: True


In [41]:
import os
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold


EXACT_GATE_SUMMARY_PATH = os.path.join(
    WORK_DIR,
    "val_cross_encoder_vs_reconstructed_baseline_summary.csv"
)

EXACT_GATE_CV_PATH = os.path.join(
    WORK_DIR,
    "val_cross_encoder_vs_reconstructed_baseline_cv.csv"
)

EXACT_GATE_FREQUENCY_PATH = os.path.join(
    WORK_DIR,
    "val_cross_encoder_vs_reconstructed_baseline_frequency.csv"
)

BOOTSTRAP_ITERATIONS = 5000
RANDOM_SEED = 641

# 1. Reconstructed frozen-route baseline

reconstructed_baseline_df = (
    best_reconstructed_predictions_df[
        [
            "row_number",
            "prediction",
            "meteor"
        ]
    ]
    .rename(
        columns={
            "prediction": "baseline_prediction",
            "meteor": "baseline_meteor"
        }
    )
    .copy()
)

reconstructed_baseline_df[
    "row_number"
] = reconstructed_baseline_df[
    "row_number"
].astype(int)

# 2. Obtain cross-encoder top-1 and top-2 scores

validation_scored_df = (
    validation_scored_df
    .copy()
)

validation_scored_df[
    "row_number"
] = validation_scored_df[
    "row_number"
].astype(int)

validation_scored_df[
    "cross_encoder_score"
] = validation_scored_df[
    "cross_encoder_score"
].astype(np.float32)

validation_scored_df = (
    validation_scored_df
    .sort_values(
        [
            "row_number",
            "cross_encoder_score"
        ],
        ascending=[
            True,
            False
        ]
    )
    .reset_index(drop=True)
)

validation_scored_df[
    "model_rank"
] = (
    validation_scored_df
    .groupby("row_number")
    .cumcount()
    + 1
)

validation_top_two_df = (
    validation_scored_df[
        validation_scored_df[
            "model_rank"
        ].le(2)
    ][
        [
            "row_number",
            "model_rank",
            "cross_encoder_score"
        ]
    ]
    .pivot(
        index="row_number",
        columns="model_rank",
        values="cross_encoder_score"
    )
    .rename(
        columns={
            1: "top1_score",
            2: "top2_score"
        }
    )
    .reset_index()
)

validation_top_two_df[
    "score_margin"
] = (
    validation_top_two_df[
        "top1_score"
    ]
    -
    validation_top_two_df[
        "top2_score"
    ]
)

# 3. Merge all deployable validation information

exact_gate_df = (
    validation_selected_df[
        [
            "row_number",
            "gold_type",
            "candidate_text",
            "meteor",
            "cross_encoder_score",
            "variant",
            "word_count"
        ]
    ]
    .rename(
        columns={
            "meteor": "cross_encoder_meteor",
            "word_count": "candidate_word_count"
        }
    )
    .merge(
        validation_top_two_df[
            [
                "row_number",
                "top2_score",
                "score_margin"
            ]
        ],
        on="row_number",
        how="inner",
        validate="one_to_one"
    )
    .merge(
        val_type_predictions_df[
            [
                "row_number",
                "prob_phrase",
                "prob_passage",
                "prob_multi"
            ]
        ],
        on="row_number",
        how="inner",
        validate="one_to_one"
    )
    .merge(
        reconstructed_baseline_df,
        on="row_number",
        how="inner",
        validate="one_to_one"
    )
    .sort_values("row_number")
    .reset_index(drop=True)
)

if len(exact_gate_df) != 168:
    raise RuntimeError(
        f"Expected 168 rows, found {len(exact_gate_df)}."
    )


def normalize_comparison_text(text):
    text = re.sub(
        r"[^\w]+",
        " ",
        str(text).lower()
    )

    return re.sub(
        r"\s+",
        " ",
        text
    ).strip()


exact_gate_df[
    "candidate_differs_from_baseline"
] = [
    normalize_comparison_text(candidate)
    != normalize_comparison_text(baseline)
    for candidate, baseline in zip(
        exact_gate_df[
            "candidate_text"
        ],
        exact_gate_df[
            "baseline_prediction"
        ]
    )
]

# 4. Small set of predetermined conservative gates
gate_configs = [
    {
        "rule": "frozen_baseline_only",
        "prob": np.inf,
        "score": np.inf,
        "margin": np.inf,
        "sentence_only": False,
        "max_words": np.inf
    },
    {
        "rule": "p060_s075_m010",
        "prob": 0.60,
        "score": 0.75,
        "margin": 0.10,
        "sentence_only": False,
        "max_words": np.inf
    },
    {
        "rule": "p060_s075_m010_sentence",
        "prob": 0.60,
        "score": 0.75,
        "margin": 0.10,
        "sentence_only": True,
        "max_words": 60
    },
    {
        "rule": "p070_s080_m015_sentence",
        "prob": 0.70,
        "score": 0.80,
        "margin": 0.15,
        "sentence_only": True,
        "max_words": 60
    },
    {
        "rule": "p080_s080_m015_sentence",
        "prob": 0.80,
        "score": 0.80,
        "margin": 0.15,
        "sentence_only": True,
        "max_words": 60
    },
    {
        "rule": "p080_s085_m020_sentence",
        "prob": 0.80,
        "score": 0.85,
        "margin": 0.20,
        "sentence_only": True,
        "max_words": 60
    },
    {
        "rule": "p085_s085_m020_sentence",
        "prob": 0.85,
        "score": 0.85,
        "margin": 0.20,
        "sentence_only": True,
        "max_words": 60
    },
    {
        "rule": "p090_s085_m020_sentence",
        "prob": 0.90,
        "score": 0.85,
        "margin": 0.20,
        "sentence_only": True,
        "max_words": 60
    },
    {
        "rule": "p080_s090_m030_sentence",
        "prob": 0.80,
        "score": 0.90,
        "margin": 0.30,
        "sentence_only": True,
        "max_words": 50
    },
    {
        "rule": "p090_s090_m030_sentence",
        "prob": 0.90,
        "score": 0.90,
        "margin": 0.30,
        "sentence_only": True,
        "max_words": 50
    }
]

# 5. Calculate row-level hybrid results

rule_score_matrix = {}
rule_usage_matrix = {}

for config in gate_configs:
    if config["rule"] == "frozen_baseline_only":
        use_cross_encoder = np.zeros(
            len(exact_gate_df),
            dtype=bool
        )

    else:
        use_cross_encoder = (
            exact_gate_df[
                "prob_passage"
            ].ge(config["prob"])
            &
            exact_gate_df[
                "cross_encoder_score"
            ].ge(config["score"])
            &
            exact_gate_df[
                "score_margin"
            ].ge(config["margin"])
            &
            exact_gate_df[
                "candidate_differs_from_baseline"
            ]
        )

        if config["sentence_only"]:
            use_cross_encoder &= (
                exact_gate_df[
                    "variant"
                ].eq("sentence")
            )

        use_cross_encoder &= (
            exact_gate_df[
                "candidate_word_count"
            ].le(config["max_words"])
        )

        use_cross_encoder = (
            use_cross_encoder.to_numpy()
        )

    hybrid_scores = np.where(
        use_cross_encoder,
        exact_gate_df[
            "cross_encoder_meteor"
        ].to_numpy(),
        exact_gate_df[
            "baseline_meteor"
        ].to_numpy()
    )

    rule_score_matrix[
        config["rule"]
    ] = hybrid_scores

    rule_usage_matrix[
        config["rule"]
    ] = use_cross_encoder


rule_score_df = pd.DataFrame(
    rule_score_matrix,
    index=exact_gate_df[
        "row_number"
    ].astype(int)
)

rule_usage_df = pd.DataFrame(
    rule_usage_matrix,
    index=exact_gate_df[
        "row_number"
    ].astype(int)
)

# 6. Paired bootstrap against reconstructed baseline

random_generator = np.random.default_rng(
    RANDOM_SEED
)

baseline_scores = exact_gate_df[
    "baseline_meteor"
].to_numpy()

bootstrap_indices = random_generator.integers(
    0,
    len(exact_gate_df),
    size=(
        BOOTSTRAP_ITERATIONS,
        len(exact_gate_df)
    )
)

summary_records = []

for config in gate_configs:
    rule_name = config["rule"]

    hybrid_scores = rule_score_df[
        rule_name
    ].to_numpy()

    paired_differences = (
        hybrid_scores
        - baseline_scores
    )

    bootstrap_mean_differences = (
        paired_differences[
            bootstrap_indices
        ].mean(axis=1)
    )

    use_cross_encoder = rule_usage_df[
        rule_name
    ].to_numpy()

    changed_rows = int(
        use_cross_encoder.sum()
    )

    true_passage_mask = exact_gate_df[
        "gold_type"
    ].eq("passage").to_numpy()

    summary_records.append({
        "rule": rule_name,
        "all_168_meteor": float(
            hybrid_scores.mean()
        ),
        "gain_vs_reconstructed_baseline": float(
            paired_differences.mean()
        ),
        "estimated_full_validation_gain": float(
            paired_differences.mean()
            * 168
            / 400
        ),
        "cross_encoder_replacements": changed_rows,
        "true_passage_replacements": int(
            (
                use_cross_encoder
                & true_passage_mask
            ).sum()
        ),
        "misrouted_replacements": int(
            (
                use_cross_encoder
                & ~true_passage_mask
            ).sum()
        ),
        "replacement_win_rate": float(
            (
                paired_differences[
                    use_cross_encoder
                ] > 0
            ).mean()
            if changed_rows
            else np.nan
        ),
        "bootstrap_p05_gain": float(
            np.quantile(
                bootstrap_mean_differences,
                0.05
            )
        ),
        "bootstrap_p50_gain": float(
            np.quantile(
                bootstrap_mean_differences,
                0.50
            )
        ),
        "bootstrap_p95_gain": float(
            np.quantile(
                bootstrap_mean_differences,
                0.95
            )
        ),
        "bootstrap_probability_positive": float(
            (
                bootstrap_mean_differences
                > 0
            ).mean()
        )
    })


exact_gate_summary_df = (
    pd.DataFrame(summary_records)
    .sort_values(
        "all_168_meteor",
        ascending=False
    )
    .reset_index(drop=True)
)

exact_gate_summary_df.to_csv(
    EXACT_GATE_SUMMARY_PATH,
    index=False
)

# 7. Repeated held-out rule selection

SPLIT_SEEDS = [
    17,
    83,
    641,
    2026,
    9917,
    44,
    112,
    333,
    777,
    1234
]

metadata_df = (
    exact_gate_df[
        [
            "row_number",
            "gold_type"
        ]
    ]
    .set_index("row_number")
    .loc[rule_score_df.index]
)

seed_records = []
fold_records = []

for split_seed in SPLIT_SEEDS:
    splitter = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=split_seed
    )

    heldout_scores = pd.Series(
        np.nan,
        index=rule_score_df.index,
        dtype=float
    )

    for fold_number, (
        train_positions,
        validation_positions
    ) in enumerate(
        splitter.split(
            rule_score_df,
            metadata_df[
                "gold_type"
            ]
        ),
        start=1
    ):
        training_rows = (
            rule_score_df.index[
                train_positions
            ]
        )

        heldout_rows = (
            rule_score_df.index[
                validation_positions
            ]
        )

        training_means = (
            rule_score_df.loc[
                training_rows
            ].mean(axis=0)
        )

        selected_rule = (
            training_means.idxmax()
        )

        selected_scores = (
            rule_score_df.loc[
                heldout_rows,
                selected_rule
            ]
        )

        heldout_scores.loc[
            heldout_rows
        ] = selected_scores

        fold_records.append({
            "seed": split_seed,
            "fold": fold_number,
            "selected_rule": selected_rule,
            "training_score": float(
                training_means[
                    selected_rule
                ]
            ),
            "heldout_score": float(
                selected_scores.mean()
            )
        })

    seed_score = float(
        heldout_scores.mean()
    )

    seed_records.append({
        "seed": split_seed,
        "heldout_all_168_meteor": seed_score,
        "gain_vs_reconstructed_baseline": (
            seed_score
            - baseline_scores.mean()
        ),
        "estimated_full_validation_gain": (
            (
                seed_score
                - baseline_scores.mean()
            )
            * 168
            / 400
        )
    })


exact_gate_cv_df = pd.DataFrame(
    seed_records
)

fold_selection_df = pd.DataFrame(
    fold_records
)

exact_gate_frequency_df = (
    fold_selection_df[
        "selected_rule"
    ]
    .value_counts()
    .rename_axis("rule")
    .reset_index(
        name="selected_folds"
    )
)

exact_gate_frequency_df[
    "selection_rate"
] = (
    exact_gate_frequency_df[
        "selected_folds"
    ]
    / len(fold_selection_df)
)

exact_gate_cv_df.to_csv(
    EXACT_GATE_CV_PATH,
    index=False
)

exact_gate_frequency_df.to_csv(
    EXACT_GATE_FREQUENCY_PATH,
    index=False
)

# 8. Report
print(
    "Reconstructed frozen-route score:",
    round(
        baseline_scores.mean(),
        6
    )
)

print(
    "\nExact row-by-row gate comparison:"
)

display(
    exact_gate_summary_df
)

print(
    "\nRepeated held-out results:"
)

display(
    exact_gate_cv_df
)

print(
    "\nRule-selection frequency:"
)

display(
    exact_gate_frequency_df
)

print(
    "\nRepeated-CV summary:"
)

print(
    "Mean held-out score:",
    round(
        exact_gate_cv_df[
            "heldout_all_168_meteor"
        ].mean(),
        6
    )
)

print(
    "Minimum held-out score:",
    round(
        exact_gate_cv_df[
            "heldout_all_168_meteor"
        ].min(),
        6
    )
)

print(
    "Mean gain versus reconstructed baseline:",
    round(
        exact_gate_cv_df[
            "gain_vs_reconstructed_baseline"
        ].mean(),
        6
    )
)

print(
    "Estimated full-validation gain:",
    round(
        exact_gate_cv_df[
            "estimated_full_validation_gain"
        ].mean(),
        6
    )
)

print(
    "\nCheckpoint complete:",
    os.path.exists(
        EXACT_GATE_SUMMARY_PATH
    )
    and os.path.exists(
        EXACT_GATE_CV_PATH
    )
    and os.path.exists(
        EXACT_GATE_FREQUENCY_PATH
    )
)

Reconstructed frozen-route score: 0.386986

Exact row-by-row gate comparison:


,rule,all_168_meteor,gain_vs_reconstructed_baseline,estimated_full_validation_gain,cross_encoder_replacements,true_passage_replacements,misrouted_replacements,replacement_win_rate,bootstrap_p05_gain,bootstrap_p50_gain,bootstrap_p95_gain,bootstrap_probability_positive
0,p060_s075_m010,0.403776,0.016790,0.007052,8,6,2,0.750000,0.004362,0.016123,0.033001,0.9964
1,p060_s075_m010_sentence,0.394283,0.007297,0.003065,6,4,2,0.666667,0.000227,0.006729,0.018602,0.9624
2,frozen_baseline_only,0.386986,0.000000,0.000000,0,0,0,NaN,0.000000,0.000000,0.000000,0.0000
3,p090_s090_m030_sentence,0.386986,0.000000,0.000000,0,0,0,NaN,0.000000,0.000000,0.000000,0.0000
4,p080_s090_m030_sentence,0.386986,0.000000,0.000000,0,0,0,NaN,0.000000,0.000000,0.000000,0.0000
5,p090_s085_m020_sentence,0.386986,0.000000,0.000000,0,0,0,NaN,0.000000,0.000000,0.000000,0.0000
6,p080_s085_m020_sentence,0.386844,-0.000142,-0.000060,1,0,1,0.000000,-0.000427,-0.000142,0.000000,0.0000
7,p080_s080_m015_sentence,0.386844,-0.000142,-0.000060,1,0,1,0.000000,-0.000427,-0.000142,0.000000,0.0000
8,p070_s080_m015_sentence,0.386844,-0.000142,-0.000060,1,0,1,0.000000,-0.000427,-0.000142,0.000000,0.0000
9,p085_s085_m020_sentence,0.386844,-0.000142,-0.000060,1,0,1,0.000000,-0.000427,-0.000142,0.000000,0.0000



Repeated held-out results:


,seed,heldout_all_168_meteor,gain_vs_reconstructed_baseline,estimated_full_validation_gain
0,17,0.403776,0.01679,0.007052
1,83,0.403776,0.01679,0.007052
2,641,0.403776,0.01679,0.007052
3,2026,0.403776,0.01679,0.007052
4,9917,0.403776,0.01679,0.007052
5,44,0.403776,0.01679,0.007052
6,112,0.403776,0.01679,0.007052
7,333,0.403776,0.01679,0.007052
8,777,0.403776,0.01679,0.007052
9,1234,0.403776,0.01679,0.007052



Rule-selection frequency:


,rule,selected_folds,selection_rate
0,p060_s075_m010,50,1.0



Repeated-CV summary:
Mean held-out score: 0.403776
Minimum held-out score: 0.403776
Mean gain versus reconstructed baseline: 0.01679
Estimated full-validation gain: 0.007052

Checkpoint complete: True


In [44]:
import os
import re
import numpy as np
import pandas as pd


AGREEMENT_SUMMARY_PATH = os.path.join(
    WORK_DIR,
    "val_exact_gate_model_agreement_summary.csv"
)

AGREEMENT_TEST_PATH = os.path.join(
    WORK_DIR,
    "test_exact_gate_model_agreement_candidates.csv"
)

BOOTSTRAP_ITERATIONS = 5000
RANDOM_SEED = 641

# 1. Text-overlap helpers

def agreement_token_set(text):
    return set(
        re.findall(
            r"\b\w+\b",
            clean_text(text).lower()
        )
    )


def calculate_agreement(first_text, second_text):
    first_tokens = agreement_token_set(
        first_text
    )

    second_tokens = agreement_token_set(
        second_text
    )

    if not first_tokens or not second_tokens:
        return {
            "jaccard": 0.0,
            "containment": 0.0
        }

    intersection_size = len(
        first_tokens & second_tokens
    )

    union_size = len(
        first_tokens | second_tokens
    )

    smaller_size = min(
        len(first_tokens),
        len(second_tokens)
    )

    return {
        "jaccard": (
            intersection_size / union_size
            if union_size
            else 0.0
        ),
        "containment": (
            intersection_size / smaller_size
            if smaller_size
            else 0.0
        )
    }

# 2. Add article-QA containing sentences to validation

validation_containing_sentence_df = (
    boundary_results_df[
        boundary_results_df[
            "rule"
        ].eq(
            "article_rank1_containing_sentence"
        )
    ][
        [
            "row_number",
            "prediction"
        ]
    ]
    .rename(
        columns={
            "prediction":
                "article_qa_containing_sentence"
        }
    )
    .drop_duplicates(
        subset=["row_number"],
        keep="first"
    )
)

agreement_validation_df = (
    exact_gate_df
    .merge(
        validation_containing_sentence_df,
        on="row_number",
        how="inner",
        validate="one_to_one"
    )
    .copy()
)

validation_agreements = [
    calculate_agreement(
        candidate,
        article_sentence
    )
    for candidate, article_sentence
    in zip(
        agreement_validation_df[
            "candidate_text"
        ],
        agreement_validation_df[
            "article_qa_containing_sentence"
        ]
    )
]

agreement_validation_df[
    "model_jaccard"
] = [
    result["jaccard"]
    for result in validation_agreements
]

agreement_validation_df[
    "model_containment"
] = [
    result["containment"]
    for result in validation_agreements
]

# 3. Add article-QA containing sentences to test

test_containing_sentence_df = (
    test_sentence_predictions_df[
        [
            "row_number",
            "containing_sentence"
        ]
    ]
    .rename(
        columns={
            "containing_sentence":
                "article_qa_containing_sentence"
        }
    )
    .copy()
)

agreement_test_df = (
    top1_selection_df
    .merge(
        test_containing_sentence_df,
        on="row_number",
        how="inner",
        validate="one_to_one"
    )
    .copy()
)

test_agreements = [
    calculate_agreement(
        candidate,
        article_sentence
    )
    for candidate, article_sentence
    in zip(
        agreement_test_df[
            "candidate_text"
        ],
        agreement_test_df[
            "article_qa_containing_sentence"
        ]
    )
]

agreement_test_df[
    "model_jaccard"
] = [
    result["jaccard"]
    for result in test_agreements
]

agreement_test_df[
    "model_containment"
] = [
    result["containment"]
    for result in test_agreements
]

# 4. Exact cross-encoder gate
validation_base_gate = (
    agreement_validation_df[
        "prob_passage"
    ].ge(0.60)
    &
    agreement_validation_df[
        "cross_encoder_score"
    ].ge(0.75)
    &
    agreement_validation_df[
        "score_margin"
    ].ge(0.10)
    &
    agreement_validation_df[
        "candidate_differs_from_baseline"
    ]
)

test_base_gate = (
    agreement_test_df[
        "prob_passage"
    ].ge(0.60)
    &
    agreement_test_df[
        "cross_encoder_score"
    ].ge(0.75)
    &
    agreement_test_df[
        "score_margin"
    ].ge(0.10)
    &
    agreement_test_df[
        "candidate_differs_from_baseline"
    ]
)

# 5. Predetermined agreement filters
agreement_rules = [
    {
        "rule": "exact_gate_no_agreement_filter",
        "minimum_jaccard": 0.0,
        "minimum_containment": 0.0
    },
    {
        "rule": "containment_at_least_025",
        "minimum_jaccard": 0.0,
        "minimum_containment": 0.25
    },
    {
        "rule": "containment_at_least_050",
        "minimum_jaccard": 0.0,
        "minimum_containment": 0.50
    },
    {
        "rule": "containment_at_least_075",
        "minimum_jaccard": 0.0,
        "minimum_containment": 0.75
    },
    {
        "rule": "jaccard_at_least_010",
        "minimum_jaccard": 0.10,
        "minimum_containment": 0.0
    },
    {
        "rule": "jaccard_at_least_025",
        "minimum_jaccard": 0.25,
        "minimum_containment": 0.0
    },
    {
        "rule": "jaccard_at_least_050",
        "minimum_jaccard": 0.50,
        "minimum_containment": 0.0
    },
    {
        "rule": "containment050_and_jaccard025",
        "minimum_jaccard": 0.25,
        "minimum_containment": 0.50
    }
]

# 6. Paired validation evaluation

baseline_scores = (
    agreement_validation_df[
        "baseline_meteor"
    ].to_numpy()
)

random_generator = np.random.default_rng(
    RANDOM_SEED
)

bootstrap_indices = (
    random_generator.integers(
        0,
        len(agreement_validation_df),
        size=(
            BOOTSTRAP_ITERATIONS,
            len(agreement_validation_df)
        )
    )
)

summary_records = []

for rule_config in agreement_rules:
    validation_use_cross_encoder = (
        validation_base_gate
        &
        agreement_validation_df[
            "model_jaccard"
        ].ge(
            rule_config[
                "minimum_jaccard"
            ]
        )
        &
        agreement_validation_df[
            "model_containment"
        ].ge(
            rule_config[
                "minimum_containment"
            ]
        )
    )

    test_use_cross_encoder = (
        test_base_gate
        &
        agreement_test_df[
            "model_jaccard"
        ].ge(
            rule_config[
                "minimum_jaccard"
            ]
        )
        &
        agreement_test_df[
            "model_containment"
        ].ge(
            rule_config[
                "minimum_containment"
            ]
        )
    )

    hybrid_scores = np.where(
        validation_use_cross_encoder,
        agreement_validation_df[
            "cross_encoder_meteor"
        ],
        agreement_validation_df[
            "baseline_meteor"
        ]
    )

    paired_gain = (
        hybrid_scores
        - baseline_scores
    )

    bootstrap_gains = (
        paired_gain[
            bootstrap_indices
        ].mean(axis=1)
    )

    replacement_gains = (
        paired_gain[
            validation_use_cross_encoder
            .to_numpy()
        ]
    )

    summary_records.append({
        "rule": rule_config["rule"],
        "minimum_jaccard":
            rule_config[
                "minimum_jaccard"
            ],
        "minimum_containment":
            rule_config[
                "minimum_containment"
            ],
        "validation_score":
            float(
                hybrid_scores.mean()
            ),
        "validation_gain":
            float(
                paired_gain.mean()
            ),
        "estimated_full_validation_gain":
            float(
                paired_gain.mean()
                * 168
                / 400
            ),
        "validation_replacements":
            int(
                validation_use_cross_encoder.sum()
            ),
        "validation_replacement_wins":
            int(
                (
                    replacement_gains > 0
                ).sum()
            ),
        "validation_replacement_losses":
            int(
                (
                    replacement_gains < 0
                ).sum()
            ),
        "replacement_win_rate":
            float(
                (
                    replacement_gains > 0
                ).mean()
                if len(replacement_gains)
                else np.nan
            ),
        "bootstrap_p05_gain":
            float(
                np.quantile(
                    bootstrap_gains,
                    0.05
                )
            ),
        "bootstrap_probability_positive":
            float(
                (
                    bootstrap_gains > 0
                ).mean()
            ),
        "test_replacements":
            int(
                test_use_cross_encoder.sum()
            ),
        "test_replacement_rate":
            float(
                test_use_cross_encoder.mean()
            )
    })


agreement_summary_df = (
    pd.DataFrame(
        summary_records
    )
    .sort_values(
        [
            "validation_gain",
            "bootstrap_probability_positive"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

agreement_summary_df.to_csv(
    AGREEMENT_SUMMARY_PATH,
    index=False
)

# 7. Detailed validation and test gate rows

validation_gate_rows_df = (
    agreement_validation_df[
        validation_base_gate
    ][
        [
            "row_number",
            "gold_type",
            "prob_passage",
            "cross_encoder_score",
            "score_margin",
            "model_jaccard",
            "model_containment",
            "baseline_prediction",
            "candidate_text",
            "article_qa_containing_sentence",
            "baseline_meteor",
            "cross_encoder_meteor"
        ]
    ]
    .copy()
)

validation_gate_rows_df[
    "paired_gain"
] = (
    validation_gate_rows_df[
        "cross_encoder_meteor"
    ]
    -
    validation_gate_rows_df[
        "baseline_meteor"
    ]
)

test_gate_rows_df = (
    agreement_test_df[
        test_base_gate
    ][
        [
            "row_number",
            "prob_passage",
            "cross_encoder_score",
            "score_margin",
            "model_jaccard",
            "model_containment",
            "variant",
            "baseline_prediction",
            "candidate_text",
            "article_qa_containing_sentence"
        ]
    ]
    .copy()
    .sort_values(
        [
            "model_containment",
            "model_jaccard",
            "score_margin"
        ],
        ascending=False
    )
)

test_gate_rows_df.to_csv(
    AGREEMENT_TEST_PATH,
    index=False
)

# 8. Report

print(
    "Reconstructed baseline score:",
    round(
        baseline_scores.mean(),
        6
    )
)

print(
    "\nAgreement-filter comparison:"
)

display(
    agreement_summary_df
)

print(
    "\nEight validation rows passing "
    "the original exact gate:"
)

display(
    validation_gate_rows_df
    .sort_values(
        "paired_gain",
        ascending=False
    )
)

print(
    "\nTest rows passing the original exact gate:"
)

display(
    test_gate_rows_df
)

print(
    "\nSaved agreement summary:",
    AGREEMENT_SUMMARY_PATH
)

print(
    "Saved test candidates:",
    AGREEMENT_TEST_PATH
)

print(
    "\nCheckpoint complete:",
    os.path.exists(
        AGREEMENT_SUMMARY_PATH
    )
    and os.path.exists(
        AGREEMENT_TEST_PATH
    )
)

Reconstructed baseline score: 0.386986

Agreement-filter comparison:


,rule,minimum_jaccard,minimum_containment,validation_score,validation_gain,estimated_full_validation_gain,validation_replacements,validation_replacement_wins,validation_replacement_losses,replacement_win_rate,bootstrap_p05_gain,bootstrap_probability_positive,test_replacements,test_replacement_rate
0,exact_gate_no_agreement_filter,0.00,0.00,0.403776,0.016790,0.007052,8,6,2,0.75,0.004362,0.9964,21,0.114754
1,containment_at_least_025,0.00,0.25,0.397318,0.010332,0.004339,4,4,0,1.00,0.000661,0.9828,14,0.076503
2,jaccard_at_least_010,0.10,0.00,0.392952,0.005966,0.002506,3,3,0,1.00,0.000178,0.9522,14,0.076503
3,containment_at_least_050,0.00,0.50,0.392113,0.005126,0.002153,1,1,0,1.00,0.000000,0.6294,12,0.065574
4,containment_at_least_075,0.00,0.75,0.386986,0.000000,0.000000,0,0,0,NaN,0.000000,0.0000,12,0.065574
5,jaccard_at_least_025,0.25,0.00,0.386986,0.000000,0.000000,0,0,0,NaN,0.000000,0.0000,12,0.065574
6,jaccard_at_least_050,0.50,0.00,0.386986,0.000000,0.000000,0,0,0,NaN,0.000000,0.0000,12,0.065574
7,containment050_and_jaccard025,0.25,0.50,0.386986,0.000000,0.000000,0,0,0,NaN,0.000000,0.0000,12,0.065574



Eight validation rows passing the original exact gate:


,row_number,gold_type,prob_passage,cross_encoder_score,score_margin,model_jaccard,model_containment,baseline_prediction,candidate_text,article_qa_containing_sentence,baseline_meteor,cross_encoder_meteor,paired_gain
91,228,passage,0.822173,0.884766,0.119629,0.019231,0.043478,"Finally, after teasing the confrontation all s...",Kit Harington admitted during filming he may h...,"Finally, after teasing the confrontation all s...",0.019841,0.976531,0.956690
34,87,passage,0.866340,0.824707,0.115234,0.114286,0.666667,The headphones are incredibly cheaply made.,One way to do this cheaply is to make some com...,The headphones are incredibly cheaply made.,0.064655,0.925896,0.861241
95,235,passage,0.608934,0.766113,0.101074,0.032258,0.333333,Antibiotic-resistant infections spread through...,Resistance to antibiotics is growing at such a...,Antibiotic-resistant infections spread through...,0.000000,0.733497,0.733497
161,389,passage,0.866921,0.860352,0.128906,0.000000,0.000000,Michael Jackson is still alive.,"OK, so she's there and there's possibly someon...",Michael Jackson is still alive.,0.016667,0.208333,0.191667
106,266,phrase,0.758680,0.869141,0.118164,0.107143,0.272727,Pacquiao: ‘I don’t want public service to be m...,"According to him, boxing helped him provide fo...",Pacquiao: ‘I don’t want public service to be m...,0.000000,0.111111,0.111111
24,55,passage,0.848549,0.791992,0.319092,0.147059,0.263158,Men get the bedroom blues too: Study finds bot...,"In fact, they can become so sad and emotional ...",Men get the bedroom blues too: Study finds bot...,0.045872,0.075758,0.029886
124,301,phrase,0.856931,0.889648,0.422119,0.072727,0.160000,Scientific Reports journal puts forward the th...,Researchers – from Japan’s Meteorological Rese...,Scientific Reports journal puts forward the th...,0.121951,0.098039,-0.023912
8,18,passage,0.825573,0.857910,0.139648,0.047619,0.095238,Officer Steve Dunham responded to the call abo...,"""He told me he was trying to sell his stuffed ...",Officer Steve Dunham responded to the call abo...,0.135659,0.096154,-0.039505



Test rows passing the original exact gate:


,row_number,prob_passage,cross_encoder_score,score_margin,model_jaccard,model_containment,variant,baseline_prediction,candidate_text,article_qa_containing_sentence
145,313,0.866267,0.853027,0.732727,1.000000,1.000000,sentence,"co/e8jsXhLBbj In a hilarious scene, Priyanka i...","In a hilarious scene, Priyanka is seen calling...","In a hilarious scene, Priyanka is seen calling..."
72,154,0.704557,0.956543,0.693848,1.000000,1.000000,sentence,MIT researchers can read a book without openin...,Researchers from the Massachusetts Institute o...,Researchers from the Massachusetts Institute o...
85,187,0.817599,0.823242,0.673218,1.000000,1.000000,sentence,The highly controversial singer has been invol...,The highly controversial singer has been invol...,The highly controversial singer has been invol...
152,332,0.879119,0.932617,0.665527,1.000000,1.000000,sentence,"Hizbullah, an Iran-backed Lebanese Shia group,...",(AP) – The evacuation of civilians and opposit...,(AP) – The evacuation of civilians and opposit...
164,357,0.704158,0.862793,0.488770,1.000000,1.000000,sentence,The National Safety Council has some advice fo...,Men are more likely to have a heart attack aft...,Men are more likely to have a heart attack aft...
181,395,0.756985,0.844238,0.473389,1.000000,1.000000,sentence,This Is What Happens When You Leave A Hotel Cl...,Instead of encountering a mound of dirty towel...,Instead of encountering a mound of dirty towel...
22,50,0.881565,0.857910,0.470215,1.000000,1.000000,sentence,You won’t believe why ESPN said they hired Jor...,The former Vanderbilt quarterback is set to ap...,The former Vanderbilt quarterback is set to ap...
46,103,0.699235,0.869141,0.325684,1.000000,1.000000,sentence,"Xbox One's Terraria Facing This ""Serious Issue...",A new update for the A new update for the Xbox...,A new update for the A new update for the Xbox...
103,223,0.817307,0.946289,0.247070,1.000000,1.000000,sentence,Police: Four Women Arrested for Spraying Anti-...,Police arrested four women in North Carolina f...,Police arrested four women in North Carolina f...
39,93,0.643515,0.895020,0.242676,1.000000,1.000000,sentence,The 1937 Bugatti Type 57S was found in the gar...,The 1937 Bugatti Type 57S was found in the gar...,The 1937 Bugatti Type 57S was found in the gar...



Saved agreement summary: /kaggle/working/task2_passage_v1/val_exact_gate_model_agreement_summary.csv
Saved test candidates: /kaggle/working/task2_passage_v1/test_exact_gate_model_agreement_candidates.csv

Checkpoint complete: True


In [45]:
import os
import re
import numpy as np
import pandas as pd


QUALITY_GATE_ANALYSIS_PATH = os.path.join(
    WORK_DIR,
    "val_agreement025_quality_gate_analysis.csv"
)

QUALITY_GATE_TEST_PATH = os.path.join(
    WORK_DIR,
    "test_agreement025_quality_gate_candidates.csv"
)

# 1. Generic repeated-phrase detector

def has_adjacent_repeated_ngram(
    text,
    minimum_ngram=3,
    maximum_ngram=8
):
    tokens = re.findall(
        r"\b\w+\b",
        str(text).lower()
    )

    for ngram_size in range(
        minimum_ngram,
        maximum_ngram + 1
    ):
        maximum_start = (
            len(tokens)
            - 2 * ngram_size
        )

        for start_index in range(
            maximum_start + 1
        ):
            first_ngram = tokens[
                start_index:
                start_index + ngram_size
            ]

            second_ngram = tokens[
                start_index + ngram_size:
                start_index + 2 * ngram_size
            ]

            if first_ngram == second_ngram:
                return True

    return False

# 2. Add deployable text-quality features

agreement_validation_df = (
    agreement_validation_df.copy()
)

agreement_test_df = (
    agreement_test_df.copy()
)

agreement_validation_df[
    "candidate_word_count"
] = (
    agreement_validation_df[
        "candidate_text"
    ]
    .astype(str)
    .str.split()
    .str.len()
)

agreement_test_df[
    "candidate_word_count"
] = (
    agreement_test_df[
        "candidate_text"
    ]
    .astype(str)
    .str.split()
    .str.len()
)

agreement_validation_df[
    "has_repeated_ngram"
] = (
    agreement_validation_df[
        "candidate_text"
    ]
    .apply(
        has_adjacent_repeated_ngram
    )
)

agreement_test_df[
    "has_repeated_ngram"
] = (
    agreement_test_df[
        "candidate_text"
    ]
    .apply(
        has_adjacent_repeated_ngram
    )
)

# 3. Rebuild the validated base gate

validation_base_gate = (
    agreement_validation_df[
        "prob_passage"
    ].ge(0.60)
    &
    agreement_validation_df[
        "cross_encoder_score"
    ].ge(0.75)
    &
    agreement_validation_df[
        "score_margin"
    ].ge(0.10)
    &
    agreement_validation_df[
        "candidate_differs_from_baseline"
    ]
)

test_base_gate = (
    agreement_test_df[
        "prob_passage"
    ].ge(0.60)
    &
    agreement_test_df[
        "cross_encoder_score"
    ].ge(0.75)
    &
    agreement_test_df[
        "score_margin"
    ].ge(0.10)
    &
    agreement_test_df[
        "candidate_differs_from_baseline"
    ]
)

# 4. Agreement and general quality safeguards

validation_final_gate = (
    validation_base_gate
    &
    agreement_validation_df[
        "model_containment"
    ].ge(0.25)
    &
    agreement_validation_df[
        "candidate_word_count"
    ].between(
        8,
        60,
        inclusive="both"
    )
    &
    ~agreement_validation_df[
        "has_repeated_ngram"
    ]
)

test_final_gate = (
    test_base_gate
    &
    agreement_test_df[
        "model_containment"
    ].ge(0.25)
    &
    agreement_test_df[
        "candidate_word_count"
    ].between(
        8,
        60,
        inclusive="both"
    )
    &
    ~agreement_test_df[
        "has_repeated_ngram"
    ]
)

# 5. Exact validation comparison against reconstructed route

validation_hybrid_scores = np.where(
    validation_final_gate,
    agreement_validation_df[
        "cross_encoder_meteor"
    ],
    agreement_validation_df[
        "baseline_meteor"
    ]
)

validation_paired_gains = (
    validation_hybrid_scores
    -
    agreement_validation_df[
        "baseline_meteor"
    ].to_numpy()
)

validation_replacement_gains = (
    validation_paired_gains[
        validation_final_gate.to_numpy()
    ]
)

validation_score = float(
    validation_hybrid_scores.mean()
)

validation_gain = float(
    validation_paired_gains.mean()
)

# 6. Prepare the exact test replacement list

final_test_candidates_df = (
    agreement_test_df[
        test_final_gate
    ]
    .copy()
    .sort_values(
        [
            "model_containment",
            "cross_encoder_score"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

final_test_candidates_df[
    "baseline_word_count"
] = (
    final_test_candidates_df[
        "baseline_prediction"
    ]
    .astype(str)
    .str.split()
    .str.len()
)

# 7. Save the analysis

validation_analysis_df = pd.DataFrame({
    "reconstructed_baseline_score": [
        float(
            agreement_validation_df[
                "baseline_meteor"
            ].mean()
        )
    ],
    "quality_gate_score": [
        validation_score
    ],
    "quality_gate_gain": [
        validation_gain
    ],
    "estimated_full_validation_gain": [
        validation_gain
        * 168
        / 400
    ],
    "validation_replacements": [
        int(
            validation_final_gate.sum()
        )
    ],
    "validation_wins": [
        int(
            (
                validation_replacement_gains
                > 0
            ).sum()
        )
    ],
    "validation_losses": [
        int(
            (
                validation_replacement_gains
                < 0
            ).sum()
        )
    ],
    "test_replacements": [
        len(
            final_test_candidates_df
        )
    ]
})

validation_analysis_df.to_csv(
    QUALITY_GATE_ANALYSIS_PATH,
    index=False
)

final_test_candidates_df.to_csv(
    QUALITY_GATE_TEST_PATH,
    index=False
)

# 8. Report

print("QUALITY-GATED AGREEMENT RULE")
print("=" * 55)

display(
    validation_analysis_df
)

print(
    "\nValidation replacements removed "
    "by malformed-text safeguard:",
    int(
        (
            validation_base_gate
            &
            agreement_validation_df[
                "model_containment"
            ].ge(0.25)
            &
            agreement_validation_df[
                "has_repeated_ngram"
            ]
        ).sum()
    )
)

print(
    "Test replacements removed "
    "by malformed-text safeguard:",
    int(
        (
            test_base_gate
            &
            agreement_test_df[
                "model_containment"
            ].ge(0.25)
            &
            agreement_test_df[
                "has_repeated_ngram"
            ]
        ).sum()
    )
)

print(
    "\nFinal test replacement count:",
    len(final_test_candidates_df)
)

print(
    "\nFinal proposed replacements:"
)

display(
    final_test_candidates_df[
        [
            "row_number",
            "prob_passage",
            "cross_encoder_score",
            "score_margin",
            "model_jaccard",
            "model_containment",
            "variant",
            "baseline_prediction",
            "candidate_text",
            "baseline_word_count",
            "candidate_word_count",
            "has_repeated_ngram"
        ]
    ]
)

print(
    "\nCheckpoint complete:",
    os.path.exists(
        QUALITY_GATE_ANALYSIS_PATH
    )
    and os.path.exists(
        QUALITY_GATE_TEST_PATH
    )
)

QUALITY-GATED AGREEMENT RULE


,reconstructed_baseline_score,quality_gate_score,quality_gate_gain,estimated_full_validation_gain,validation_replacements,validation_wins,validation_losses,test_replacements
0,0.386986,0.397318,0.010332,0.004339,4,4,0,13



Validation replacements removed by malformed-text safeguard: 0
Test replacements removed by malformed-text safeguard: 1

Final test replacement count: 13

Final proposed replacements:


,row_number,prob_passage,cross_encoder_score,score_margin,model_jaccard,model_containment,variant,baseline_prediction,candidate_text,baseline_word_count,candidate_word_count,has_repeated_ngram
0,154,0.704557,0.956543,0.693848,1.000000,1.000000,sentence,MIT researchers can read a book without openin...,Researchers from the Massachusetts Institute o...,35,25,False
1,223,0.817307,0.946289,0.247070,1.000000,1.000000,sentence,Police: Four Women Arrested for Spraying Anti-...,Police arrested four women in North Carolina f...,36,24,False
2,332,0.879119,0.932617,0.665527,1.000000,1.000000,sentence,"Hizbullah, an Iran-backed Lebanese Shia group,...",(AP) – The evacuation of civilians and opposit...,91,37,False
3,93,0.643515,0.895020,0.242676,1.000000,1.000000,sentence,The 1937 Bugatti Type 57S was found in the gar...,The 1937 Bugatti Type 57S was found in the gar...,50,29,False
4,357,0.704158,0.862793,0.488770,1.000000,1.000000,sentence,The National Safety Council has some advice fo...,Men are more likely to have a heart attack aft...,66,25,False
5,50,0.881565,0.857910,0.470215,1.000000,1.000000,sentence,You won’t believe why ESPN said they hired Jor...,The former Vanderbilt quarterback is set to ap...,75,32,False
6,313,0.866267,0.853027,0.732727,1.000000,1.000000,sentence,"co/e8jsXhLBbj In a hilarious scene, Priyanka i...","In a hilarious scene, Priyanka is seen calling...",45,44,False
7,395,0.756985,0.844238,0.473389,1.000000,1.000000,sentence,This Is What Happens When You Leave A Hotel Cl...,Instead of encountering a mound of dirty towel...,40,26,False
8,94,0.755383,0.841309,0.106445,1.000000,1.000000,sentence,Photo: Peter Duke (left); Getty Images (right)...,President Obama’s Kenyan half-brother wants to...,47,17,False
9,187,0.817599,0.823242,0.673218,1.000000,1.000000,sentence,The highly controversial singer has been invol...,The highly controversial singer has been invol...,61,33,False



Checkpoint complete: True


In [46]:
import os
import re
import json
import pandas as pd

# configuration
FROZEN_BASELINE_PATH = os.path.join(
    ASSET_ROOT,
    "baseline",
    "submission_boundary_phrase_multi_cap150.csv"
)

FINAL_CANDIDATES_PATH = os.path.join(
    WORK_DIR,
    "test_agreement025_quality_gate_candidates.csv"
)

FINAL_SUBMISSION_PATH = os.path.join(
    WORK_DIR,
    "submission_crossencoder_agreement025_quality_v1.csv"
)

FINAL_AUDIT_PATH = os.path.join(
    WORK_DIR,
    "submission_crossencoder_agreement025_quality_v1_audit.csv"
)

FINAL_MANIFEST_PATH = os.path.join(
    WORK_DIR,
    "submission_crossencoder_agreement025_quality_v1_manifest.json"
)

EXPECTED_REPLACEMENT_COUNT = 13

# 1. Load the exact frozen baseline

if os.path.exists(FROZEN_BASELINE_PATH):
    frozen_submission_df = pd.read_csv(
        FROZEN_BASELINE_PATH
    )

elif "baseline_submission_df" in globals():
    frozen_submission_df = (
        baseline_submission_df
        .copy()
        .reset_index(drop=True)
    )

else:
    raise FileNotFoundError(
        "The frozen 0.45453 baseline could not be found."
    )


final_candidates_df = pd.read_csv(
    FINAL_CANDIDATES_PATH
)

frozen_submission_df = (
    frozen_submission_df
    .copy()
    .reset_index(drop=True)
)


print("Frozen baseline path:")
print(FROZEN_BASELINE_PATH)

print(
    "\nFrozen baseline shape:",
    frozen_submission_df.shape
)

print(
    "Frozen baseline columns:",
    frozen_submission_df.columns.tolist()
)

print(
    "Proposed replacement rows:",
    len(final_candidates_df)
)


# 2. Basic input checks

if "spoiler" not in frozen_submission_df.columns:
    raise KeyError(
        "Frozen baseline does not contain a "
        "'spoiler' column."
    )

required_candidate_columns = {
    "row_number",
    "candidate_text",
    "prob_passage",
    "cross_encoder_score",
    "score_margin",
    "model_jaccard",
    "model_containment",
    "variant",
    "has_repeated_ngram"
}

missing_candidate_columns = (
    required_candidate_columns
    - set(final_candidates_df.columns)
)

if missing_candidate_columns:
    raise KeyError(
        "Missing candidate columns: "
        f"{sorted(missing_candidate_columns)}"
    )

final_candidates_df[
    "row_number"
] = final_candidates_df[
    "row_number"
].astype(int)

if len(final_candidates_df) != EXPECTED_REPLACEMENT_COUNT:
    raise RuntimeError(
        "Expected "
        f"{EXPECTED_REPLACEMENT_COUNT} replacements, "
        f"but found {len(final_candidates_df)}."
    )

if final_candidates_df[
    "row_number"
].duplicated().any():
    raise RuntimeError(
        "Duplicate replacement row numbers were found."
    )

if not final_candidates_df[
    "row_number"
].between(
        0,
        len(frozen_submission_df) - 1
    ).all():
    raise RuntimeError(
        "At least one replacement row number is invalid."
    )

if final_candidates_df[
    "candidate_text"
].isna().any():
    raise RuntimeError(
        "At least one candidate prediction is missing."
    )

if final_candidates_df[
    "candidate_text"
].astype(str).str.strip().eq("").any():
    raise RuntimeError(
        "At least one candidate prediction is empty."
    )

if final_candidates_df[
    "has_repeated_ngram"
].astype(str).str.lower().eq("true").any():
    raise RuntimeError(
        "A malformed repeated-text candidate survived "
        "the quality gate."
    )


# 3. Source-extraction validation
def normalize_source_text(text):
    text = re.sub(
        r"\s+",
        " ",
        str(text)
    ).strip()

    return text.lower()


source_match_results = []

for candidate_row in final_candidates_df.itertuples(
    index=False
):
    row_number = int(
        candidate_row.row_number
    )

    article_paragraphs = test_df.iloc[
        row_number
    ].get(
        "targetParagraphs",
        []
    )

    if isinstance(
        article_paragraphs,
        str
    ):
        article_paragraphs = [
            article_paragraphs
        ]

    article_text = normalize_source_text(
        " ".join(
            str(paragraph)
            for paragraph in article_paragraphs
        )
    )

    candidate_text = normalize_source_text(
        candidate_row.candidate_text
    )

    source_match_results.append(
        candidate_text in article_text
    )

final_candidates_df[
    "candidate_found_in_article"
] = source_match_results

if not final_candidates_df[
    "candidate_found_in_article"
].all():
    failed_rows = final_candidates_df.loc[
        ~final_candidates_df[
            "candidate_found_in_article"
        ],
        "row_number"
    ].tolist()

    raise RuntimeError(
        "Candidate text was not found in the source "
        f"article for rows: {failed_rows}"
    )


# 4. Apply replacements to a copy of the frozen baseline

final_submission_df = (
    frozen_submission_df
    .copy()
)

audit_records = []

for candidate_row in final_candidates_df.itertuples(
    index=False
):
    row_number = int(
        candidate_row.row_number
    )

    baseline_prediction = str(
        final_submission_df.iloc[
            row_number
        ][
            "spoiler"
        ]
    )

    new_prediction = str(
        candidate_row.candidate_text
    ).strip()

    final_submission_df.iloc[
        row_number,
        final_submission_df.columns.get_loc(
            "spoiler"
        )
    ] = new_prediction

    audit_record = {
        "row_number":
            row_number,
        "baseline_prediction":
            baseline_prediction,
        "new_prediction":
            new_prediction,
        "baseline_word_count":
            len(
                baseline_prediction.split()
            ),
        "new_word_count":
            len(
                new_prediction.split()
            ),
        "prob_passage":
            float(
                candidate_row.prob_passage
            ),
        "cross_encoder_score":
            float(
                candidate_row.cross_encoder_score
            ),
        "score_margin":
            float(
                candidate_row.score_margin
            ),
        "model_jaccard":
            float(
                candidate_row.model_jaccard
            ),
        "model_containment":
            float(
                candidate_row.model_containment
            ),
        "variant":
            candidate_row.variant,
        "candidate_found_in_article":
            bool(
                candidate_row.candidate_found_in_article
            )
    }

    if "id" in final_submission_df.columns:
        audit_record[
            "submission_id"
        ] = final_submission_df.iloc[
            row_number
        ][
            "id"
        ]

    audit_records.append(
        audit_record
    )


final_audit_df = pd.DataFrame(
    audit_records
).sort_values(
    "row_number"
).reset_index(drop=True)

# 5. Final submission integrity checks

if final_submission_df.shape != frozen_submission_df.shape:
    raise RuntimeError(
        "Submission shape changed unexpectedly."
    )

if (
    final_submission_df.columns.tolist()
    != frozen_submission_df.columns.tolist()
):
    raise RuntimeError(
        "Submission columns or column order changed."
    )

if "id" in frozen_submission_df.columns:
    if not final_submission_df[
        "id"
    ].equals(
        frozen_submission_df[
            "id"
        ]
    ):
        raise RuntimeError(
            "Submission ID values or ordering changed."
        )

if final_submission_df[
    "spoiler"
].isna().any():
    raise RuntimeError(
        "Final submission contains missing spoilers."
    )

if final_submission_df[
    "spoiler"
].astype(str).str.strip().eq("").any():
    raise RuntimeError(
        "Final submission contains empty spoilers."
    )

changed_mask = (
    final_submission_df[
        "spoiler"
    ].astype(str)
    !=
    frozen_submission_df[
        "spoiler"
    ].astype(str)
)

changed_row_numbers = set(
    final_submission_df.index[
        changed_mask
    ].astype(int)
)

expected_row_numbers = set(
    final_candidates_df[
        "row_number"
    ].astype(int)
)

if changed_row_numbers != expected_row_numbers:
    raise RuntimeError(
        "Actual changed rows do not match the "
        "quality-gated replacement rows."
    )

if int(changed_mask.sum()) != EXPECTED_REPLACEMENT_COUNT:
    raise RuntimeError(
        "Expected exactly "
        f"{EXPECTED_REPLACEMENT_COUNT} changed rows, "
        f"but found {int(changed_mask.sum())}."
    )

if final_submission_df[
    "spoiler"
].astype(str).str.contains(
    r"\n|\r",
    regex=True
).any():
    raise RuntimeError(
        "A spoiler contains a newline character."
    )

# 6. Save submission, audit, and manifest

if os.path.exists(FINAL_SUBMISSION_PATH):
    raise FileExistsError(
        "The output submission already exists. "
        "Rename the version before overwriting it."
    )

final_submission_df.to_csv(
    FINAL_SUBMISSION_PATH,
    index=False
)

final_audit_df.to_csv(
    FINAL_AUDIT_PATH,
    index=False
)

manifest = {
    "submission_name":
        os.path.basename(
            FINAL_SUBMISSION_PATH
        ),
    "frozen_public_score":
        0.45453,
    "source_baseline":
        FROZEN_BASELINE_PATH,
    "replacement_rule": {
        "minimum_prob_passage":
            0.60,
        "minimum_cross_encoder_score":
            0.75,
        "minimum_score_margin":
            0.10,
        "minimum_model_containment":
            0.25,
        "minimum_candidate_words":
            8,
        "maximum_candidate_words":
            60,
        "reject_adjacent_repeated_ngram":
            True
    },
    "validation_evidence": {
        "reconstructed_baseline_score":
            0.386986,
        "quality_gate_score":
            0.397318,
        "routed_validation_gain":
            0.010332,
        "estimated_full_validation_gain":
            0.004339,
        "validation_replacements":
            4,
        "validation_wins":
            4,
        "validation_losses":
            0
    },
    "test_replacements":
        EXPECTED_REPLACEMENT_COUNT,
    "changed_row_numbers":
        sorted(
            expected_row_numbers
        )
}

with open(
    FINAL_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as manifest_file:
    json.dump(
        manifest,
        manifest_file,
        indent=2
    )


# 7. Final report

print("\nFINAL SUBMISSION AUDIT")
print("=" * 60)

print(
    "Submission rows:",
    len(final_submission_df)
)

print(
    "Changed rows:",
    int(changed_mask.sum())
)

print(
    "Unchanged rows:",
    int((~changed_mask).sum())
)

print(
    "Source matches:",
    int(
        final_audit_df[
            "candidate_found_in_article"
        ].sum()
    ),
    "/",
    len(final_audit_df)
)

print(
    "Missing spoilers:",
    int(
        final_submission_df[
            "spoiler"
        ].isna().sum()
    )
)

print(
    "Empty spoilers:",
    int(
        final_submission_df[
            "spoiler"
        ]
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )
)

print(
    "Submission shape:",
    final_submission_df.shape
)

print(
    "Submission columns:",
    final_submission_df.columns.tolist()
)

print("\nSaved submission:")
print(FINAL_SUBMISSION_PATH)

print("\nSaved audit:")
print(FINAL_AUDIT_PATH)

print("\nSaved manifest:")
print(FINAL_MANIFEST_PATH)

print(
    "\nCheckpoint complete:",
    os.path.exists(
        FINAL_SUBMISSION_PATH
    )
    and os.path.exists(
        FINAL_AUDIT_PATH
    )
    and os.path.exists(
        FINAL_MANIFEST_PATH
    )
    and int(changed_mask.sum())
        == EXPECTED_REPLACEMENT_COUNT
)

print("\nFinal 13 replacements:")

display(
    final_audit_df[
        [
            "row_number",
            "baseline_prediction",
            "new_prediction",
            "baseline_word_count",
            "new_word_count",
            "prob_passage",
            "cross_encoder_score",
            "score_margin",
            "model_containment"
        ]
    ]
)

Frozen baseline path:
/kaggle/input/datasets/stex098/task2-passage-model-assets/Task2_Kaggle_Passage/baseline/submission_boundary_phrase_multi_cap150.csv

Frozen baseline shape: (400, 2)
Frozen baseline columns: ['id', 'spoiler']
Proposed replacement rows: 13

FINAL SUBMISSION AUDIT
Submission rows: 400
Changed rows: 13
Unchanged rows: 387
Source matches: 13 / 13
Missing spoilers: 0
Empty spoilers: 0
Submission shape: (400, 2)
Submission columns: ['id', 'spoiler']

Saved submission:
/kaggle/working/task2_passage_v1/submission_crossencoder_agreement025_quality_v1.csv

Saved audit:
/kaggle/working/task2_passage_v1/submission_crossencoder_agreement025_quality_v1_audit.csv

Saved manifest:
/kaggle/working/task2_passage_v1/submission_crossencoder_agreement025_quality_v1_manifest.json

Checkpoint complete: True

Final 13 replacements:


,row_number,baseline_prediction,new_prediction,baseline_word_count,new_word_count,prob_passage,cross_encoder_score,score_margin,model_containment
0,50,You won’t believe why ESPN said they hired Jor...,The former Vanderbilt quarterback is set to ap...,75,32,0.881565,0.857910,0.470215,1.000000
1,83,she had plastic surgery years ago to make her ...,Julie Chen revealed on Wednesday that she had ...,57,22,0.681752,0.914551,0.418701,0.428571
2,93,The 1937 Bugatti Type 57S was found in the gar...,The 1937 Bugatti Type 57S was found in the gar...,50,29,0.643515,0.895020,0.242676,1.000000
3,94,Photo: Peter Duke (left); Getty Images (right)...,President Obama’s Kenyan half-brother wants to...,47,17,0.755383,0.841309,0.106445,1.000000
4,129,Won’t Get Playstation 4.5-Style Upgrades Of co...,Speaking after Microsoft’s Build 2016 keynote ...,18,34,0.891870,0.806152,0.590698,0.428571
5,154,MIT researchers can read a book without openin...,Researchers from the Massachusetts Institute o...,35,25,0.704557,0.956543,0.693848,1.000000
6,187,The highly controversial singer has been invol...,The highly controversial singer has been invol...,61,33,0.817599,0.823242,0.673218,1.000000
7,223,Police: Four Women Arrested for Spraying Anti-...,Police arrested four women in North Carolina f...,36,24,0.817307,0.946289,0.247070,1.000000
8,249,Thousands of angry demonstrators have poured o...,Thousands of angry demonstrators have poured o...,63,26,0.840791,0.769043,0.139160,1.000000
9,313,"co/e8jsXhLBbj In a hilarious scene, Priyanka i...","In a hilarious scene, Priyanka is seen calling...",45,44,0.866267,0.853027,0.732727,1.000000
